# Adaptive Personalized Federated Learning Framework
### Privacy Preserving Next Word Prediction

This notebook is fully configured for local CPU execution. It contains all modules from data loading through client partitioning, profiling, training, gated routing (CAGER v1, v2, v3), and comparative ablation analysis.

## Local Configuration & Environment
Defines paths, device options, and QUICK_TEST settings for local execution.

In [3]:
import os
import sys

# Workspace configuration
WORKSPACE_DIR = "d:/project"
DATASET_PATH = os.path.join(WORKSPACE_DIR, "dataset")
PROCESSED_PATH = os.path.join(WORKSPACE_DIR, "processed_dataset")
TOKENIZED_PATH = os.path.join(WORKSPACE_DIR, "tokenized_dataset")
CLIENT_PATH = os.path.join(WORKSPACE_DIR, "clients")
MODEL_SAVE_PATH = os.path.join(WORKSPACE_DIR, "client_models")
CHECKPOINT_PATH = os.path.join(WORKSPACE_DIR, "checkpoints")
CONFIG_PATH = os.path.join(WORKSPACE_DIR, "adaptive_client_config.json")
EVALUATION_PATH = os.path.join(WORKSPACE_DIR, "personalized_evaluation")

# Quick test configurations
# Set this to True to train on a very small subset of data for testing.
QUICK_TEST = True
QUICK_TEST_SAMPLES = 100  # Number of samples per client/dataset for quick testing

# Device settings
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Privacy & Personalization Enhancements
ENABLE_DP = True
DP_CLIP_NORM = 1.0
DP_NOISE_MULTIPLIER = 0.1

ENABLE_FEDPROX = True
FEDPROX_MU = 0.01

print(f"[Config] Workspace: {WORKSPACE_DIR}")
print(f"[Config] Device: {DEVICE}")
print(f"[Config] Quick Test Mode: {QUICK_TEST} (Samples per dataset: {QUICK_TEST_SAMPLES})")


[Config] Workspace: d:/project
[Config] Device: cpu
[Config] Quick Test Mode: True (Samples per dataset: 100)


## Module 1: Multilingual Data Loading
Loads datasets from local CSV files and displays metadata.

In [4]:
# ============================================
# Module 1 : Data Loading
# Project:
# Adaptive Personalized Federated Learning
# for Privacy-Preserving Next Word Prediction
# ============================================

import os
import pandas as pd

# -------------------------------------------------
# Dataset Path
# -------------------------------------------------

import sys
try:
    MODULE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    MODULE_DIR = os.path.abspath(os.path.join(os.getcwd(), 'code', 'modules'))
if MODULE_DIR not in sys.path:
    sys.path.append(MODULE_DIR)
import local_config

DATASET_PATH = local_config.DATASET_PATH

# -------------------------------------------------
# Dataset Names
# -------------------------------------------------

DATASETS = {
    "shakespeare": "Shakespeare_cleaned.csv",
    "tamil_news": "tamil_news_cleaned.csv",
    "tanglish": "tanglish.csv",
    "codemix": "codemix.csv",
    "movie_reviews": "tamil_movie_reviews_test (3).csv",
    "colloquial": "colloquial.csv"
}

# -------------------------------------------------
# Load Dataset
# -------------------------------------------------

def load_dataset(dataset_name, filename):

    file_path = os.path.join(DATASET_PATH, filename)

    if not os.path.exists(file_path):
        print(f"❌ {filename} not found.")
        return None

    df = pd.read_csv(file_path)

    print("="*60)
    print(f"Dataset : {dataset_name}")
    print("="*60)

    print(f"Shape : {df.shape}")

    print("\nColumns")
    print(df.columns.tolist())

    print("\nMissing Values")
    print(df.isnull().sum())

    print("\nFirst 5 Rows")
    print(df.head())

    print()

    return df

# -------------------------------------------------
# Load All Datasets
# -------------------------------------------------

def load_all_datasets():

    dataset_dict = {}

    for dataset_name, filename in DATASETS.items():

        dataset_dict[dataset_name] = load_dataset(
            dataset_name,
            filename
        )

    return dataset_dict

# -------------------------------------------------
# Main
# -------------------------------------------------

if __name__ == "__main__":

    datasets = load_all_datasets()

    print("\nSuccessfully Loaded Datasets")

    print(list(datasets.keys()))

[Config] Workspace: d:/project
[Config] Device: cpu
[Config] Quick Test Mode: True (Samples per dataset: 100)
Dataset : shakespeare
Shape : (108741, 1)

Columns
['PlayerLine']

Missing Values
PlayerLine    0
dtype: int64

First 5 Rows
                                      PlayerLine
0         so shaken as we are, so wan with care,
1     find we a time for frighted peace to pant,
2  and breathe shortwinded accents of new broils
3        to be commenced in strands afar remote.
4      no more the thirsty entrance of this soil

Dataset : tamil_news
Shape : (14521, 2)

Columns
['NewsInTamil', 'CategoryInTamil']

Missing Values
NewsInTamil        0
CategoryInTamil    0
dtype: int64

First 5 Rows
                                         NewsInTamil CategoryInTamil
0         ஈராக்கில் 43 ஆண்டுகள் கழித்து அழகிப்போட்டி           உலகம்
1  இந்திய அளவில் ட்ரெண்ட் ஆன அஜித்தின் தள்லே தில்...          சினிமா
2  சொந்த செலவில் வாகன காப்பீடு எடுத்து கொடுத்த கா...       தமிழ்நாடு
3              பிறந்தநாளி

## Module 2: Multilingual Preprocessing
Cleans text datasets, removing empty rows and duplicates.

In [5]:
# ============================================
# Module 2 : Data Preprocessing
# Adaptive Personalized Federated Learning
# ============================================

import os
import re
import pandas as pd

# ----------------------------------------------------
# Paths
# ----------------------------------------------------

import sys
try:
    MODULE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    MODULE_DIR = os.path.abspath(os.path.join(os.getcwd(), 'code', 'modules'))
if MODULE_DIR not in sys.path:
    sys.path.append(MODULE_DIR)
import local_config

DATASET_PATH = local_config.DATASET_PATH
SAVE_PATH = local_config.PROCESSED_PATH

os.makedirs(SAVE_PATH, exist_ok=True)

# ----------------------------------------------------
# Dataset Configuration
# ----------------------------------------------------

DATASETS = {

    "shakespeare": {
        "file": "Shakespeare_cleaned.csv",
        "text_column": "PlayerLine"
    },

    "tamil_news": {
        "file": "tamil_news_cleaned.csv",
        "text_column": "NewsInTamil"
    },

    "tanglish": {
        "file": "tanglish.csv",
        "text_column": "text"
    },

    "codemix": {
        "file": "codemix.csv",
        "text_column": "text"
    },

    "movie_reviews": {
        "file": "tamil_movie_reviews_test (3).csv",
        "text_column": "ReviewInTamil"
    },

    "colloquial": {
        "file": "colloquial.csv",
        "text_column": "Colloquial_Text"
    }

}

# ----------------------------------------------------
# Text Cleaning
# ----------------------------------------------------

def clean_text(text):

    text = str(text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    # Remove leading/trailing spaces
    text = text.strip()
    return text


# ----------------------------------------------------
# Preprocess One Dataset
# ----------------------------------------------------

def preprocess_dataset(file_name, text_column):

    file_path = os.path.join(DATASET_PATH, file_name)

    df = pd.read_csv(file_path)

    # Keep only text column
    df = df[[text_column]].copy()

    # Rename to common name
    df.columns = ["text"]

    # Remove null values
    df.dropna(inplace=True)

    # Convert to string
    df["text"] = df["text"].astype(str)

    # Clean text
    df["text"] = df["text"].apply(clean_text)

    # Remove empty rows
    df = df[df["text"] != ""]

    df = df[df["text"] != "nan"]

    # Remove duplicate rows
    df.drop_duplicates(inplace=True)

    # Reset index
    df.reset_index(drop=True, inplace=True)

    if local_config.QUICK_TEST:
        n_samples = min(local_config.QUICK_TEST_SAMPLES, len(df))
        df = df.head(n_samples).copy()
        print(f"  [Quick Test] Subsampled to {len(df)} rows.")

    return df


# ----------------------------------------------------
# Process All Datasets
# ----------------------------------------------------

def preprocess_all():

    processed = {}

    print("=" * 70)
    print("Starting Data Preprocessing")
    print("=" * 70)

    for dataset_name, info in DATASETS.items():

        print(f"\nProcessing : {dataset_name}")

        df = preprocess_dataset(

            info["file"],
            info["text_column"]

        )

        save_file = os.path.join(

            SAVE_PATH,

            dataset_name + ".csv"

        )

        df.to_csv(save_file, index=False)

        processed[dataset_name] = df

        print(f"Samples : {len(df)}")
        print(f"Saved    : {save_file}")

    print("\nAll datasets processed successfully.")

    return processed


# ----------------------------------------------------
# Main
# ----------------------------------------------------

if __name__ == "__main__":

    datasets = preprocess_all()

    print("\nExample:")

    for name, df in datasets.items():

        print("\n", "="*40)
        print(name)
        print(df.head())

Starting Data Preprocessing

Processing : shakespeare
  [Quick Test] Subsampled to 100 rows.
Samples : 100
Saved    : d:/project\processed_dataset\shakespeare.csv

Processing : tamil_news
  [Quick Test] Subsampled to 100 rows.
Samples : 100
Saved    : d:/project\processed_dataset\tamil_news.csv

Processing : tanglish
  [Quick Test] Subsampled to 100 rows.
Samples : 100
Saved    : d:/project\processed_dataset\tanglish.csv

Processing : codemix
  [Quick Test] Subsampled to 100 rows.
Samples : 100
Saved    : d:/project\processed_dataset\codemix.csv

Processing : movie_reviews
  [Quick Test] Subsampled to 100 rows.
Samples : 100
Saved    : d:/project\processed_dataset\movie_reviews.csv

Processing : colloquial
  [Quick Test] Subsampled to 100 rows.
Samples : 100
Saved    : d:/project\processed_dataset\colloquial.csv

All datasets processed successfully.

Example:

shakespeare
                                            text
0         so shaken as we are, so wan with care,
1     find we a t

## Module 3: Text Tokenization
Tokenizes preprocessed datasets using the DistilGPT2 tokenizer.

In [6]:
# ============================================
# Module 3
# Tokenization for Causal Language Modeling
# ============================================

import os
import pandas as pd

from datasets import Dataset
from transformers import AutoTokenizer

# ----------------------------------------------------
# Paths
# ----------------------------------------------------

import sys
try:
    MODULE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    MODULE_DIR = os.path.abspath(os.path.join(os.getcwd(), 'code', 'modules'))
if MODULE_DIR not in sys.path:
    sys.path.append(MODULE_DIR)
import local_config

DATASET_PATH = local_config.PROCESSED_PATH
SAVE_PATH = local_config.TOKENIZED_PATH

os.makedirs(SAVE_PATH, exist_ok=True)

# ----------------------------------------------------
# Base Model
# ----------------------------------------------------

MODEL_NAME = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# GPT2 has no padding token
tokenizer.pad_token = tokenizer.eos_token

MAX_LENGTH = 128

# ----------------------------------------------------
# Tokenization Function
# ----------------------------------------------------

def tokenize_function(example):

    text = str(example["text"])

    if text.strip() == "":
        text = "[EMPTY]"

    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

    # Labels are same as input_ids for Causal LM
    tokens["labels"] = tokens["input_ids"].copy()

    return tokens


# ----------------------------------------------------
# Process One Dataset
# ----------------------------------------------------

def process_dataset(file_name):

    print("=" * 60)
    print(f"Processing : {file_name}")

    file_path = os.path.join(DATASET_PATH, file_name)

    # Read CSV
    df = pd.read_csv(file_path)

    # Check text column
    if "text" not in df.columns:
        raise ValueError(
            f"'text' column not found in {file_name}"
        )

    # Remove null values
    df = df.dropna(subset=["text"])

    # Convert to string
    df["text"] = df["text"].astype(str)

    # Remove empty rows
    df = df[df["text"].str.strip() != ""]

    # Reset index
    df.reset_index(drop=True, inplace=True)

    # Convert to HuggingFace Dataset
    dataset = Dataset.from_pandas(df)

    # Tokenization
    tokenized_dataset = dataset.map(
        tokenize_function,
        batched=False
    )

    # Save Folder
    save_folder = os.path.join(
        SAVE_PATH,
        os.path.splitext(file_name)[0]
    )

    tokenized_dataset.save_to_disk(save_folder)

    print("Saved :", save_folder)
    print("Samples :", len(tokenized_dataset))


# ----------------------------------------------------
# Process All Datasets
# ----------------------------------------------------

def process_all():

    files = sorted(os.listdir(DATASET_PATH))

    for file in files:

        if file.endswith(".csv"):

            process_dataset(file)


# ----------------------------------------------------
# Main
# ----------------------------------------------------

if __name__ == "__main__":

    process_all()

    print("\nTokenization Completed Successfully.")

d:\project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processing : codemix.csv


Saving the dataset (1/1 shards): 100%|██████████| 100/100 [00:00<00:00, 18134.39 examples/s]


Saved : d:/project\tokenized_dataset\codemix
Samples : 100
Processing : colloquial.csv


Saving the dataset (1/1 shards): 100%|██████████| 100/100 [00:00<00:00, 18414.65 examples/s]


...

Saved : d:/project\tokenized_dataset\shakespeare
Samples : 100
Processing : tamil_news.csv


Saving the dataset (1/1 shards): 100%|██████████| 100/100 [00:00<00:00, 22792.65 examples/s]


Saved : d:/project\tokenized_dataset\tamil_news
Samples : 100
Processing : tanglish.csv


Saving the dataset (1/1 shards): 100%|██████████| 100/100 [00:00<00:00, 15864.08 examples/s]

Saved : d:/project\tokenized_dataset\tanglish
Samples : 100

Tokenization Completed Successfully.


## Module 4: Client Split Partitioning
Partitions tokenized data into train/test sets across 11 clients.

In [7]:
# ============================================
# Module 4
# Federated Client Creation
# ============================================

import os
from datasets import load_from_disk

# --------------------------------------------
# Paths
# --------------------------------------------

import sys
try:
    MODULE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    MODULE_DIR = os.path.abspath(os.path.join(os.getcwd(), 'code', 'modules'))
if MODULE_DIR not in sys.path:
    sys.path.append(MODULE_DIR)
import local_config

TOKENIZED_PATH = local_config.TOKENIZED_PATH
CLIENT_PATH = local_config.CLIENT_PATH

os.makedirs(CLIENT_PATH, exist_ok=True)

# --------------------------------------------
# Number of Clients Per Dataset
# --------------------------------------------

CLIENT_SPLITS = {

    "shakespeare":3,

    "tamil_news":2,

    "tanglish":2,

    "codemix":2,

    "movie_reviews":1,

    "colloquial":1

}

# --------------------------------------------
# Create Clients
# --------------------------------------------

def create_clients(dataset_name):

    dataset_path = os.path.join(
        TOKENIZED_PATH,
        dataset_name
    )

    dataset = load_from_disk(dataset_path)

    n_clients = CLIENT_SPLITS[dataset_name]

    dataset = dataset.shuffle(seed=42)

    total = len(dataset)

    split_size = total // n_clients

    print()

    print("="*60)

    print(dataset_name)

    print("Total Samples :", total)

    print("Clients :", n_clients)

    for i in range(n_clients):

        start = i * split_size

        if i == n_clients-1:

            end = total

        else:

            end = (i+1) * split_size

        client_dataset = dataset.select(
            range(start,end)
        )

        if len(client_dataset) < 5:
            # Safe fallback for tiny datasets
            train_dataset = client_dataset
            test_dataset = client_dataset
        else:
            train_test = client_dataset.train_test_split(
                test_size=0.2,
                seed=42
            )
            train_dataset = train_test["train"]
            test_dataset = train_test["test"]

        client_folder = os.path.join(

            CLIENT_PATH,

            f"client_{create_clients.client_id}"

        )

        os.makedirs(client_folder,exist_ok=True)

        train_dataset.save_to_disk(
            os.path.join(
                client_folder,
                "train"
            )
        )

        test_dataset.save_to_disk(
            os.path.join(
                client_folder,
                "test"
            )
        )

        print(

            f"Client {create_clients.client_id}",
            len(train_dataset),
            len(test_dataset)

        )

        create_clients.client_id += 1

create_clients.client_id = 1

# --------------------------------------------
# Main
# --------------------------------------------

def main():

    for dataset in CLIENT_SPLITS.keys():

        create_clients(dataset)

    print()

    print("="*60)

    print("Federated Clients Created Successfully.")

if __name__ == "__main__":

    main()


shakespeare
Total Samples : 100
Clients : 3


Saving the dataset (1/1 shards): 100%|██████████| 7/7 [00:00<00:00, 1155.91 examples/s]


Client 1 26 7


Saving the dataset (1/1 shards): 100%|██████████| 7/7 [00:00<00:00, 1840.99 examples/s]


Client 2 26 7


...

Client 9 40 10

movie_reviews
Total Samples : 100
Clients : 1


Saving the dataset (1/1 shards): 100%|██████████| 20/20 [00:00<00:00, 2963.02 examples/s]


Client 10 80 20

colloquial
Total Samples : 100
Clients : 1


Saving the dataset (1/1 shards): 100%|██████████| 20/20 [00:00<00:00, 4888.18 examples/s]

Client 11 80 20

Federated Clients Created Successfully.


## Module 5: Adaptive PEFT Model Base
Defines LoRA expert architecture and masking parameters.

In [8]:
# ==========================================================
# Module 5
# Adaptive PEFT Model
# DistilGPT2 + Adaptive LoRA
# ==========================================================

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model
)

# ----------------------------------------------------------
# Model Configuration
# ----------------------------------------------------------

MODEL_NAME = "distilgpt2"

# Maximum physical LoRA rank
MAX_LORA_RANK = 16

LORA_ALPHA = 32
LORA_DROPOUT = 0.1

import sys
try:
    MODULE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    MODULE_DIR = os.path.abspath(os.path.join(os.getcwd(), 'code', 'modules'))
if MODULE_DIR not in sys.path:
    sys.path.append(MODULE_DIR)
import local_config

DEVICE = local_config.DEVICE

# ----------------------------------------------------------
# Build Adaptive PEFT Model
# ----------------------------------------------------------

def build_peft_model(active_rank):

    if active_rank > MAX_LORA_RANK:
        raise ValueError(
            f"Active rank {active_rank} "
            f"cannot exceed maximum rank {MAX_LORA_RANK}"
        )

    print("=" * 60)
    print("Building Adaptive PEFT Model")
    print("=" * 60)

    print("Maximum LoRA Rank :", MAX_LORA_RANK)
    print("Active LoRA Rank  :", active_rank)

    # ------------------------------------------------------
    # Tokenizer
    # ------------------------------------------------------

    print("\nLoading Tokenizer...")

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME
    )

    tokenizer.pad_token = tokenizer.eos_token

    # ------------------------------------------------------
    # Base Model
    # ------------------------------------------------------

    print("Loading DistilGPT2...")

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME
    )

    # ------------------------------------------------------
    # Freeze Base Model
    # ------------------------------------------------------

    print("Freezing Base Model...")

    for param in model.parameters():

        param.requires_grad = False

    # ------------------------------------------------------
    # LoRA Configuration
    # ------------------------------------------------------

    config = LoraConfig(

        task_type=TaskType.CAUSAL_LM,

        # Physical rank is always 16
        r=MAX_LORA_RANK,

        lora_alpha=LORA_ALPHA,

        lora_dropout=LORA_DROPOUT,

        bias="none",

        target_modules=[
            "c_attn",
            "c_proj"
        ]

    )

    # ------------------------------------------------------
    # Attach LoRA
    # ------------------------------------------------------

    print("Attaching LoRA...")

    model = get_peft_model(
        model,
        config
    )

    # ------------------------------------------------------
    # Activate Required Rank
    # ------------------------------------------------------

    apply_active_rank(
        model,
        active_rank
    )

    model.to(DEVICE)

    return tokenizer, model


# ==========================================================
# Adaptive Rank Masking
# ==========================================================

def apply_active_rank(model, active_rank):

    print(
        f"Applying Active LoRA Rank : {active_rank}"
    )

    for module in model.modules():

        # LoRA modules contain lora_A and lora_B
        if hasattr(module, "lora_A") and hasattr(module, "lora_B"):

            adapter_name = "default"

            if (
                adapter_name not in module.lora_A
                or adapter_name not in module.lora_B
            ):
                continue

            lora_A = module.lora_A[adapter_name]
            lora_B = module.lora_B[adapter_name]

            # --------------------------------------------------
            # Create masks
            # --------------------------------------------------

            rank = lora_A.out_features

            mask_A = torch.zeros(
                rank,
                device=lora_A.weight.device
            )

            mask_B = torch.zeros(
                rank,
                device=lora_B.weight.device
            )

            mask_A[:active_rank] = 1.0
            mask_B[:active_rank] = 1.0

            # --------------------------------------------------
            # Freeze inactive LoRA dimensions using gradients
            # --------------------------------------------------

            lora_A.weight.register_hook(
                lambda grad, mask=mask_A:
                grad * mask.view(-1, 1)
            )

            lora_B.weight.register_hook(
                lambda grad, mask=mask_B:
                grad * mask.view(1, -1)
            )

            # --------------------------------------------------
            # Initialize inactive dimensions to zero
            # --------------------------------------------------

            with torch.no_grad():

                if active_rank < rank:

                    lora_A.weight[
                        active_rank:
                    ].zero_()

                    lora_B.weight[
                        :,
                        active_rank:
                    ].zero_()

            # --------------------------------------------------
            # Adjust LoRA scaling
            # --------------------------------------------------

            if hasattr(module, "scaling"):

                module.scaling[
                    adapter_name
                ] = LORA_ALPHA / active_rank


# ==========================================================
# Model Summary
# ==========================================================

def print_summary(
    model,
    active_rank
):

    total = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    frozen = total - trainable

    print("=" * 60)
    print("Adaptive PEFT Model Summary")
    print("=" * 60)

    print(
        f"Maximum LoRA Rank   : {MAX_LORA_RANK}"
    )

    print(
        f"Active LoRA Rank    : {active_rank}"
    )

    print(
        f"Total Parameters    : {total:,}"
    )

    print(
        f"Trainable Parameters: {trainable:,}"
    )

    print(
        f"Frozen Parameters   : {frozen:,}"
    )

    print(
        f"Trainable Percentage: "
        f"{(trainable / total) * 100:.4f}%"
    )

    print("=" * 60)


# ==========================================================
# Test
# ==========================================================

if __name__ == "__main__":

    # Example client
    active_rank = 16

    tokenizer, model = build_peft_model(
        active_rank
    )

    print_summary(
        model,
        active_rank
    )

Building Adaptive PEFT Model
Maximum LoRA Rank : 16
Active LoRA Rank  : 16

Loading Tokenizer...
Loading DistilGPT2...


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 5269.84it/s]


Freezing Base Model...
Attaching LoRA...
Applying Active LoRA Rank : 16
Adaptive PEFT Model Summary
Maximum LoRA Rank   : 16
Active LoRA Rank    : 16
Total Parameters    : 82,723,584
Trainable Parameters: 811,008
Frozen Parameters   : 81,912,576
Trainable Percentage: 0.9804%


d:\project\venv\Lib\site-packages\peft\tuners\lora\layer.py:2631: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


## Module 6: Client Personalization Profiling
Calculates client personalization scores and active LoRA ranks.

In [9]:
# ==========================================================
# Module 6
# Adaptive Personalization Profiling
# ==========================================================

import os
import json

from datasets import load_from_disk

# ----------------------------------------------------------
# Paths
# ----------------------------------------------------------

import sys
try:
    MODULE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    MODULE_DIR = os.path.abspath(os.path.join(os.getcwd(), 'code', 'modules'))
if MODULE_DIR not in sys.path:
    sys.path.append(MODULE_DIR)
import local_config

CLIENT_PATH = local_config.CLIENT_PATH
SAVE_PATH = local_config.WORKSPACE_DIR

# ----------------------------------------------------------
# Adaptive PEFT Configuration
# ----------------------------------------------------------

MAX_RANK = 16

# ----------------------------------------------------------
# Get Client Sample Count
# ----------------------------------------------------------

def get_client_samples():

    clients = sorted(

        [
            client
            for client in os.listdir(CLIENT_PATH)
            if client.startswith("client_")
        ],

        key=lambda x: int(
            x.split("_")[1]
        )
    )

    sample_count = {}

    for client in clients:

        train_path = os.path.join(
            CLIENT_PATH,
            client,
            "train"
        )

        train_dataset = load_from_disk(
            train_path
        )

        sample_count[client] = len(
            train_dataset
        )

    return sample_count


# ----------------------------------------------------------
# Compute Personalization Score
# ----------------------------------------------------------

def compute_scores(sample_count):

    max_samples = max(
        sample_count.values()
    )

    scores = {}

    for client, samples in sample_count.items():

        score = (
            samples / max_samples
        )

        scores[client] = round(
            score,
            4
        )

    return scores


# ----------------------------------------------------------
# Convert Score → Adaptive Rank
# ----------------------------------------------------------

def calculate_rank(score):

    if score >= 0.70:

        return 16

    elif score >= 0.40:

        return 12

    elif score >= 0.15:

        return 8

    else:

        return 4


# ----------------------------------------------------------
# Build Client Configuration
# ----------------------------------------------------------

def build_client_config(
    sample_count,
    scores
):

    client_config = {}

    for client in sample_count:

        score = scores[client]

        rank = calculate_rank(
            score
        )

        client_config[client] = {

            "samples":
                sample_count[client],

            "personalization_score":
                score,

            "max_rank":
                MAX_RANK,

            "active_rank":
                rank

        }

    return client_config


# ----------------------------------------------------------
# Display
# ----------------------------------------------------------

def display_config(
    client_config
):

    print("=" * 80)

    print(
        "ADAPTIVE PEFT CLIENT CONFIGURATION"
    )

    print("=" * 80)

    print(
        f"{'Client':<12}"
        f"{'Samples':<12}"
        f"{'Score':<12}"
        f"{'Active Rank':<15}"
    )

    print("-" * 80)

    clients = sorted(

        client_config.keys(),

        key=lambda x: int(
            x.split("_")[1]
        )

    )

    for client in clients:

        info = client_config[client]

        print(

            f"{client:<12}"

            f"{info['samples']:<12}"

            f"{info['personalization_score']:<12.4f}"

            f"{info['active_rank']:<15}"

        )


# ----------------------------------------------------------
# Save Configuration
# ----------------------------------------------------------

def save_config(
    client_config
):

    file_path = os.path.join(

        SAVE_PATH,

        "adaptive_client_config.json"

    )

    with open(
        file_path,
        "w"
    ) as f:

        json.dump(
            client_config,
            f,
            indent=4
        )

    print(
        "\nSaved :",
        file_path
    )


# ----------------------------------------------------------
# Main
# ----------------------------------------------------------

if __name__ == "__main__":

    sample_count = (
        get_client_samples()
    )

    scores = (
        compute_scores(
            sample_count
        )
    )

    client_config = (
        build_client_config(
            sample_count,
            scores
        )
    )

    display_config(
        client_config
    )

    save_config(
        client_config
    )

ADAPTIVE PEFT CLIENT CONFIGURATION
Client      Samples     Score       Active Rank    
--------------------------------------------------------------------------------
client_1    26          0.3250      8              
client_2    26          0.3250      8              
client_3    27          0.3375      8              
client_4    40          0.5000      12             
client_5    40          0.5000      12             
client_6    40          0.5000      12             
client_7    40          0.5000      12             
client_8    40          0.5000      12             
client_9    40          0.5000      12             
client_10   80          1.0000      16             
client_11   80          1.0000      16             

Saved : d:/project\adaptive_client_config.json


## Module 7: Adaptive PEFT Client Training
Fine-tunes personalized LoRA adapters for each client.

In [10]:
# ==========================================================
# Module 7
# Adaptive Personalized Federated Client Training
# Resume-Safe Version
# ==========================================================

import os
import json
import torch

from datasets import load_from_disk

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model
)


# ==========================================================
# Paths
# ==========================================================

import sys
try:
    MODULE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    MODULE_DIR = os.path.abspath(os.path.join(os.getcwd(), 'code', 'modules'))
if MODULE_DIR not in sys.path:
    sys.path.append(MODULE_DIR)
import local_config

CLIENT_PATH = local_config.CLIENT_PATH
MODEL_SAVE_PATH = local_config.MODEL_SAVE_PATH
CHECKPOINT_PATH = local_config.CHECKPOINT_PATH
CONFIG_PATH = local_config.CONFIG_PATH


os.makedirs(
    MODEL_SAVE_PATH,
    exist_ok=True
)

os.makedirs(
    CHECKPOINT_PATH,
    exist_ok=True
)


# ==========================================================
# Model Configuration
# ==========================================================

MODEL_NAME = "distilgpt2"

MAX_LORA_RANK = 16

LORA_ALPHA = 32

LORA_DROPOUT = 0.1

DEVICE = local_config.DEVICE


# ==========================================================
# Load Adaptive Client Configuration
# ==========================================================

if not os.path.exists(CONFIG_PATH):

    raise FileNotFoundError(
        f"Adaptive client configuration not found:\n"
        f"{CONFIG_PATH}\n\n"
        f"Please run Module 6 first."
    )


with open(
    CONFIG_PATH,
    "r"
) as f:

    CLIENT_CONFIG = json.load(f)


print("=" * 70)

print(
    "Adaptive Client Configuration Loaded"
)

print("=" * 70)


# ==========================================================
# Tokenizer
# ==========================================================

print(
    "Loading Tokenizer..."
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

tokenizer.pad_token = tokenizer.eos_token


# ==========================================================
# Apply Adaptive Active Rank
# ==========================================================

def apply_active_rank(
    model,
    active_rank
):

    print(
        f"Applying Active LoRA Rank : {active_rank}"
    )

    for module in model.modules():

        # --------------------------------------------------
        # Check whether this is a LoRA layer
        # --------------------------------------------------

        if not hasattr(
            module,
            "lora_A"
        ):
            continue

        if not hasattr(
            module,
            "lora_B"
        ):
            continue

        adapter_name = "default"

        if (
            adapter_name
            not in module.lora_A
        ):
            continue

        if (
            adapter_name
            not in module.lora_B
        ):
            continue

        # --------------------------------------------------
        # Get LoRA matrices
        # --------------------------------------------------

        lora_A = module.lora_A[
            adapter_name
        ]

        lora_B = module.lora_B[
            adapter_name
        ]

        rank = lora_A.out_features

        # --------------------------------------------------
        # Safety checks
        # --------------------------------------------------

        if active_rank < 1:

            raise ValueError(
                "Active LoRA rank must be >= 1."
            )

        if active_rank > rank:

            raise ValueError(
                f"Active rank {active_rank} "
                f"is greater than physical rank {rank}."
            )

        # --------------------------------------------------
        # Full rank
        # --------------------------------------------------

        if active_rank == rank:

            print(
                f"Full LoRA rank active: "
                f"{rank}/{rank}"
            )

            continue

        # --------------------------------------------------
        # Zero inactive LoRA dimensions
        # --------------------------------------------------

        with torch.no_grad():

            lora_A.weight[
                active_rank:
            ].zero_()

            lora_B.weight[
                :,
                active_rank:
            ].zero_()

        # --------------------------------------------------
        # Gradient mask for LoRA A
        # --------------------------------------------------

        def mask_A_gradient(
            grad,
            active_rank=active_rank
        ):

            mask = torch.zeros(
                grad.shape[0],
                device=grad.device,
                dtype=grad.dtype
            )

            mask[
                :active_rank
            ] = 1.0

            return (
                grad
                * mask.view(-1, 1)
            )

        # --------------------------------------------------
        # Gradient mask for LoRA B
        # --------------------------------------------------

        def mask_B_gradient(
            grad,
            active_rank=active_rank
        ):

            mask = torch.zeros(
                grad.shape[1],
                device=grad.device,
                dtype=grad.dtype
            )

            mask[
                :active_rank
            ] = 1.0

            return (
                grad
                * mask.view(1, -1)
            )

        # --------------------------------------------------
        # Register gradient hooks
        # --------------------------------------------------

        lora_A.weight.register_hook(
            mask_A_gradient
        )

        lora_B.weight.register_hook(
            mask_B_gradient
        )

        # --------------------------------------------------
        # Adaptive LoRA scaling
        # --------------------------------------------------

        if hasattr(
            module,
            "scaling"
        ):

            module.scaling[
                adapter_name
            ] = (
                LORA_ALPHA
                / active_rank
            )

        print(
            f"Adaptive masking applied: "
            f"{active_rank}/{rank}"
        )


# ==========================================================
# Create PEFT Model
# ==========================================================

def create_model(
    active_rank
):

    print(
        f"Creating model "
        f"with Active Rank = {active_rank}"
    )

    # ------------------------------------------------------
    # Load DistilGPT2
    # ------------------------------------------------------

    model = (
        AutoModelForCausalLM
        .from_pretrained(
            MODEL_NAME
        )
    )

    # ------------------------------------------------------
    # Freeze Base Model
    # ------------------------------------------------------

    for param in model.parameters():

        param.requires_grad = False

    # ------------------------------------------------------
    # LoRA Configuration
    # ------------------------------------------------------

    config = LoraConfig(

        task_type=TaskType.CAUSAL_LM,

        # Physical rank remains fixed.
        # Adaptive rank is implemented using masking.

        r=MAX_LORA_RANK,

        lora_alpha=LORA_ALPHA,

        lora_dropout=LORA_DROPOUT,

        bias="none",

        target_modules=[
            "c_attn",
            "c_proj"
        ]
    )

    # ------------------------------------------------------
    # Attach LoRA
    # ------------------------------------------------------

    model = get_peft_model(
        model,
        config
    )

    # ------------------------------------------------------
    # Apply client-specific active rank
    # ------------------------------------------------------

    apply_active_rank(
        model,
        active_rank
    )

    model.to(DEVICE)

    return model


# ==========================================================
# Load Client Dataset
# ==========================================================

def load_client_dataset(
    client_name
):

    train_path = os.path.join(
        CLIENT_PATH,
        client_name,
        "train"
    )

    if not os.path.exists(
        train_path
    ):

        raise FileNotFoundError(
            f"Training dataset not found:\n"
            f"{train_path}"
        )

    dataset = load_from_disk(
        train_path
    )

    # Remove raw text if present.
    # Tokenized columns are retained.

    if "text" in dataset.column_names:

        dataset = dataset.remove_columns(
            ["text"]
        )

    return dataset


# ==========================================================
# Data Collator
# ==========================================================

data_collator = (
    DataCollatorForLanguageModeling(

        tokenizer=tokenizer,

        mlm=False
    )
)


# ==========================================================
# Check Whether Client Is Already Trained
# ==========================================================

def is_client_trained(
    client_name,
    expected_rank
):

    save_folder = os.path.join(
        MODEL_SAVE_PATH,
        client_name
    )

    metrics_file = os.path.join(
        save_folder,
        "metrics.json"
    )

    adapter_config_file = os.path.join(
        save_folder,
        "adapter_config.json"
    )

    adapter_model_file = os.path.join(
        save_folder,
        "adapter_model.safetensors"
    )

    # ------------------------------------------------------
    # Check required files
    # ------------------------------------------------------

    if not os.path.exists(
        metrics_file
    ):

        return False

    if not os.path.exists(
        adapter_config_file
    ):

        return False

    # PEFT may save either safetensors
    # or a .bin adapter file depending on version.

    adapter_bin_file = os.path.join(
        save_folder,
        "adapter_model.bin"
    )

    if not (
        os.path.exists(
            adapter_model_file
        )
        or
        os.path.exists(
            adapter_bin_file
        )
    ):

        return False

    # ------------------------------------------------------
    # Check saved metadata
    # ------------------------------------------------------

    try:

        with open(
            metrics_file,
            "r"
        ) as f:

            saved_metrics = json.load(f)

    except Exception:

        return False

    saved_rank = saved_metrics.get(
        "active_lora_rank"
    )

    saved_max_rank = saved_metrics.get(
        "max_lora_rank"
    )

    # ------------------------------------------------------
    # Configuration consistency check
    # ------------------------------------------------------

    if saved_rank != expected_rank:

        print(
            f"{client_name}: "
            f"Saved rank = {saved_rank}, "
            f"Current rank = {expected_rank}"
        )

        return False

    if saved_max_rank != MAX_LORA_RANK:

        print(
            f"{client_name}: "
            f"Saved max rank = {saved_max_rank}, "
            f"Current max rank = {MAX_LORA_RANK}"
        )

        return False

    # Check DP and FedProx configurations
    saved_dp = saved_metrics.get("enable_dp", False)
    current_dp = getattr(local_config, "ENABLE_DP", False)
    if saved_dp != current_dp:
        print(f"{client_name}: Saved DP = {saved_dp}, Current DP = {current_dp}")
        return False

    saved_fedprox = saved_metrics.get("enable_fedprox", False)
    current_fedprox = getattr(local_config, "ENABLE_FEDPROX", False)
    if saved_fedprox != current_fedprox:
        print(f"{client_name}: Saved FedProx = {saved_fedprox}, Current FedProx = {current_fedprox}")
        return False

    return True


# ==========================================================
# Train One Client
# ==========================================================

def train_client(
    client_name
):

    print("\n")

    print("=" * 70)

    print(
        f"Training {client_name}"
    )

    print("=" * 70)

    # ------------------------------------------------------
    # Get client configuration
    # ------------------------------------------------------

    if client_name not in CLIENT_CONFIG:

        raise ValueError(
            f"No adaptive configuration "
            f"found for {client_name}"
        )

    client_info = (
        CLIENT_CONFIG[
            client_name
        ]
    )

    active_rank = int(
        client_info[
            "active_rank"
        ]
    )

    personalization_score = float(
        client_info[
            "personalization_score"
        ]
    )

    # ------------------------------------------------------
    # Display client information
    # ------------------------------------------------------

    print(
        f"Samples              : "
        f"{client_info['samples']}"
    )

    print(
        f"Personalization Score: "
        f"{personalization_score}"
    )

    print(
        f"Maximum LoRA Rank    : "
        f"{MAX_LORA_RANK}"
    )

    print(
        f"Active LoRA Rank     : "
        f"{active_rank}"
    )

    # ------------------------------------------------------
    # Check if already trained
    # ------------------------------------------------------

    if is_client_trained(
        client_name,
        active_rank
    ):

        print("\n")

        print(
            f"{client_name} : "
            "ALREADY TRAINED"
        )

        print(
            "Existing adapter is compatible "
            "with the current configuration."
        )

        print(
            "Skipping training."
        )

        print("=" * 70)

        return

    # ------------------------------------------------------
    # Load dataset
    # ------------------------------------------------------

    train_dataset = (
        load_client_dataset(
            client_name
        )
    )

    print(
        f"Training Samples     : "
        f"{len(train_dataset)}"
    )

    # ------------------------------------------------------
    # Create model
    # ------------------------------------------------------

    model = create_model(
        active_rank
    )

    # ------------------------------------------------------
    # Training arguments
    # ------------------------------------------------------

    max_steps = -1
    if local_config.QUICK_TEST:
        max_steps = 3
        print(f"  [Quick Test] Setting max_steps = {max_steps} for quick training validation.")

    args = TrainingArguments(
        output_dir=os.path.join(
            CHECKPOINT_PATH,
            client_name
        ),
        num_train_epochs=1,
        max_steps=max_steps,
        per_device_train_batch_size=2,
        learning_rate=2e-4,
        logging_steps=1 if local_config.QUICK_TEST else 50,
        logging_strategy="steps",
        save_strategy="no" if local_config.QUICK_TEST else "epoch",
        save_total_limit=1,
        eval_strategy="no",
        report_to="none",
        remove_unused_columns=False,
        fp16=torch.cuda.is_available(),
        use_cpu=not torch.cuda.is_available()
    )

    # ------------------------------------------------------
    # FedProx & DP Setup
    # ------------------------------------------------------
    global_weights = None
    if getattr(local_config, "ENABLE_FEDPROX", False):
        global_weights = {
            name: param.clone().detach()
            for name, param in model.named_parameters()
            if param.requires_grad
        }

    class PersonalizedFedTrainer(Trainer):
        def __init__(self, *args, global_model_weights=None, fedprox_mu=0.0, enable_dp=False, dp_clip_norm=1.0, dp_noise_multiplier=0.1, **kwargs):
            super().__init__(*args, **kwargs)
            self.global_model_weights = global_model_weights
            self.fedprox_mu = fedprox_mu
            self.enable_dp = enable_dp
            self.dp_clip_norm = dp_clip_norm
            self.dp_noise_multiplier = dp_noise_multiplier

        def compute_loss(self, model, inputs, return_outputs=False):
            outputs = model(**inputs)
            loss = outputs.loss if isinstance(outputs, dict) else outputs[0]
            
            if self.fedprox_mu > 0.0 and self.global_model_weights is not None:
                prox_loss = 0.0
                for name, param in model.named_parameters():
                    if param.requires_grad and name in self.global_model_weights:
                        diff = param - self.global_model_weights[name]
                        prox_loss += torch.sum(diff * diff)
                loss += (self.fedprox_mu / 2.0) * prox_loss
                
            return (loss, outputs) if return_outputs else loss

        def training_step(self, model, inputs, *args, **kwargs):
            model.train()
            inputs = self._prepare_inputs(inputs)
            
            loss = self.compute_loss(model, inputs)
            
            if self.args.n_gpu > 1:
                loss = loss.mean()
                
            loss.backward()
            
            if self.enable_dp:
                parameters = [p for p in model.parameters() if p.requires_grad and p.grad is not None]
                if len(parameters) > 0:
                    torch.nn.utils.clip_grad_norm_(parameters, self.dp_clip_norm)
                    sensitivity = 2.0 * self.dp_clip_norm
                    std = (sensitivity * self.dp_noise_multiplier) / self.args.per_device_train_batch_size
                    for p in parameters:
                        noise = torch.randn_like(p.grad) * std
                        p.grad.add_(noise)
                        
            return loss.detach()

    trainer = PersonalizedFedTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        data_collator=data_collator,
        global_model_weights=global_weights,
        fedprox_mu=getattr(local_config, "FEDPROX_MU", 0.0) if getattr(local_config, "ENABLE_FEDPROX", False) else 0.0,
        enable_dp=getattr(local_config, "ENABLE_DP", False),
        dp_clip_norm=getattr(local_config, "DP_CLIP_NORM", 1.0),
        dp_noise_multiplier=getattr(local_config, "DP_NOISE_MULTIPLIER", 0.1)
    )

    # ------------------------------------------------------
    # Train
    # ------------------------------------------------------

    print("\n")

    print(
        f"Starting training for {client_name}..."
    )

    train_result = (
        trainer.train()
    )

    training_loss = float(
        train_result.training_loss
    )

    print(
        f"\nTraining Loss : "
        f"{training_loss:.4f}"
    )

    # ------------------------------------------------------
    # Save adapter
    # ------------------------------------------------------

    save_folder = os.path.join(
        MODEL_SAVE_PATH,
        client_name
    )

    os.makedirs(
        save_folder,
        exist_ok=True
    )

    model.save_pretrained(
        save_folder
    )

    tokenizer.save_pretrained(
        save_folder
    )

    print(
        f"Adapter Saved : "
        f"{save_folder}"
    )

    # ------------------------------------------------------
    # Save metrics
    # ------------------------------------------------------

    metrics = {

        "client":
            client_name,

        "samples":
            len(train_dataset),

        "personalization_score":
            personalization_score,

        "max_lora_rank":
            MAX_LORA_RANK,

        "active_lora_rank":
            active_rank,

        "training_loss":
            training_loss,

        "enable_dp":
            getattr(local_config, "ENABLE_DP", False),

        "enable_fedprox":
            getattr(local_config, "ENABLE_FEDPROX", False),

        "status":
            "completed"
    }

    metrics_path = os.path.join(
        save_folder,
        "metrics.json"
    )

    with open(
        metrics_path,
        "w"
    ) as f:

        json.dump(
            metrics,
            f,
            indent=4
        )

    print(
        "Metrics Saved"
    )

    print("=" * 70)

    print(
        f"{client_name} "
        "Training Completed"
    )

    print("=" * 70)


# ==========================================================
# Train All Clients
# ==========================================================

def train_all_clients():

    clients = sorted(

        [
            client
            for client in os.listdir(
                CLIENT_PATH
            )
            if client.startswith(
                "client_"
            )
        ],

        key=lambda x: int(
            x.split("_")[1]
        )
    )

    print("\n")

    print("=" * 70)

    print(
        "ADAPTIVE PERSONALIZED "
        "FEDERATED TRAINING"
    )

    print("=" * 70)

    trained_count = 0

    skipped_count = 0

    for client in clients:

        expected_rank = int(
            CLIENT_CONFIG[
                client
            ][
                "active_rank"
            ]
        )

        # --------------------------------------------------
        # Skip completed client
        # --------------------------------------------------

        if is_client_trained(
            client,
            expected_rank
        ):

            print("\n")

            print("-" * 70)

            print(
                f"{client} : ALREADY TRAINED"
            )

            print(
                f"Active Rank : "
                f"{expected_rank}"
            )

            print(
                "Skipping training..."
            )

            print("-" * 70)

            skipped_count += 1

            continue

        # --------------------------------------------------
        # Train unfinished client
        # --------------------------------------------------

        print("\n")

        print("-" * 70)

        print(
            f"{client} : NOT COMPLETED"
        )

        print(
            f"Required Rank : "
            f"{expected_rank}"
        )

        print(
            "Starting training..."
        )

        print("-" * 70)

        train_client(
            client
        )

        trained_count += 1

    # ------------------------------------------------------
    # Final summary
    # ------------------------------------------------------

    print("\n")

    print("=" * 70)

    print(
        "CLIENT TRAINING SUMMARY"
    )

    print("=" * 70)

    print(
        f"Newly Trained : "
        f"{trained_count}"
    )

    print(
        f"Skipped       : "
        f"{skipped_count}"
    )

    print(
        f"Total Clients : "
        f"{len(clients)}"
    )

    print("=" * 70)


# ==========================================================
# Main
# ==========================================================

if __name__ == "__main__":

    train_all_clients()

Adaptive Client Configuration Loaded
Loading Tokenizer...


ADAPTIVE PERSONALIZED FEDERATED TRAINING


----------------------------------------------------------------------
client_1 : ALREADY TRAINED
Active Rank : 8
Skipping training...
----------------------------------------------------------------------


----------------------------------------------------------------------
client_2 : ALREADY TRAINED
Active Rank : 8
Skipping training...
----------------------------------------------------------------------


----------------------------------------------------------------------
client_3 : ALREADY TRAINED
Active Rank : 8
Skipping training...
----------------------------------------------------------------------


----------------------------------------------------------------------
client_4 : ALREADY TRAINED
Active Rank : 12
Skipping training...
----------------------------------------------------------------------


----------------------------------------------------------------

## Module 8A: Baseline Evaluation
Evaluates base model vs client personalized models to establish gains.

In [11]:
# ==========================================================
# Module 8A
# Personalized Evaluation
#
# Base DistilGPT2
#          VS
# Client-specific Adaptive PEFT
#
# Metrics:
# 1. Loss
# 2. Perplexity
# 3. Top-1 Next-Token Accuracy
# 4. Top-5 Next-Token Accuracy
# 5. Personalization Gain
# ==========================================================

import os
import json
import math

import torch
import pandas as pd

from datasets import load_from_disk
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling
)

from peft import PeftModel


# ==========================================================
# Paths
# ==========================================================

import sys
try:
    MODULE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    MODULE_DIR = os.path.abspath(os.path.join(os.getcwd(), 'code', 'modules'))
if MODULE_DIR not in sys.path:
    sys.path.append(MODULE_DIR)
import local_config

CLIENT_PATH = local_config.CLIENT_PATH
CLIENT_MODEL_PATH = local_config.MODEL_SAVE_PATH
EVALUATION_PATH = local_config.EVALUATION_PATH

os.makedirs(
    EVALUATION_PATH,
    exist_ok=True
)


# ==========================================================
# Model Configuration
# ==========================================================

MODEL_NAME = "distilgpt2"

MAX_LORA_RANK = 16

LORA_ALPHA = 32

DEVICE = local_config.DEVICE

BATCH_SIZE = 8


# ==========================================================
# Load Tokenizer
# ==========================================================

print("=" * 70)

print(
    "Loading DistilGPT2 Tokenizer"
)

print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

tokenizer.pad_token = tokenizer.eos_token


# ==========================================================
# Data Collator
# ==========================================================

data_collator = (
    DataCollatorForLanguageModeling(

        tokenizer=tokenizer,

        mlm=False
    )
)


# ==========================================================
# Get Clients
# ==========================================================

clients = sorted(

    [
        client
        for client in os.listdir(
            CLIENT_PATH
        )
        if client.startswith("client_")
    ],

    key=lambda x: int(
        x.split("_")[1]
    )
)


print(
    f"\nTotal Clients Found : "
    f"{len(clients)}"
)

print(
    "Clients :",
    clients
)


# ==========================================================
# Load Client Test Dataset
# ==========================================================

def load_test_dataset(
    client_name
):

    test_path = os.path.join(
        CLIENT_PATH,
        client_name,
        "test"
    )

    if not os.path.exists(
        test_path
    ):

        raise FileNotFoundError(
            f"Test dataset not found:\n"
            f"{test_path}"
        )

    dataset = load_from_disk(
        test_path
    )

    # ------------------------------------------------------
    # Remove raw text column if present
    # ------------------------------------------------------

    if "text" in dataset.column_names:
        dataset = dataset.remove_columns(
            ["text"]
        )

    if local_config.QUICK_TEST:
        n_samples = min(20, len(dataset))
        dataset = dataset.select(range(n_samples))
        print(f"  [Quick Test] Subsampled test dataset to {len(dataset)} samples.")

    return dataset


# ==========================================================
# Load Adaptive Client Configuration
# ==========================================================

CONFIG_PATH = local_config.CONFIG_PATH


if not os.path.exists(CONFIG_PATH):

    raise FileNotFoundError(
        "adaptive_client_config.json "
        "not found. Run Module 6 first."
    )


with open(
    CONFIG_PATH,
    "r"
) as f:

    CLIENT_CONFIG = json.load(f)


# ==========================================================
# Load Client Metrics
# ==========================================================

def load_client_metrics(
    client_name
):

    metrics_path = os.path.join(

        CLIENT_MODEL_PATH,

        client_name,

        "metrics.json"

    )

    if not os.path.exists(
        metrics_path
    ):

        raise FileNotFoundError(
            f"Metrics not found:\n"
            f"{metrics_path}"
        )

    with open(
        metrics_path,
        "r"
    ) as f:

        return json.load(f)


# ==========================================================
# Apply Adaptive Rank During Evaluation
# ==========================================================
#
# Important:
#
# Every saved adapter has physical rank 16.
#
# The client's effective rank may be:
#
# client_1  -> 16
# client_4  -> 8
# client_8  -> 12
# client_10 -> 4
#
# We therefore restore the correct adaptive
# scaling and keep inactive dimensions disabled.
#
# ==========================================================

def apply_adaptive_inference(
    model,
    active_rank
):

    print(
        f"Applying inference rank "
        f"{active_rank}/{MAX_LORA_RANK}"
    )

    for module in model.modules():

        if not hasattr(
            module,
            "lora_A"
        ):

            continue

        if not hasattr(
            module,
            "lora_B"
        ):

            continue

        adapter_name = "default"

        if adapter_name not in module.lora_A:
            continue

        if adapter_name not in module.lora_B:
            continue

        lora_A = module.lora_A[
            adapter_name
        ]

        lora_B = module.lora_B[
            adapter_name
        ]

        physical_rank = (
            lora_A.out_features
        )

        if active_rank > physical_rank:

            raise ValueError(
                f"Active rank {active_rank} "
                f"exceeds physical rank "
                f"{physical_rank}"
            )

        # --------------------------------------------------
        # Disable inactive dimensions
        # --------------------------------------------------

        if active_rank < physical_rank:

            with torch.no_grad():

                lora_A.weight[
                    active_rank:
                ].zero_()

                lora_B.weight[
                    :,
                    active_rank:
                ].zero_()

        # --------------------------------------------------
        # Restore adaptive scaling
        # --------------------------------------------------

        if hasattr(
            module,
            "scaling"
        ):

            module.scaling[
                adapter_name
            ] = (
                LORA_ALPHA
                / active_rank
            )


# ==========================================================
# Evaluate Model
# ==========================================================

def evaluate_model(
    model,
    dataset,
    model_name
):

    print(
        f"\nEvaluating : {model_name}"
    )

    # ------------------------------------------------------
    # DataLoader
    # ------------------------------------------------------

    loader = DataLoader(

        dataset,

        batch_size=BATCH_SIZE,

        shuffle=False,

        collate_fn=data_collator

    )

    # ------------------------------------------------------
    # Evaluation Variables
    # ------------------------------------------------------

    total_loss = 0.0

    total_batches = 0

    top1_correct = 0

    top5_correct = 0

    total_tokens = 0

    # ------------------------------------------------------
    # Evaluation
    # ------------------------------------------------------

    model.eval()

    with torch.no_grad():

        for batch in loader:

            input_ids = (
                batch[
                    "input_ids"
                ].to(DEVICE)
            )

            attention_mask = (
                batch[
                    "attention_mask"
                ].to(DEVICE)
            )

            labels = (
                batch[
                    "labels"
                ].to(DEVICE)
            )

            # --------------------------------------------------
            # Forward pass
            # --------------------------------------------------

            outputs = model(

                input_ids=input_ids,

                attention_mask=attention_mask,

                labels=labels

            )

            # --------------------------------------------------
            # Loss
            # --------------------------------------------------

            total_loss += (
                outputs.loss.item()
            )

            total_batches += 1

            # --------------------------------------------------
            # Causal LM next-token prediction
            # --------------------------------------------------
            #
            # logits at position t predict token t+1
            #
            # Therefore:
            #
            # logits[:, :-1]
            #       VS
            # labels[:, 1:]
            #
            # --------------------------------------------------

            logits = outputs.logits

            shift_logits = (
                logits[:, :-1, :]
            )

            shift_labels = (
                labels[:, 1:]
            )

            # --------------------------------------------------
            # Ignore padding tokens
            # --------------------------------------------------

            valid_mask = (
                shift_labels != -100
            )

            # --------------------------------------------------
            # Top-1 Prediction
            # --------------------------------------------------

            top1_predictions = (
                torch.argmax(
                    shift_logits,
                    dim=-1
                )
            )

            top1_correct += (

                (
                    top1_predictions
                    == shift_labels
                )
                & valid_mask

            ).sum().item()

            # --------------------------------------------------
            # Top-5 Prediction
            # --------------------------------------------------

            top5_predictions = (
                torch.topk(
                    shift_logits,
                    k=5,
                    dim=-1
                ).indices
            )

            expanded_labels = (
                shift_labels
                .unsqueeze(-1)
                .expand_as(
                    top5_predictions
                )
            )

            top5_matches = (
                top5_predictions
                == expanded_labels
            )

            top5_correct += (

                top5_matches.any(
                    dim=-1
                )
                & valid_mask

            ).sum().item()

            # --------------------------------------------------
            # Total valid tokens
            # --------------------------------------------------

            total_tokens += (
                valid_mask.sum().item()
            )

    # ======================================================
    # Final Metrics
    # ======================================================

    if total_batches == 0:

        raise ValueError(
            "No batches were evaluated."
        )

    if total_tokens == 0:

        raise ValueError(
            "No valid tokens found."
        )

    average_loss = (
        total_loss
        / total_batches
    )

    # ------------------------------------------------------
    # Perplexity
    # ------------------------------------------------------

    try:

        perplexity = math.exp(
            average_loss
        )

    except OverflowError:

        perplexity = float("inf")

    # ------------------------------------------------------
    # Accuracy
    # ------------------------------------------------------

    top1_accuracy = (

        top1_correct
        / total_tokens

    ) * 100

    top5_accuracy = (

        top5_correct
        / total_tokens

    ) * 100

    # ------------------------------------------------------
    # Display
    # ------------------------------------------------------

    print(
        f"{model_name}"
    )

    print(
        f"Loss       : "
        f"{average_loss:.4f}"
    )

    print(
        f"Perplexity : "
        f"{perplexity:.4f}"
    )

    print(
        f"Top-1 Acc  : "
        f"{top1_accuracy:.2f}%"
    )

    print(
        f"Top-5 Acc  : "
        f"{top5_accuracy:.2f}%"
    )

    print(
        f"Tokens     : "
        f"{total_tokens}"
    )

    return {

        "loss":
            float(average_loss),

        "perplexity":
            float(perplexity),

        "top1_accuracy":
            float(top1_accuracy),

        "top5_accuracy":
            float(top5_accuracy),

        "tokens":
            int(total_tokens)
    }


# ==========================================================
# Evaluate Base DistilGPT2
# ==========================================================

def evaluate_base_model(
    dataset
):

    print("\n")

    print(
        "=" * 70
    )

    print(
        "Loading Base DistilGPT2"
    )

    print(
        "=" * 70
    )

    base_model = (
        AutoModelForCausalLM
        .from_pretrained(
            MODEL_NAME
        )
    )

    base_model.to(
        DEVICE
    )

    base_model.eval()

    results = evaluate_model(

        base_model,

        dataset,

        "Base DistilGPT2"

    )

    # ------------------------------------------------------
    # Free memory
    # ------------------------------------------------------

    del base_model

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

    return results


# ==========================================================
# Evaluate Personalized Client Model
# ==========================================================

def evaluate_personalized_model(

    client_name,

    dataset,

    active_rank

):

    adapter_path = os.path.join(

        CLIENT_MODEL_PATH,

        client_name

    )

    if not os.path.exists(
        adapter_path
    ):

        raise FileNotFoundError(
            f"Adapter not found:\n"
            f"{adapter_path}"
        )

    print("\n")

    print(
        "=" * 70
    )

    print(
        f"Loading Personalized Adapter: "
        f"{client_name}"
    )

    print(
        "=" * 70
    )

    # ------------------------------------------------------
    # Load fresh base model
    # ------------------------------------------------------

    base_model = (
        AutoModelForCausalLM
        .from_pretrained(
            MODEL_NAME
        )
    )

    # ------------------------------------------------------
    # Attach saved LoRA adapter
    # ------------------------------------------------------

    model = PeftModel.from_pretrained(

        base_model,

        adapter_path,

        is_trainable=False

    )

    model.to(
        DEVICE
    )

    # ------------------------------------------------------
    # Restore adaptive rank behavior
    # ------------------------------------------------------

    apply_adaptive_inference(

        model,

        active_rank

    )

    model.eval()

    # ------------------------------------------------------
    # Evaluate
    # ------------------------------------------------------

    results = evaluate_model(

        model,

        dataset,

        f"{client_name} "
        f"Adaptive PEFT"

    )

    # ------------------------------------------------------
    # Free memory
    # ------------------------------------------------------

    del model

    del base_model

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

    return results


# ==========================================================
# Evaluate One Client
# ==========================================================

def evaluate_client(
    client_name
):

    print("\n")

    print(
        "#" * 75
    )

    print(
        f"PERSONALIZED EVALUATION : "
        f"{client_name}"
    )

    print(
        "#" * 75
    )

    # ------------------------------------------------------
    # Client configuration
    # ------------------------------------------------------

    if client_name not in CLIENT_CONFIG:

        raise ValueError(
            f"{client_name} not found "
            f"in adaptive_client_config.json"
        )

    client_info = (
        CLIENT_CONFIG[
            client_name
        ]
    )

    active_rank = int(
        client_info[
            "active_rank"
        ]
    )

    personalization_score = float(
        client_info[
            "personalization_score"
        ]
    )

    # ------------------------------------------------------
    # Load test set
    # ------------------------------------------------------

    test_dataset = (
        load_test_dataset(
            client_name
        )
    )

    print(
        f"Test Samples          : "
        f"{len(test_dataset)}"
    )

    print(
        f"Personalization Score : "
        f"{personalization_score:.4f}"
    )

    print(
        f"Active LoRA Rank      : "
        f"{active_rank}"
    )

    # ------------------------------------------------------
    # Base model evaluation
    # ------------------------------------------------------

    base_results = (
        evaluate_base_model(
            test_dataset
        )
    )

    # ------------------------------------------------------
    # Personalized model evaluation
    # ------------------------------------------------------

    personalized_results = (
        evaluate_personalized_model(

            client_name,

            test_dataset,

            active_rank

        )
    )

    # ------------------------------------------------------
    # Calculate gains
    # ------------------------------------------------------

    accuracy_gain = (

        personalized_results[
            "top1_accuracy"
        ]

        -

        base_results[
            "top1_accuracy"
        ]

    )

    top5_gain = (

        personalized_results[
            "top5_accuracy"
        ]

        -

        base_results[
            "top5_accuracy"
        ]

    )

    perplexity_change = (

        base_results[
            "perplexity"
        ]

        -

        personalized_results[
            "perplexity"
        ]

    )

    loss_change = (

        base_results[
            "loss"
        ]

        -

        personalized_results[
            "loss"
        ]

    )

    # ------------------------------------------------------
    # Client result
    # ------------------------------------------------------

    result = {

        "client":
            client_name,

        "test_samples":
            len(test_dataset),

        "personalization_score":
            personalization_score,

        "active_lora_rank":
            active_rank,

        # Base model
        "base_loss":
            base_results["loss"],

        "base_perplexity":
            base_results["perplexity"],

        "base_top1_accuracy":
            base_results["top1_accuracy"],

        "base_top5_accuracy":
            base_results["top5_accuracy"],

        # Personalized model
        "personalized_loss":
            personalized_results["loss"],

        "personalized_perplexity":
            personalized_results["perplexity"],

        "personalized_top1_accuracy":
            personalized_results[
                "top1_accuracy"
            ],

        "personalized_top5_accuracy":
            personalized_results[
                "top5_accuracy"
            ],

        # Improvements
        "top1_accuracy_gain":
            accuracy_gain,

        "top5_accuracy_gain":
            top5_gain,

        "loss_improvement":
            loss_change,

        "perplexity_improvement":
            perplexity_change,

        "tokens_evaluated":
            personalized_results["tokens"]

    }

    # ------------------------------------------------------
    # Display comparison
    # ------------------------------------------------------

    print("\n")

    print(
        "=" * 70
    )

    print(
        f"{client_name} RESULT"
    )

    print(
        "=" * 70
    )

    print(
        f"Base Loss              : "
        f"{base_results['loss']:.4f}"
    )

    print(
        f"Personalized Loss      : "
        f"{personalized_results['loss']:.4f}"
    )

    print(
        f"Base Perplexity        : "
        f"{base_results['perplexity']:.4f}"
    )

    print(
        f"Personalized Perplexity: "
        f"{personalized_results['perplexity']:.4f}"
    )

    print(
        f"Base Top-1 Accuracy    : "
        f"{base_results['top1_accuracy']:.2f}%"
    )

    print(
        f"Personalized Top-1    : "
        f"{personalized_results['top1_accuracy']:.2f}%"
    )

    print(
        f"Top-1 Accuracy Gain    : "
        f"{accuracy_gain:+.2f}%"
    )

    print(
        f"Base Top-5 Accuracy    : "
        f"{base_results['top5_accuracy']:.2f}%"
    )

    print(
        f"Personalized Top-5     : "
        f"{personalized_results['top5_accuracy']:.2f}%"
    )

    print(
        f"Top-5 Accuracy Gain    : "
        f"{top5_gain:+.2f}%"
    )

    print(
        "=" * 70
    )

    return result


# ==========================================================
# Evaluate All Clients
# ==========================================================

def evaluate_all_clients():

    all_results = []

    print("\n")

    print(
        "=" * 75
    )

    print(
        "ADAPTIVE PEFT PERSONALIZED EVALUATION"
    )

    print(
        "=" * 75
    )

    clients_to_eval = clients

    for client in clients_to_eval:

        try:

            result = evaluate_client(
                client
            )

            all_results.append(
                result
            )

        except Exception as e:

            print("\n")

            print(
                f"ERROR while evaluating "
                f"{client}"
            )

            print(
                str(e)
            )

            print(
                "Continuing with next client..."
            )

    # ------------------------------------------------------
    # Create DataFrame
    # ------------------------------------------------------

    if len(all_results) == 0:

        raise RuntimeError(
            "No clients were evaluated successfully."
        )

    df = pd.DataFrame(
        all_results
    )

    # ------------------------------------------------------
    # Save CSV
    # ------------------------------------------------------

    csv_path = os.path.join(

        EVALUATION_PATH,

        "personalized_evaluation.csv"

    )

    df.to_csv(
        csv_path,
        index=False
    )

    # ------------------------------------------------------
    # Save JSON
    # ------------------------------------------------------

    json_path = os.path.join(

        EVALUATION_PATH,

        "personalized_evaluation.json"

    )

    with open(
        json_path,
        "w"
    ) as f:

        json.dump(

            all_results,

            f,

            indent=4

        )

    # ------------------------------------------------------
    # Overall statistics
    # ------------------------------------------------------

    avg_base_accuracy = (
        df[
            "base_top1_accuracy"
        ].mean()
    )

    avg_personalized_accuracy = (
        df[
            "personalized_top1_accuracy"
        ].mean()
    )

    avg_accuracy_gain = (
        df[
            "top1_accuracy_gain"
        ].mean()
    )

    avg_base_ppl = (
        df[
            "base_perplexity"
        ].mean()
    )

    avg_personalized_ppl = (
        df[
            "personalized_perplexity"
        ].mean()
    )

    avg_ppl_improvement = (
        df[
            "perplexity_improvement"
        ].mean()
    )

    # Calculate consolidated (token-weighted) metrics
    total_tokens = sum(r.get("tokens_evaluated", 0) for r in all_results)
    
    overall_base_correct_top1 = sum(round((r.get("base_top1_accuracy", 0.0) / 100.0) * r.get("tokens_evaluated", 0)) for r in all_results)
    overall_personalized_correct_top1 = sum(round((r.get("personalized_top1_accuracy", 0.0) / 100.0) * r.get("tokens_evaluated", 0)) for r in all_results)
    
    overall_base_correct_top5 = sum(round((r.get("base_top5_accuracy", 0.0) / 100.0) * r.get("tokens_evaluated", 0)) for r in all_results)
    overall_personalized_correct_top5 = sum(round((r.get("personalized_top5_accuracy", 0.0) / 100.0) * r.get("tokens_evaluated", 0)) for r in all_results)
    
    overall_base_total_loss = sum(r.get("base_loss", 0.0) * r.get("tokens_evaluated", 0) for r in all_results)
    overall_personalized_total_loss = sum(r.get("personalized_loss", 0.0) * r.get("tokens_evaluated", 0) for r in all_results)
    
    overall_base_accuracy_top1 = (overall_base_correct_top1 / max(total_tokens, 1)) * 100.0
    overall_personalized_accuracy_top1 = (overall_personalized_correct_top1 / max(total_tokens, 1)) * 100.0
    overall_accuracy_gain_top1 = overall_personalized_accuracy_top1 - overall_base_accuracy_top1
    
    overall_base_accuracy_top5 = (overall_base_correct_top5 / max(total_tokens, 1)) * 100.0
    overall_personalized_accuracy_top5 = (overall_personalized_correct_top5 / max(total_tokens, 1)) * 100.0
    overall_accuracy_gain_top5 = overall_personalized_accuracy_top5 - overall_base_accuracy_top5
    
    overall_base_ppl = math.exp(min(overall_base_total_loss / max(total_tokens, 1), 20))
    overall_personalized_ppl = math.exp(min(overall_personalized_total_loss / max(total_tokens, 1), 20))
    overall_ppl_improvement = overall_base_ppl - overall_personalized_ppl

    # ------------------------------------------------------
    # Summary
    # ------------------------------------------------------

    summary = {

        "total_clients_evaluated":
            len(df),

        "average_base_top1_accuracy":
            float(avg_base_accuracy),

        "average_personalized_top1_accuracy":
            float(avg_personalized_accuracy),

        "average_top1_accuracy_gain":
            float(avg_accuracy_gain),

        "average_base_perplexity":
            float(avg_base_ppl),

        "average_personalized_perplexity":
            float(avg_personalized_ppl),

        "average_perplexity_improvement":
            float(avg_ppl_improvement),

        "total_tokens_evaluated":
            int(total_tokens),

        "overall_base_top1_accuracy":
            float(overall_base_accuracy_top1),

        "overall_personalized_top1_accuracy":
            float(overall_personalized_accuracy_top1),

        "overall_top1_accuracy_gain":
            float(overall_accuracy_gain_top1),

        "overall_base_perplexity":
            float(overall_base_ppl),

        "overall_personalized_perplexity":
            float(overall_personalized_ppl),

        "overall_perplexity_improvement":
            float(overall_ppl_improvement)

    }

    summary_path = os.path.join(

        EVALUATION_PATH,

        "personalization_summary.json"

    )

    with open(
        summary_path,
        "w"
    ) as f:

        json.dump(

            summary,

            f,

            indent=4

        )

    # ======================================================
    # Display Final Table
    # ======================================================

    print("\n")

    print(
        "=" * 110
    )

    print(
        "PERSONALIZED EVALUATION RESULTS"
    )

    print(
        "=" * 110
    )

    display_columns = [

        "client",

        "active_lora_rank",

        "base_top1_accuracy",

        "personalized_top1_accuracy",

        "top1_accuracy_gain",

        "base_perplexity",

        "personalized_perplexity"

    ]

    print(
        df[
            display_columns
        ].to_string(
            index=False
        )
    )

    print(
        "=" * 110
    )

    print(
        "\n--- CLIENT-WISE AVERAGE METRICS ---"
    )

    print(
        "Average Base Top-1 Accuracy       : "
        f"{avg_base_accuracy:.2f}%"
    )

    print(
        "Average Personalized Top-1 Accuracy: "
        f"{avg_personalized_accuracy:.2f}%"
    )

    print(
        "Average Personalization Gain      : "
        f"{avg_accuracy_gain:+.2f}%"
    )

    print(
        "\nAverage Base Perplexity            : "
        f"{avg_base_ppl:.4f}"
    )

    print(
        "Average Personalized Perplexity   : "
        f"{avg_personalized_ppl:.4f}"
    )

    print(
        "Average Perplexity Improvement     : "
        f"{avg_ppl_improvement:+.4f}"
    )

    print(
        "\n--- OVERALL CONSOLIDATED METRICS (TOKEN-WEIGHTED) ---"
    )

    print(
        f"Total Evaluated Tokens             : {total_tokens}"
    )

    print(
        "Overall Base Top-1 Accuracy        : "
        f"{overall_base_accuracy_top1:.2f}%"
    )

    print(
        "Overall Personalized Top-1 Accuracy: "
        f"{overall_personalized_accuracy_top1:.2f}%"
    )

    print(
        "Overall Personalization Gain       : "
        f"{overall_accuracy_gain_top1:+.2f}%"
    )

    print(
        "\nOverall Base Perplexity            : "
        f"{overall_base_ppl:.4f}"
    )

    print(
        "Overall Personalized Perplexity   : "
        f"{overall_personalized_ppl:.4f}"
    )

    print(
        "Overall Perplexity Improvement     : "
        f"{overall_ppl_improvement:+.4f}"
    )

    print(
        "=" * 110
    )

    print(
        "\nCSV Saved    :",
        csv_path
    )

    print(
        "JSON Saved   :",
        json_path
    )

    print(
        "Summary Saved:",
        summary_path
    )

    return df


# ==========================================================
# Main
# ==========================================================

if __name__ == "__main__":

    results_df = (
        evaluate_all_clients()
    )

Loading DistilGPT2 Tokenizer

Total Clients Found : 11
Clients : ['client_1', 'client_2', 'client_3', 'client_4', 'client_5', 'client_6', 'client_7', 'client_8', 'client_9', 'client_10', 'client_11']


ADAPTIVE PEFT PERSONALIZED EVALUATION


###########################################################################
PERSONALIZED EVALUATION : client_1
###########################################################################
  [Quick Test] Subsampled test dataset to 7 samples.
Test Samples          : 7
Personalization Score : 0.3250
Active LoRA Rank      : 8


Loading Base DistilGPT2


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 7522.53it/s]



Evaluating : Base DistilGPT2


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Base DistilGPT2
Loss       : 7.1111
Perplexity : 1225.5477
Top-1 Acc  : 9.68%
Top-5 Acc  : 29.03%
Tokens     : 62


Loading Personalized Adapter: client_1


...

Applying inference rank 16/16

Evaluating : client_10 Adaptive PEFT
client_10 Adaptive PEFT
Loss       : 1.6062
Perplexity : 4.9838
Top-1 Acc  : 58.62%
Top-5 Acc  : 81.97%
Tokens     : 2540


client_10 RESULT
Base Loss              : 1.6070
Personalized Loss      : 1.6062
Base Perplexity        : 4.9876
Personalized Perplexity: 4.9838
Base Top-1 Accuracy    : 58.62%
Personalized Top-1    : 58.62%
Top-1 Accuracy Gain    : +0.00%
Base Top-5 Accuracy    : 82.01%
Personalized Top-5     : 81.97%
Top-5 Accuracy Gain    : -0.04%


###########################################################################
PERSONALIZED EVALUATION : client_11
###########################################################################
  [Quick Test] Subsampled test dataset to 20 samples.
Test Samples          : 20
Personalization Score : 1.0000
Active LoRA Rank      : 16


Loading Base DistilGPT2


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 6994.34it/s]



Evaluating : Base DistilGPT2
Base DistilGPT2
Loss       : 8.5355
Perplexity : 5092.2632
Top-1 Acc  : 3.68%
Top-5 Acc  : 9.20%
Tokens     : 163


Loading Personalized Adapter: client_11


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 9127.19it/s]


Applying inference rank 16/16

Evaluating : client_11 Adaptive PEFT
client_11 Adaptive PEFT
Loss       : 8.5345
Perplexity : 5087.1504
Top-1 Acc  : 3.68%
Top-5 Acc  : 9.20%
Tokens     : 163


client_11 RESULT
Base Loss              : 8.5355
Personalized Loss      : 8.5345
Base Perplexity        : 5092.2632
Personalized Perplexity: 5087.1504
Base Top-1 Accuracy    : 3.68%
Personalized Top-1    : 3.68%
Top-1 Accuracy Gain    : +0.00%
Base Top-5 Accuracy    : 9.20%
Personalized Top-5     : 9.20%
Top-5 Accuracy Gain    : +0.00%


PERSONALIZED EVALUATION RESULTS
   client  active_lora_rank  base_top1_accuracy  personalized_top1_accuracy  top1_accuracy_gain  base_perplexity  personalized_perplexity
 client_1                 8            9.677419                    9.677419            0.000000      1225.547684              1224.482810
 client_2                 8           10.769231                   10.769231            0.000000       663.020340               662.760197
 client_3             

## Module 8B: CAGER Router Training
Fine-tunes the CAGER v1 router on a test client.

In [12]:
# ==========================================================
# Module 8B
# Cross-Attention Gated Expert Router (CAGER)
#
# Adaptive PEFT + Cross-Attention + Gated Expert Routing
#
# Existing client adapters are treated as frozen experts.
# Only the lightweight router is trained.
# ==========================================================

import os
import json
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_from_disk
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling
)

from peft import PeftModel


# ==========================================================
# Configuration
# ==========================================================

MODEL_NAME = "distilgpt2"

import sys
try:
    MODULE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    MODULE_DIR = os.path.abspath(os.path.join(os.getcwd(), 'code', 'modules'))
if MODULE_DIR not in sys.path:
    sys.path.append(MODULE_DIR)
import local_config

CLIENT_PATH = local_config.CLIENT_PATH
ADAPTER_PATH = local_config.MODEL_SAVE_PATH
CAGER_SAVE_PATH = os.path.join(local_config.WORKSPACE_DIR, "cager_models")

os.makedirs(
    CAGER_SAVE_PATH,
    exist_ok=True
)

DEVICE = local_config.DEVICE


# ==========================================================
# CAGER Configuration
# ==========================================================

HIDDEN_SIZE = 768

ROUTER_HIDDEN_SIZE = 256

NUM_EXPERTS = 11

TOP_K = 3

TEMPERATURE = 1.0

ROUTER_EPOCHS = 1

ROUTER_BATCH_SIZE = 2

ROUTER_LEARNING_RATE = 1e-4


# ==========================================================
# Expert Clients
# ==========================================================

CLIENTS = sorted(

    [
        c
        for c in os.listdir(
            ADAPTER_PATH
        )
        if c.startswith("client_")
    ],

    key=lambda x: int(
        x.split("_")[1]
    )
)


if len(CLIENTS) != NUM_EXPERTS:

    raise ValueError(
        f"Expected {NUM_EXPERTS} experts, "
        f"but found {len(CLIENTS)}."
    )


EXPERT_NAMES = CLIENTS


print("=" * 70)
print("CAGER CONFIGURATION")
print("=" * 70)

print(
    f"Number of Experts : {NUM_EXPERTS}"
)

print(
    f"Top-K Experts     : {TOP_K}"
)

print(
    f"Device            : {DEVICE}"
)

print(
    "Experts            :",
    EXPERT_NAMES
)

print("=" * 70)


# ==========================================================
# Tokenizer
# ==========================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

tokenizer.pad_token = tokenizer.eos_token


data_collator = (
    DataCollatorForLanguageModeling(

        tokenizer=tokenizer,

        mlm=False
    )
)


# ==========================================================
# Load Adaptive Client Configuration
# ==========================================================

CONFIG_PATH = local_config.CONFIG_PATH


with open(
    CONFIG_PATH,
    "r"
) as f:

    CLIENT_CONFIG = json.load(f)


# ==========================================================
# Client Context Embeddings
# ==========================================================
#
# Each client gets a learnable context vector.
#
# Example:
#
# client_1  -> context vector
# client_2  -> context vector
# ...
#
# This allows the router to distinguish different
# personalization contexts.
# ==========================================================

CLIENT_TO_ID = {

    client: index

    for index, client
    in enumerate(CLIENTS)

}


# ==========================================================
# Cross-Attention Module
# ==========================================================

class ClientCrossAttention(
    nn.Module
):

    def __init__(
        self,
        hidden_size,
        context_size
    ):

        super().__init__()

        self.query = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.key = nn.Linear(
            context_size,
            hidden_size
        )

        self.value = nn.Linear(
            context_size,
            hidden_size
        )

        self.scale = math.sqrt(
            hidden_size
        )


    def forward(
        self,
        hidden_states,
        client_context
    ):

        # --------------------------------------------------
        # Query
        # --------------------------------------------------

        q = self.query(
            hidden_states
        )

        # --------------------------------------------------
        # Key
        # --------------------------------------------------

        k = self.key(
            client_context
        )

        # --------------------------------------------------
        # Value
        # --------------------------------------------------

        v = self.value(
            client_context
        )

        # --------------------------------------------------
        # Cross-attention score
        # --------------------------------------------------

        scores = torch.matmul(

            q,

            k.transpose(
                -1,
                -2
            )

        ) / self.scale


        attention = F.softmax(
            scores,
            dim=-1
        )


        attended = torch.matmul(
            attention,
            v
        )


        return attended


# ==========================================================
# Gated Expert Router
# ==========================================================

class GatedExpertRouter(
    nn.Module
):

    def __init__(
        self,
        hidden_size,
        num_experts,
        router_hidden_size
    ):

        super().__init__()

        self.router = nn.Sequential(

            nn.Linear(
                hidden_size * 2,
                router_hidden_size
            ),

            nn.ReLU(),

            nn.Linear(
                router_hidden_size,
                num_experts
            )

        )


    def forward(
        self,
        hidden_states,
        cross_attention_output
    ):

        # --------------------------------------------------
        # Mean pooling of token representations
        # --------------------------------------------------

        pooled_hidden = (
            hidden_states.mean(
                dim=1
            )
        )


        pooled_cross = (
            cross_attention_output.mean(
                dim=1
            )
        )


        # --------------------------------------------------
        # Combine original representation
        # with cross-attended client context
        # --------------------------------------------------

        router_input = torch.cat(

            [
                pooled_hidden,
                pooled_cross
            ],

            dim=-1

        )


        # --------------------------------------------------
        # Expert logits
        # --------------------------------------------------

        expert_logits = self.router(
            router_input
        )


        # --------------------------------------------------
        # Gating probabilities
        # --------------------------------------------------

        gates = F.softmax(

            expert_logits
            / TEMPERATURE,

            dim=-1

        )


        return gates


# ==========================================================
# Complete CAGER Router
# ==========================================================

class CAGER(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()


        # --------------------------------------------------
        # Client context
        # --------------------------------------------------

        self.client_context = (
            nn.Embedding(

                NUM_EXPERTS,

                HIDDEN_SIZE

            )
        )


        # --------------------------------------------------
        # Cross attention
        # --------------------------------------------------

        self.cross_attention = (
            ClientCrossAttention(

                HIDDEN_SIZE,

                HIDDEN_SIZE

            )
        )


        # --------------------------------------------------
        # Expert router
        # --------------------------------------------------

        self.expert_router = (
            GatedExpertRouter(

                HIDDEN_SIZE,

                NUM_EXPERTS,

                ROUTER_HIDDEN_SIZE

            )
        )


    def forward(

        self,
        hidden_states,
        client_ids

    ):

        # --------------------------------------------------
        # Get client context
        # --------------------------------------------------

        context = (
            self.client_context(
                client_ids
            )
        )


        # --------------------------------------------------
        # Convert to sequence context
        # --------------------------------------------------

        context = context.unsqueeze(
            1
        )


        # --------------------------------------------------
        # Cross attention
        # --------------------------------------------------

        attended = (
            self.cross_attention(

                hidden_states,

                context

            )
        )


        # --------------------------------------------------
        # Gated expert routing
        # --------------------------------------------------

        gates = (
            self.expert_router(

                hidden_states,

                attended

            )
        )


        return gates


# ==========================================================
# Load Expert Model
# ==========================================================
#
# One base DistilGPT2 is loaded.
#
# Each saved client adapter is loaded into the same
# PEFT model as a separate adapter.
#
# These adapters are FROZEN.
# ==========================================================

def load_expert_model():

    print("\n")
    print("=" * 70)
    print("Loading Expert Model")
    print("=" * 70)


    base_model = (
        AutoModelForCausalLM
        .from_pretrained(
            MODEL_NAME
        )
    )


    # ------------------------------------------------------
    # Load first adapter
    # ------------------------------------------------------

    first_client = EXPERT_NAMES[0]

    first_adapter_path = os.path.join(

        ADAPTER_PATH,

        first_client

    )


    model = PeftModel.from_pretrained(

        base_model,

        first_adapter_path,

        is_trainable=False

    )


    # ------------------------------------------------------
    # Load remaining adapters
    # ------------------------------------------------------

    for client in EXPERT_NAMES[1:]:

        adapter_path = os.path.join(

            ADAPTER_PATH,

            client

        )


        print(
            f"Loading expert adapter: "
            f"{client}"
        )


        model.load_adapter(

            adapter_path,

            adapter_name=client

        )


    # ------------------------------------------------------
    # Rename first adapter
    # ------------------------------------------------------

    #
    # The first adapter was loaded using "default".
    #

    print(
        "Expert adapters loaded."
    )


    # ------------------------------------------------------
    # Freeze ALL expert parameters
    # ------------------------------------------------------

    for parameter in model.parameters():

        parameter.requires_grad = False


    model.to(
        DEVICE
    )


    model.eval()


    return model


# ==========================================================
# Get Expert Logits
# ==========================================================
#
# Each expert generates next-token logits.
#
# The router later combines these logits.
# ==========================================================

def get_expert_logits(

    model,

    input_ids,

    attention_mask

):

    expert_logits = []


    for index, client in enumerate(
        EXPERT_NAMES
    ):

        # --------------------------------------------------
        # First adapter
        # --------------------------------------------------

        if index == 0:

            adapter_name = "default"

        else:

            adapter_name = client


        # --------------------------------------------------
        # Activate expert
        # --------------------------------------------------

        model.set_adapter(
            adapter_name
        )


        with torch.no_grad():

            outputs = model(

                input_ids=input_ids,

                attention_mask=attention_mask

            )


        expert_logits.append(
            outputs.logits
        )


    # ------------------------------------------------------
    # Stack expert predictions
    # ------------------------------------------------------

    #
    # Shape:
    #
    # [batch, experts, sequence, vocabulary]
    #

    stacked_logits = torch.stack(

        expert_logits,

        dim=1

    )


    return stacked_logits


# ==========================================================
# Top-K Gating
# ==========================================================

def apply_top_k_gating(
    gates
):

    # ------------------------------------------------------
    # Find top-K experts
    # ------------------------------------------------------

    top_values, top_indices = torch.topk(

        gates,

        k=TOP_K,

        dim=-1

    )


    # ------------------------------------------------------
    # Normalize selected gates
    # ------------------------------------------------------

    top_values = (

        top_values
        /
        (
            top_values.sum(
                dim=-1,
                keepdim=True
            )
            + 1e-8
        )

    )


    sparse_gates = torch.zeros_like(
        gates
    )


    sparse_gates.scatter_(
        -1,
        top_indices,
        top_values
    )


    return (
        sparse_gates,
        top_indices
    )


# ==========================================================
# Combine Expert Predictions
# ==========================================================

def combine_expert_logits(

    expert_logits,

    gates

):

    # ------------------------------------------------------
    # gates:
    #
    # [batch, experts]
    #
    # expert_logits:
    #
    # [batch, experts, seq, vocab]
    # ------------------------------------------------------

    gates = gates.unsqueeze(
        -1
    ).unsqueeze(
        -1
    )


    combined_logits = (
        expert_logits
        * gates
    ).sum(
        dim=1
    )


    return combined_logits


# ==========================================================
# Load Client Training Dataset
# ==========================================================

def load_client_dataset(
    client_name
):

    path = os.path.join(

        CLIENT_PATH,

        client_name,

        "train"

    )


    dataset = load_from_disk(
        path
    )


    if "text" in dataset.column_names:

        dataset = dataset.remove_columns(
            ["text"]
        )


    return dataset


# ==========================================================
# CAGER Trainer
# ==========================================================

class CAGERTrainer:

    def __init__(
        self,
        expert_model,
        router
    ):

        self.expert_model = (
            expert_model
        )

        self.router = router.to(
            DEVICE
        )


        # --------------------------------------------------
        # Only router parameters train
        # --------------------------------------------------

        self.optimizer = torch.optim.AdamW(

            self.router.parameters(),

            lr=ROUTER_LEARNING_RATE

        )


    def train_client(
        self,
        client_name
    ):

        print("\n")
        print("=" * 70)

        print(
            f"CAGER Training : "
            f"{client_name}"
        )

        print("=" * 70)


        # --------------------------------------------------
        # Dataset
        # --------------------------------------------------

        dataset = load_client_dataset(
            client_name
        )


        loader = DataLoader(

            dataset,

            batch_size=ROUTER_BATCH_SIZE,

            shuffle=True,

            collate_fn=data_collator

        )


        client_id = torch.tensor(

            [
                CLIENT_TO_ID[
                    client_name
                ]
            ],

            dtype=torch.long,

            device=DEVICE

        )


        total_loss = 0.0


        # --------------------------------------------------
        # Train router
        # --------------------------------------------------

        self.router.train()


        for epoch in range(
            ROUTER_EPOCHS
        ):

            for batch in loader:

                input_ids = (
                    batch[
                        "input_ids"
                    ].to(DEVICE)
                )

                attention_mask = (
                    batch[
                        "attention_mask"
                    ].to(DEVICE)
                )

                labels = (
                    batch[
                        "labels"
                    ].to(DEVICE)
                )


                # --------------------------------------------------
                # Get hidden representation
                # --------------------------------------------------

                with torch.no_grad():

                    # Use first expert to obtain hidden states.
                    #
                    # All experts share DistilGPT2 base architecture.

                    self.expert_model.set_adapter(
                        "default"
                    )

                    base_outputs = (
                        self.expert_model.base_model.model(
                            input_ids=input_ids,
                            attention_mask=attention_mask,
                            output_hidden_states=True
                        )
                    )


                    hidden_states = (
                        base_outputs.hidden_states[-1]
                    )


                    # --------------------------------------------------
                    # Expert predictions
                    # --------------------------------------------------

                    expert_logits = (
                        get_expert_logits(

                            self.expert_model,

                            input_ids,

                            attention_mask

                        )
                    )


                # --------------------------------------------------
                # Client IDs
                # --------------------------------------------------

                batch_client_ids = (
                    client_id.expand(
                        input_ids.size(0)
                    )
                )


                # --------------------------------------------------
                # Router
                # --------------------------------------------------

                gates = self.router(

                    hidden_states,

                    batch_client_ids

                )


                # --------------------------------------------------
                # Top-K experts
                # --------------------------------------------------

                sparse_gates, top_indices = (
                    apply_top_k_gating(
                        gates
                    )
                )


                # --------------------------------------------------
                # Combine expert predictions
                # --------------------------------------------------

                combined_logits = (
                    combine_expert_logits(

                        expert_logits,

                        sparse_gates

                    )
                )


                # --------------------------------------------------
                # Causal LM shift
                # --------------------------------------------------

                shift_logits = (
                    combined_logits[
                        :, :-1, :
                    ]
                )

                shift_labels = (
                    labels[
                        :, 1:
                    ]
                )


                # --------------------------------------------------
                # Cross entropy
                # --------------------------------------------------

                loss = F.cross_entropy(

                    shift_logits.reshape(
                        -1,
                        shift_logits.size(-1)
                    ),

                    shift_labels.reshape(
                        -1
                    ),

                    ignore_index=-100

                )


                # --------------------------------------------------
                # Backpropagation
                # --------------------------------------------------

                self.optimizer.zero_grad()

                loss.backward()

                self.optimizer.step()


                total_loss += (
                    loss.item()
                )


        average_loss = (
            total_loss
            /
            max(
                len(loader),
                1
            )
        )


        print(
            f"CAGER Router Loss : "
            f"{average_loss:.4f}"
        )


        return average_loss


# ==========================================================
# Save Router
# ==========================================================

def save_router(
    router,
    client_name
):

    save_path = os.path.join(

        CAGER_SAVE_PATH,

        client_name

    )


    os.makedirs(

        save_path,

        exist_ok=True

    )


    router_path = os.path.join(

        save_path,

        "cager_router.pt"

    )


    torch.save(

        router.state_dict(),

        router_path

    )


    print(
        f"CAGER Router Saved : "
        f"{router_path}"
    )


# ==========================================================
# Main
# ==========================================================

if __name__ == "__main__":

    # ------------------------------------------------------
    # Load frozen expert pool
    # ------------------------------------------------------

    expert_model = (
        load_expert_model()
    )


    print("\n" + "=" * 70)
    print("TRAINING CAGER ROUTERS FOR ALL CLIENTS")
    print("=" * 70)

    for client_name in CLIENTS:
        router = CAGER()
        trainer = CAGERTrainer(expert_model, router)
        loss = trainer.train_client(client_name)
        save_router(router, client_name)

    print("\n" + "=" * 70)
    print("CAGER ALL CLIENTS TRAINING COMPLETED")
    print("=" * 70)

CAGER CONFIGURATION
Number of Experts : 11
Top-K Experts     : 3
Device            : cpu
Experts            : ['client_1', 'client_2', 'client_3', 'client_4', 'client_5', 'client_6', 'client_7', 'client_8', 'client_9', 'client_10', 'client_11']


Loading Expert Model


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 6352.35it/s]


Loading expert adapter: client_2
Loading expert adapter: client_3
Loading expert adapter: client_4
Loading expert adapter: client_5
Loading expert adapter: client_6
Loading expert adapter: client_7
Loading expert adapter: client_8
Loading expert adapter: client_9
Loading expert adapter: client_10
Loading expert adapter: client_11
Expert adapters loaded.

TRAINING CAGER ROUTERS FOR ALL CLIENTS


CAGER Training : client_1
CAGER Router Loss : 6.5018
CAGER Router Saved : d:/project\cager_models\client_1\cager_router.pt


CAGER Training : client_2
CAGER Router Loss : 6.6940
CAGER Router Saved : d:/project\cager_models\client_2\cager_router.pt


CAGER Training : client_3
CAGER Router Loss : 7.1688
CAGER Router Saved : d:/project\cager_models\client_3\cager_router.pt


CAGER Training : client_4
CAGER Router Loss : 1.5460
CAGER Router Saved : d:/project\cager_models\client_4\cager_router.pt


CAGER Training : client_5
CAGER Router Loss : 1.5621
CAGER Router Saved : d:/project\cager_models\clie

## Module 8C: CAGER Router Evaluation
Evaluates CAGER v1 routing gates against Adaptive PEFT.

In [13]:
# ==========================================================
# Module 8C
# CAGER Evaluation (All Clients)
#
# Evaluates:
#   Adaptive PEFT + Cross-Attention Gated Expert Router
#   Runs evaluation over all 11 clients.
# ==========================================================

import os
import json
import math
import sys
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_from_disk
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling
)
from peft import PeftModel

# Load Config
try:
    MODULE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    MODULE_DIR = os.path.abspath(os.path.join(os.getcwd(), 'code', 'modules'))
if MODULE_DIR not in sys.path:
    sys.path.append(MODULE_DIR)
import local_config

MODEL_NAME = "distilgpt2"
CLIENT_PATH = local_config.CLIENT_PATH
ADAPTER_PATH = local_config.MODEL_SAVE_PATH
CAGER_PATH = os.path.join(local_config.WORKSPACE_DIR, "cager_models")
RESULT_PATH = os.path.join(local_config.WORKSPACE_DIR, "cager_evaluation")

os.makedirs(RESULT_PATH, exist_ok=True)

DEVICE = local_config.DEVICE
HIDDEN_SIZE = 768
ROUTER_HIDDEN_SIZE = 256
NUM_EXPERTS = 11
TOP_K = 3
TEMPERATURE = 1.0
BATCH_SIZE = 4

CLIENTS = [f"client_{i}" for i in range(1, 12)]
CLIENT_TO_ID = {client: index for index, client in enumerate(CLIENTS)}

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# CAGER Classes
class ClientCrossAttention(torch.nn.Module):
    def __init__(self, hidden_size, context_size):
        super().__init__()
        self.query = torch.nn.Linear(hidden_size, hidden_size)
        self.key = torch.nn.Linear(context_size, hidden_size)
        self.value = torch.nn.Linear(context_size, hidden_size)
        self.scale = math.sqrt(hidden_size)

    def forward(self, hidden_states, client_context):
        # hidden_states: [batch, seq, hidden]
        # client_context: [batch, 1, context]
        q = self.query(hidden_states)
        k = self.key(client_context)
        v = self.value(client_context)
        
        # [batch, seq, 1]
        scores = torch.matmul(q, k.transpose(-2, -1)) / self.scale
        attn = F.softmax(scores, dim=-1)
        # [batch, seq, hidden]
        attended = attn * v
        return attended

class GatedExpertRouter(torch.nn.Module):
    def __init__(self, hidden_size, num_experts, router_hidden_size):
        super().__init__()
        self.router = torch.nn.Sequential(
            torch.nn.Linear(hidden_size * 2, router_hidden_size),
            torch.nn.GELU(),
            torch.nn.Linear(router_hidden_size, num_experts)
        )

    def forward(self, hidden_states, cross_attention_output):
        pooled_hidden = hidden_states.mean(dim=1)
        pooled_cross = cross_attention_output.mean(dim=1)
        router_input = torch.cat([pooled_hidden, pooled_cross], dim=-1)
        expert_logits = self.router(router_input)
        gates = F.softmax(expert_logits / TEMPERATURE, dim=-1)
        return gates

class CAGER(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.client_context = torch.nn.Embedding(NUM_EXPERTS, HIDDEN_SIZE)
        self.cross_attention = ClientCrossAttention(HIDDEN_SIZE, HIDDEN_SIZE)
        self.expert_router = GatedExpertRouter(HIDDEN_SIZE, NUM_EXPERTS, ROUTER_HIDDEN_SIZE)

    def forward(self, hidden_states, client_ids):
        context = self.client_context(client_ids).unsqueeze(1)
        attended = self.cross_attention(hidden_states, context)
        gates = self.expert_router(hidden_states, attended)
        return gates

# Load All 11 Frozen Expert Adapters
print("\n" + "=" * 70)
print("Loading Frozen Expert Adapters")
print("=" * 70)

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
first_client = CLIENTS[0]
first_adapter_path = os.path.join(ADAPTER_PATH, first_client)

expert_model = PeftModel.from_pretrained(
    base_model,
    first_adapter_path,
    is_trainable=False
)

for client in CLIENTS[1:]:
    adapter_path = os.path.join(ADAPTER_PATH, client)
    print(f"Loading expert: {client}")
    expert_model.load_adapter(adapter_path, adapter_name=client)

for parameter in expert_model.parameters():
    parameter.requires_grad = False

expert_model.to(DEVICE)
expert_model.eval()
print("\nAll 11 experts loaded and frozen.")

# Helper Functions
def get_all_expert_logits(input_ids, attention_mask):
    expert_logits = []
    for index, client in enumerate(CLIENTS):
        adapter_name = "default" if index == 0 else client
        expert_model.set_adapter(adapter_name)
        with torch.no_grad():
            outputs = expert_model(input_ids=input_ids, attention_mask=attention_mask)
        expert_logits.append(outputs.logits)
    return torch.stack(expert_logits, dim=1)

def top_k_gates(gates):
    top_values, top_indices = torch.topk(gates, k=TOP_K, dim=-1)
    top_values = top_values / (top_values.sum(dim=-1, keepdim=True) + 1e-8)
    sparse_gates = torch.zeros_like(gates)
    sparse_gates.scatter_(-1, top_indices, top_values)
    return sparse_gates, top_indices, top_values

def evaluate_client(client_name):
    print("\n" + "=" * 70)
    print(f"Running CAGER Evaluation : {client_name}")
    print("=" * 70)

    # 1. Dataset loading
    test_path = os.path.join(CLIENT_PATH, client_name, "test")
    if not os.path.exists(test_path):
        raise FileNotFoundError(f"Test dataset not found: {test_path}")
    test_dataset = load_from_disk(test_path)
    if "text" in test_dataset.column_names:
        test_dataset = test_dataset.remove_columns(["text"])
    if local_config.QUICK_TEST:
        test_dataset = test_dataset.select(range(min(10, len(test_dataset))))
        print(f"  [Quick Test] Subsampled test dataset to {len(test_dataset)} samples.")
    print(f"Test Samples : {len(test_dataset)}")

    # 2. Loader
    loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=data_collator
    )

    # 3. Load Router
    router_path = os.path.join(CAGER_PATH, client_name, "cager_router.pt")
    if not os.path.exists(router_path):
        raise FileNotFoundError(f"CAGER router not found: {router_path}. Run Module 8B first.")
    
    router = CAGER().to(DEVICE)
    router.load_state_dict(torch.load(router_path, map_location=DEVICE))
    router.eval()
    print("CAGER Router Loaded Successfully.")

    # 4. Evaluation variables
    total_loss = 0.0
    total_batches = 0
    top1_correct = 0
    top5_correct = 0
    total_tokens = 0

    expert_selection_count = {c: 0 for c in CLIENTS}
    gate_sum = torch.zeros(NUM_EXPERTS, device=DEVICE)

    # 5. Run Evaluation Loop
    client_id = torch.tensor([CLIENT_TO_ID[client_name]], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        for batch_number, batch in enumerate(loader):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            expert_model.set_adapter("default")
            hidden_outputs = expert_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
            hidden_states = hidden_outputs.hidden_states[-1]

            batch_client_ids = client_id.expand(input_ids.size(0))
            gates = router(hidden_states, batch_client_ids)
            sparse_gates, top_indices, top_values = top_k_gates(gates)

            # Combine logits
            expert_logits = get_all_expert_logits(input_ids, attention_mask)
            combined_logits = (expert_logits * sparse_gates.unsqueeze(-1).unsqueeze(-1)).sum(dim=1)

            # Shift
            shift_logits = combined_logits[:, :-1, :]
            shift_labels = labels[:, 1:]

            # Loss
            loss = F.cross_entropy(
                shift_logits.reshape(-1, shift_logits.size(-1)),
                shift_labels.reshape(-1),
                ignore_index=-100
            )
            total_loss += loss.item()
            total_batches += 1

            # Accuracy
            valid_mask = shift_labels != -100
            total_tokens += valid_mask.sum().item()

            predictions = torch.argmax(shift_logits, dim=-1)
            top1_correct += ((predictions == shift_labels) & valid_mask).sum().item()

            top5_predictions = torch.topk(shift_logits, k=5, dim=-1).indices
            expanded_labels = shift_labels.unsqueeze(-1).expand_as(top5_predictions)
            top5_correct += ((top5_predictions == expanded_labels).any(dim=-1) & valid_mask).sum().item()

            # Tracking selection
            gate_sum += gates.sum(dim=0)
            for seq_indices in top_indices:
                for idx in seq_indices:
                    expert_selection_count[CLIENTS[idx.item()]] += 1

    # 6. Calculate Average Metrics
    average_loss = total_loss / max(total_batches, 1)
    perplexity = math.exp(min(average_loss, 20))
    top1_accuracy = (top1_correct / max(total_tokens, 1)) * 100
    top5_accuracy = (top5_correct / max(total_tokens, 1)) * 100

    print("\n" + "=" * 70)
    print(f"CAGER RESULTS : {client_name}")
    print("=" * 70)
    print(f"Loss             : {average_loss:.4f}")
    print(f"Perplexity       : {perplexity:.4f}")
    print(f"Top-1 Accuracy   : {top1_accuracy:.2f}%")
    print(f"Top-5 Accuracy   : {top5_accuracy:.2f}%")
    print(f"Tokens Evaluated : {total_tokens}")
    print("=" * 70)

    # Expert Gate selection
    average_gates = gate_sum / max(gate_sum.sum(), 1)
    print("\n" + "=" * 70)
    print("EXPERT SELECTION DISTRIBUTION")
    print("=" * 70)
    for index, c in enumerate(CLIENTS):
        print(f"{c:<12}Gate={average_gates[index].item():.4f}    Top-K Count={expert_selection_count[c]}")
    print("=" * 70)

    # 7. Baseline PEFT comparison
    baseline_csv = "personalized_evaluation/personalized_evaluation.csv"
    baseline_top1 = None
    baseline_top5 = None
    baseline_ppl = None

    if os.path.exists(baseline_csv):
        baseline_df = pd.read_csv(baseline_csv)
        baseline_row = baseline_df[baseline_df["client"] == client_name]
        if len(baseline_row) > 0:
            baseline_top1 = float(baseline_row["personalized_top1_accuracy"].iloc[0])
            baseline_top5 = float(baseline_row["personalized_top5_accuracy"].iloc[0])
            baseline_ppl = float(baseline_row["personalized_perplexity"].iloc[0])

    if baseline_top1 is not None:
        top1_gain = top1_accuracy - baseline_top1
        top5_gain = top5_accuracy - baseline_top5
        ppl_change = baseline_ppl - perplexity

        print("\n" + "=" * 70)
        print("ADAPTIVE PEFT vs CAGER")
        print("=" * 70)
        print(f"Adaptive PEFT Top-1 : {baseline_top1:.2f}%")
        print(f"CAGER Top-1        : {top1_accuracy:.2f}%")
        print(f"Top-1 Gain         : {top1_gain:+.2f}%")
        print()
        print(f"Adaptive PEFT Top-5 : {baseline_top5:.2f}%")
        print(f"CAGER Top-5        : {top5_accuracy:.2f}%")
        print(f"Top-5 Gain         : {top5_gain:+.2f}%")
        print()
        print(f"Adaptive PEFT PPL   : {baseline_ppl:.4f}")
        print(f"CAGER PPL          : {perplexity:.4f}")
        print(f"PPL Improvement    : {ppl_change:+.4f}")
        print("=" * 70)

    # 8. Save JSON
    result = {
        "client": client_name,
        "loss": float(average_loss),
        "perplexity": float(perplexity),
        "top1_accuracy": float(top1_accuracy),
        "top5_accuracy": float(top5_accuracy),
        "tokens": int(total_tokens),
        "expert_gate_distribution": {
            c: float(average_gates[idx].item()) for idx, c in enumerate(CLIENTS)
        },
        "expert_selection_count": expert_selection_count,
        "adaptive_peft_top1": baseline_top1,
        "adaptive_peft_top5": baseline_top5,
        "adaptive_peft_perplexity": baseline_ppl
    }

    json_path = os.path.join(RESULT_PATH, f"{client_name}_cager_results.json")
    with open(json_path, "w") as f:
        json.dump(result, f, indent=4)
    print(f"\nResults Saved : {json_path}")

def compile_overall_consolidated_metrics():
    all_results = []
    for client_name in CLIENTS:
        json_path = os.path.join(RESULT_PATH, f"{client_name}_cager_results.json")
        if os.path.exists(json_path):
            with open(json_path, "r") as f:
                all_results.append(json.load(f))
                
    if len(all_results) == 0:
        return
        
    total_tokens = sum(r.get("tokens", 0) for r in all_results)
    overall_correct_top1 = sum(round((r.get("top1_accuracy", 0.0) / 100.0) * r.get("tokens", 0)) for r in all_results)
    overall_correct_top5 = sum(round((r.get("top5_accuracy", 0.0) / 100.0) * r.get("tokens", 0)) for r in all_results)
    overall_total_loss = sum(r.get("loss", 0.0) * r.get("tokens", 0) for r in all_results)
    
    overall_accuracy_top1 = (overall_correct_top1 / max(total_tokens, 1)) * 100.0
    overall_accuracy_top5 = (overall_correct_top5 / max(total_tokens, 1)) * 100.0
    overall_ppl = math.exp(min(overall_total_loss / max(total_tokens, 1), 20))
    
    # Baseline PEFT
    overall_peft_correct_top1 = sum(round((r.get("adaptive_peft_top1", 0.0) / 100.0) * r.get("tokens", 0)) for r in all_results if r.get("adaptive_peft_top1") is not None)
    overall_peft_correct_top5 = sum(round((r.get("adaptive_peft_top5", 0.0) / 100.0) * r.get("tokens", 0)) for r in all_results if r.get("adaptive_peft_top5") is not None)
    overall_peft_total_loss = sum(math.log(r.get("adaptive_peft_perplexity", 1.0)) * r.get("tokens", 0) for r in all_results if r.get("adaptive_peft_perplexity") is not None)
    
    overall_peft_accuracy_top1 = (overall_peft_correct_top1 / max(total_tokens, 1)) * 100.0
    overall_peft_accuracy_top5 = (overall_peft_correct_top5 / max(total_tokens, 1)) * 100.0
    overall_peft_ppl = math.exp(min(overall_peft_total_loss / max(total_tokens, 1), 20))
    
    print("\n" + "=" * 70)
    print("CAGER v1 OVERALL CONSOLIDATED METRICS (TOKEN-WEIGHTED)")
    print("=" * 70)
    print(f"Total Evaluated Tokens             : {total_tokens}")
    print(f"Overall Adaptive PEFT Top-1 Acc    : {overall_peft_accuracy_top1:.2f}%")
    print(f"Overall CAGER v1 Top-1 Acc         : {overall_accuracy_top1:.2f}%")
    print(f"Overall Top-1 Gain                 : {overall_accuracy_top1 - overall_peft_accuracy_top1:+.2f}%")
    print()
    print(f"Overall Adaptive PEFT Top-5 Acc    : {overall_peft_accuracy_top5:.2f}%")
    print(f"Overall CAGER v1 Top-5 Acc         : {overall_accuracy_top5:.2f}%")
    print(f"Overall Top-5 Gain                 : {overall_accuracy_top5 - overall_peft_accuracy_top5:+.2f}%")
    print()
    print(f"Overall Adaptive PEFT Perplexity   : {overall_peft_ppl:.4f}")
    print(f"Overall CAGER v1 Perplexity        : {overall_ppl:.4f}")
    print(f"Overall Perplexity Improvement     : {overall_peft_ppl - overall_ppl:+.4f}")
    print("=" * 70)

if __name__ == "__main__":
    print("\n" + "=" * 70)
    print("STARTING ALL CLIENTS CAGER v1 EVALUATION")
    print("=" * 70)
    for client in CLIENTS:
        evaluate_client(client)
    print("\n" + "=" * 70)
    print("CAGER v1 ALL CLIENTS EVALUATION COMPLETED")
    print("=" * 70)
    compile_overall_consolidated_metrics()


Loading Frozen Expert Adapters


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 6778.67it/s]


Loading expert: client_2
Loading expert: client_3
Loading expert: client_4
Loading expert: client_5
Loading expert: client_6
Loading expert: client_7
Loading expert: client_8
Loading expert: client_9
Loading expert: client_10
Loading expert: client_11

All 11 experts loaded and frozen.

STARTING ALL CLIENTS CAGER v1 EVALUATION

Running CAGER Evaluation : client_1
  [Quick Test] Subsampled test dataset to 7 samples.
Test Samples : 7
CAGER Router Loaded Successfully.

CAGER RESULTS : client_1
Loss             : 7.1112
Perplexity       : 1225.5678
Top-1 Accuracy   : 9.68%
Top-5 Accuracy   : 29.03%
Tokens Evaluated : 62

EXPERT SELECTION DISTRIBUTION
client_1    Gate=0.0003    Top-K Count=0
client_2    Gate=0.0019    Top-K Count=0
client_3    Gate=0.9762    Top-K Count=7
client_4    Gate=0.0047    Top-K Count=7
client_5    Gate=0.0023    Top-K Count=0
client_6    Gate=0.0031    Top-K Count=0
client_7    Gate=0.0016    Top-K Count=0
client_8    Gate=0.0052    Top-K Count=7
client_9    Gate=

## Module 8B-v2: Shared CAGER Router Training
Trains the shared CAGER v2 router across all clients.

In [14]:
# ==========================================================
# Module 8B-v2
# Shared Multi-Client Cross-Attention Gated Expert Router
#
# CAGER v2
#
# Frozen Experts:
#   client_1  -> LoRA rank 16
#   client_2  -> LoRA rank 16
#   ...
#   client_11 -> LoRA rank 4
#
# Trainable:
#   Client context
#   Cross-attention
#   Gated router
#
# ==========================================================

import os
import json
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_from_disk
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling
)

from peft import PeftModel


# ==========================================================
# 1. Paths
# ==========================================================

import sys
try:
    MODULE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    MODULE_DIR = os.path.abspath(os.path.join(os.getcwd(), 'code', 'modules'))
if MODULE_DIR not in sys.path:
    sys.path.append(MODULE_DIR)
import local_config

CLIENT_PATH = local_config.CLIENT_PATH
ADAPTER_PATH = local_config.MODEL_SAVE_PATH
CONFIG_PATH = local_config.CONFIG_PATH
SAVE_PATH = os.path.join(local_config.WORKSPACE_DIR, "cager_v2_models")

os.makedirs(
    SAVE_PATH,
    exist_ok=True
)


# ==========================================================
# 2. Configuration
# ==========================================================

MODEL_NAME = "distilgpt2"

DEVICE = local_config.DEVICE


# ----------------------------------------------------------
# CAGER
# ----------------------------------------------------------

HIDDEN_SIZE = 768

ROUTER_SIZE = 256

NUM_HEADS = 4

NUM_EXPERTS = 11

TOP_K = 3


# ----------------------------------------------------------
# Calibration
# ----------------------------------------------------------

# Start small.
# We can increase this after the architecture works.

CALIBRATION_SAMPLES = 10 if local_config.QUICK_TEST else 200
BATCH_SIZE = 2 if local_config.QUICK_TEST else 8
NUM_EPOCHS = 1
LEARNING_RATE = 1e-4

TARGET_TEMPERATURE = 0.5

BALANCE_WEIGHT = 0.005


# ==========================================================
# 3. Clients
# ==========================================================

CLIENTS = sorted(

    [
        c
        for c in os.listdir(
            ADAPTER_PATH
        )
        if c.startswith("client_")
    ],

    key=lambda x: int(
        x.split("_")[1]
    )

)


if len(CLIENTS) != NUM_EXPERTS:

    raise ValueError(

        f"Expected {NUM_EXPERTS} experts, "
        f"but found {len(CLIENTS)}"

    )


CLIENT_TO_ID = {

    client: i

    for i, client in enumerate(
        CLIENTS
    )

}


print("=" * 70)

print(
    "CAGER v2 CONFIGURATION"
)

print("=" * 70)

print(
    f"Experts              : {NUM_EXPERTS}"
)

print(
    f"Top-K                : {TOP_K}"
)

print(
    f"Calibration Samples  : "
    f"{CALIBRATION_SAMPLES} / client"
)

print(
    f"Batch Size           : {BATCH_SIZE}"
)

print(
    f"Device               : {DEVICE}"
)

print(
    "Clients              :",
    CLIENTS
)

print("=" * 70)


# ==========================================================
# 4. Adaptive Client Configuration
# ==========================================================

if not os.path.exists(
    CONFIG_PATH
):

    raise FileNotFoundError(

        f"Adaptive configuration not found:\n"
        f"{CONFIG_PATH}"

    )


with open(
    CONFIG_PATH,
    "r"
) as f:

    CLIENT_CONFIG = json.load(f)


# ==========================================================
# 5. Tokenizer
# ==========================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

tokenizer.pad_token = tokenizer.eos_token


data_collator = (
    DataCollatorForLanguageModeling(

        tokenizer=tokenizer,

        mlm=False

    )
)


# ==========================================================
# 6. Load Frozen Base Model
#
# Used only for generic input representations.
# ==========================================================

print("\nLoading frozen base model...")

base_model = (
    AutoModelForCausalLM
    .from_pretrained(
        MODEL_NAME
    )
)

base_model.to(
    DEVICE
)

base_model.eval()


for parameter in base_model.parameters():

    parameter.requires_grad = False


print(
    "Base model loaded."
)


# ==========================================================
# 7. Load All Frozen Expert Adapters
# ==========================================================

print("\n")

print("=" * 70)

print(
    "Loading Frozen Expert Pool"
)

print("=" * 70)


# ----------------------------------------------------------
# First expert
# ----------------------------------------------------------

first_client = CLIENTS[0]

first_adapter_path = os.path.join(

    ADAPTER_PATH,

    first_client

)


expert_model = (
    PeftModel.from_pretrained(

        base_model,

        first_adapter_path,

        is_trainable=False

    )
)


# ----------------------------------------------------------
# Remaining experts
# ----------------------------------------------------------

for client in CLIENTS[1:]:

    adapter_path = os.path.join(

        ADAPTER_PATH,

        client

    )


    print(
        f"Loading expert: "
        f"{client}"
    )


    expert_model.load_adapter(

        adapter_path,

        adapter_name=client

    )


# ----------------------------------------------------------
# Freeze experts
# ----------------------------------------------------------

for parameter in expert_model.parameters():

    parameter.requires_grad = False


expert_model.to(
    DEVICE
)

expert_model.eval()


print(
    "\nAll 11 Adaptive PEFT experts loaded."
)

print(
    "All expert parameters are frozen."
)


# ==========================================================
# 8. CAGER v2
#
# Multi-token Client Context
# ==========================================================

class CAGERv2(
    nn.Module
):

    def __init__(self):

        super().__init__()


        # --------------------------------------------------
        # Client identity embedding
        # --------------------------------------------------

        self.client_embedding = nn.Embedding(

            NUM_EXPERTS,

            ROUTER_SIZE

        )


        # --------------------------------------------------
        # Rank embedding
        #
        # Ranks:
        # 4, 8, 12, 16
        # --------------------------------------------------

        self.rank_embedding = nn.Embedding(

            4,

            ROUTER_SIZE

        )


        # --------------------------------------------------
        # Input hidden projection
        # --------------------------------------------------

        self.query_projection = nn.Linear(

            HIDDEN_SIZE,

            ROUTER_SIZE

        )


        # --------------------------------------------------
        # Personalization score projection
        # --------------------------------------------------

        self.score_projection = nn.Sequential(

            nn.Linear(
                1,
                ROUTER_SIZE
            ),

            nn.ReLU()

        )


        # --------------------------------------------------
        # Cross Attention
        # --------------------------------------------------

        self.cross_attention = nn.MultiheadAttention(

            embed_dim=ROUTER_SIZE,

            num_heads=NUM_HEADS,

            batch_first=True

        )


        # --------------------------------------------------
        # Layer normalization
        # --------------------------------------------------

        self.norm = nn.LayerNorm(
            ROUTER_SIZE
        )


        # --------------------------------------------------
        # Gated expert router
        # --------------------------------------------------

        self.router = nn.Sequential(

            nn.Linear(

                ROUTER_SIZE * 2,

                ROUTER_SIZE

            ),

            nn.GELU(),

            nn.Dropout(0.1),

            nn.Linear(

                ROUTER_SIZE,

                NUM_EXPERTS

            )

        )


    def forward(

        self,

        hidden_states,

        client_ids,

        rank_ids,

        personalization_scores

    ):

        # ==================================================
        # Query
        # ==================================================

        query = self.query_projection(

            hidden_states

        )


        # ==================================================
        # Client identity
        # ==================================================

        client_token = (
            self.client_embedding(
                client_ids
            )
        )


        # ==================================================
        # Adaptive rank
        # ==================================================

        rank_token = (
            self.rank_embedding(
                rank_ids
            )
        )


        # ==================================================
        # Personalization score
        # ==================================================

        score_token = (
            self.score_projection(

                personalization_scores
                .unsqueeze(-1)

            )
        )


        # ==================================================
        # Multi-token client context
        #
        # 3 tokens:
        #
        # [client identity]
        # [LoRA rank]
        # [personalization score]
        # ==================================================

        context = torch.stack(

            [
                client_token,
                rank_token,
                score_token
            ],

            dim=1

        )


        # ==================================================
        # Cross Attention
        #
        # Query = input hidden states
        #
        # Key / Value = client context
        # ==================================================

        attended, attention_weights = (
            self.cross_attention(

                query=query,

                key=context,

                value=context

            )
        )


        attended = self.norm(
            attended
        )


        # ==================================================
        # Pool input representation
        # ==================================================

        pooled_query = query.mean(
            dim=1
        )


        pooled_attended = attended.mean(
            dim=1
        )


        # ==================================================
        # Gated router input
        # ==================================================

        router_input = torch.cat(

            [
                pooled_query,
                pooled_attended
            ],

            dim=-1

        )


        # ==================================================
        # Expert logits
        # ==================================================

        expert_logits = self.router(

            router_input

        )


        # ==================================================
        # Expert probabilities
        # ==================================================

        gates = F.softmax(

            expert_logits,

            dim=-1

        )


        return (

            gates,

            attention_weights

        )


# ==========================================================
# 9. Rank Encoding
# ==========================================================

RANK_VALUES = {

    4: 0,
    8: 1,
    12: 2,
    16: 3

}


def get_client_features(
    client_name,
    batch_size
):

    config = CLIENT_CONFIG[
        client_name
    ]


    rank = int(
        config[
            "active_rank"
        ]
    )


    score = float(
        config[
            "personalization_score"
        ]
    )


    if rank not in RANK_VALUES:

        raise ValueError(

            f"Unsupported LoRA rank "
            f"{rank}"

        )


    client_id = torch.full(

        (batch_size,),

        CLIENT_TO_ID[
            client_name
        ],

        dtype=torch.long,

        device=DEVICE

    )


    rank_id = torch.full(

        (batch_size,),

        RANK_VALUES[
            rank
        ],

        dtype=torch.long,

        device=DEVICE

    )


    score_tensor = torch.full(

        (batch_size,),

        score,

        dtype=torch.float32,

        device=DEVICE

    )


    return (

        client_id,

        rank_id,

        score_tensor

    )


# ==========================================================
# 10. Load Calibration Dataset
# ==========================================================

def load_calibration_dataset(
    client_name
):

    train_path = os.path.join(

        CLIENT_PATH,

        client_name,

        "train"

    )


    dataset = load_from_disk(
        train_path
    )


    # ------------------------------------------------------
    # Use only a small subset initially
    # ------------------------------------------------------

    number = min(

        CALIBRATION_SAMPLES,

        len(dataset)

    )


    dataset = dataset.shuffle(
        seed=42
    ).select(

        range(number)

    )


    # ------------------------------------------------------
    # Remove raw text
    # ------------------------------------------------------

    if "text" in dataset.column_names:

        dataset = dataset.remove_columns(
            ["text"]
        )


    return dataset


# ==========================================================
# 11. Calculate Per-Expert Sample Loss
#
# These losses become the routing target.
#
# Better expert → lower loss → higher target weight
# ==========================================================

def calculate_expert_losses(

    input_ids,

    attention_mask,

    labels

):

    losses = []


    for index, client in enumerate(
        CLIENTS
    ):

        # --------------------------------------------------
        # First adapter
        # --------------------------------------------------

        if index == 0:

            adapter_name = "default"

        else:

            adapter_name = client


        expert_model.set_adapter(
            adapter_name
        )


        with torch.no_grad():

            outputs = expert_model(

                input_ids=input_ids,

                attention_mask=attention_mask,

                output_hidden_states=False

            )


        logits = outputs.logits


        # --------------------------------------------------
        # Causal shift
        # --------------------------------------------------

        shift_logits = (

            logits[:, :-1, :]

        )


        shift_labels = (

            labels[:, 1:]

        )


        # --------------------------------------------------
        # Token loss
        # --------------------------------------------------

        token_loss = F.cross_entropy(

            shift_logits.reshape(

                -1,

                shift_logits.size(-1)

            ),

            shift_labels.reshape(
                -1
            ),

            ignore_index=-100,

            reduction="none"

        )


        token_loss = token_loss.reshape(

            shift_labels.shape

        )


        valid_mask = (

            shift_labels != -100

        )


        # --------------------------------------------------
        # Loss per sample
        # --------------------------------------------------

        sample_loss = (

            (
                token_loss
                * valid_mask
            ).sum(dim=1)

            /

            valid_mask.sum(
                dim=1
            ).clamp(
                min=1
            )

        )


        losses.append(
            sample_loss
        )


    # ------------------------------------------------------
    # [batch, experts]
    # ------------------------------------------------------

    return torch.stack(

        losses,

        dim=1

    )


# ==========================================================
# 12. Convert Expert Losses to Routing Targets
# ==========================================================

def create_routing_target(
    expert_losses
):

    target = F.softmax(

        -expert_losses
        /
        TARGET_TEMPERATURE,

        dim=-1

    )


    return target


# ==========================================================
# 13. Top-K Gating
# ==========================================================

def apply_top_k(
    gates
):

    top_values, top_indices = torch.topk(

        gates,

        k=TOP_K,

        dim=-1

    )


    normalized_values = (

        top_values

        /

        (
            top_values.sum(
                dim=-1,
                keepdim=True
            )

            + 1e-8

        )

    )


    sparse_gates = torch.zeros_like(
        gates
    )


    sparse_gates.scatter_(
        -1,
        top_indices,
        normalized_values
    )


    return (

        sparse_gates,

        top_indices

    )


# ==========================================================
# 14. Checkpoint Functions
# ==========================================================

ROUTER_FILE = os.path.join(

    SAVE_PATH,

    "shared_cager_router.pt"

)


STATE_FILE = os.path.join(

    SAVE_PATH,

    "training_state.json"

)


def save_checkpoint(

    router,

    optimizer,

    completed_clients

):

    torch.save(

        {

            "router":
                router.state_dict(),

            "optimizer":
                optimizer.state_dict()

        },

        ROUTER_FILE

    )


    state = {

        "completed_clients":
            completed_clients,

        "calibration_samples":
            CALIBRATION_SAMPLES,

        "top_k":
            TOP_K,

        "num_experts":
            NUM_EXPERTS

    }


    with open(
        STATE_FILE,
        "w"
    ) as f:

        json.dump(

            state,

            f,

            indent=4

        )


    print(
        "\nCAGER v2 checkpoint saved."
    )


# ==========================================================
# 15. Load Existing Checkpoint
# ==========================================================

router = CAGERv2().to(
    DEVICE
)


optimizer = torch.optim.AdamW(

    router.parameters(),

    lr=LEARNING_RATE

)


completed_clients = []


if os.path.exists(
    ROUTER_FILE
):

    print(
        "\nExisting CAGER v2 checkpoint found."
    )


    checkpoint = torch.load(
        ROUTER_FILE,
        map_location=DEVICE
    )

    if isinstance(checkpoint, dict) and "router" in checkpoint:
        router.load_state_dict(
            checkpoint["router"]
        )
        if "optimizer" in checkpoint:
            try:
                optimizer.load_state_dict(
                    checkpoint["optimizer"]
                )
            except Exception:
                pass
    else:
        router.load_state_dict(checkpoint)


    print(
        "Router checkpoint loaded."
    )


if os.path.exists(
    STATE_FILE
):

    with open(
        STATE_FILE,
        "r"
    ) as f:

        state = json.load(f)


    completed_clients = state.get(

        "completed_clients",

        []

    )


    print(
        "Completed clients:",
        completed_clients
    )


# ==========================================================
# 16. Train Shared Router
# ==========================================================

router.train()


for client_name in CLIENTS:

    # ------------------------------------------------------
    # Resume protection
    # ------------------------------------------------------

    if client_name in completed_clients:

        print("\n")

        print(
            f"{client_name} "
            "already completed."
        )

        print(
            "Skipping..."
        )

        continue


    print("\n")

    print("=" * 70)

    print(
        f"CAGER v2 Training : "
        f"{client_name}"
    )

    print("=" * 70)


    # ------------------------------------------------------
    # Dataset
    # ------------------------------------------------------

    dataset = load_calibration_dataset(
        client_name
    )


    print(
        f"Calibration Samples : "
        f"{len(dataset)}"
    )


    loader = DataLoader(

        dataset,

        batch_size=BATCH_SIZE,

        shuffle=True,

        collate_fn=data_collator

    )


    client_loss = 0.0

    batch_count = 0


    # ------------------------------------------------------
    # Training
    # ------------------------------------------------------

    for batch in loader:

        input_ids = (

            batch[
                "input_ids"
            ].to(DEVICE)

        )


        attention_mask = (

            batch[
                "attention_mask"
            ].to(DEVICE)

        )


        labels = (

            batch[
                "labels"
            ].to(DEVICE)

        )


        # ==================================================
        # Base hidden representation
        # ==================================================

        with torch.no_grad():

            base_outputs = base_model(

                input_ids=input_ids,

                attention_mask=attention_mask,

                output_hidden_states=True

            )


            hidden_states = (

                base_outputs
                .hidden_states[-1]
                .detach()

            )


        # ==================================================
        # Expert losses
        # ==================================================

        with torch.no_grad():

            expert_losses = (
                calculate_expert_losses(

                    input_ids,

                    attention_mask,

                    labels

                )

            )


            routing_target = (
                create_routing_target(

                    expert_losses

                )
            )


        # ==================================================
        # Client context
        # ==================================================

        (

            client_ids,

            rank_ids,

            score_tensor

        ) = get_client_features(

            client_name,

            input_ids.size(0)

        )


        # ==================================================
        # Router
        # ==================================================

        gates, attention_weights = router(

            hidden_states,

            client_ids,

            rank_ids,

            score_tensor

        )


        # ==================================================
        # Routing loss
        # ==================================================

        log_gates = torch.log(

            gates.clamp(
                min=1e-8
            )

        )


        routing_loss = F.kl_div(

            log_gates,

            routing_target,

            reduction="batchmean"

        )


        # ==================================================
        # Load balancing
        # ==================================================

        mean_gates = gates.mean(
            dim=0
        )


        balance_loss = (

            NUM_EXPERTS

            *

            torch.sum(
                mean_gates ** 2
            )

        )


        # ==================================================
        # Total loss
        # ==================================================

        loss = (

            routing_loss

            +

            BALANCE_WEIGHT
            *
            balance_loss

        )


        # ==================================================
        # Backpropagation
        # ==================================================

        optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(

            router.parameters(),

            max_norm=1.0

        )

        optimizer.step()


        client_loss += (
            loss.item()
        )

        batch_count += 1


        # --------------------------------------------------
        # Progress
        # --------------------------------------------------

        if batch_count % 10 == 0:

            print(

                f"Batch {batch_count} | "
                f"Loss {loss.item():.4f}"

            )


    # ------------------------------------------------------
    # Client average loss
    # ------------------------------------------------------

    average_client_loss = (

        client_loss

        /

        max(
            batch_count,
            1
        )

    )


    print("\n")

    print(
        f"{client_name} "
        f"Average Router Loss : "
        f"{average_client_loss:.4f}"
    )


    # ------------------------------------------------------
    # Mark completed
    # ------------------------------------------------------

    completed_clients.append(
        client_name
    )


    # ------------------------------------------------------
    # Save checkpoint
    # ------------------------------------------------------

    save_checkpoint(

        router,

        optimizer,

        completed_clients

    )


# ==========================================================
# 17. Final Save
# ==========================================================

router.eval()


torch.save(

    router.state_dict(),

    ROUTER_FILE

)


print("\n")

print("=" * 70)

print(
    "CAGER v2 TRAINING COMPLETED"
)

print("=" * 70)

print(
    f"Router Saved : "
    f"{ROUTER_FILE}"
)

print(
    f"Clients Completed : "
    f"{len(completed_clients)} / "
    f"{NUM_EXPERTS}"
)

print("=" * 70)

CAGER v2 CONFIGURATION
Experts              : 11
Top-K                : 3
Calibration Samples  : 10 / client
Batch Size           : 2
Device               : cpu
Clients              : ['client_1', 'client_2', 'client_3', 'client_4', 'client_5', 'client_6', 'client_7', 'client_8', 'client_9', 'client_10', 'client_11']

Loading frozen base model...


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 7209.15it/s]


Base model loaded.


Loading Frozen Expert Pool
Loading expert: client_2
Loading expert: client_3
Loading expert: client_4
Loading expert: client_5
Loading expert: client_6
Loading expert: client_7
Loading expert: client_8
Loading expert: client_9
Loading expert: client_10
Loading expert: client_11

All 11 Adaptive PEFT experts loaded.
All expert parameters are frozen.

Existing CAGER v2 checkpoint found.
Router checkpoint loaded.
Completed clients: ['client_1', 'client_2', 'client_3', 'client_4', 'client_5', 'client_6', 'client_7', 'client_8', 'client_9', 'client_10', 'client_11']


client_1 already completed.
Skipping...


client_2 already completed.
Skipping...


client_3 already completed.
Skipping...


client_4 already completed.
Skipping...


client_5 already completed.
Skipping...


client_6 already completed.
Skipping...


client_7 already completed.
Skipping...


client_8 already completed.
Skipping...


client_9 already completed.
Skipping...


client_10 already completed.
Sk

## Module 8C-v2: Shared CAGER Router Evaluation
Evaluates the performance of the shared CAGER v2 router.

In [15]:
# ==========================================================
# Module 8C-v2
# Shared CAGER Evaluation
#
# Adaptive PEFT + Shared Cross-Attention Gated Expert Router
#
# Evaluates all 11 clients.
#
# Metrics:
#   - Loss
#   - Perplexity
#   - Top-1 Accuracy
#   - Top-5 Accuracy
#   - Adaptive PEFT improvement
#   - Expert gate distribution
#   - Top-K expert selection
#
# IMPORTANT:
#   - LoRA experts are completely frozen.
#   - Only the saved CAGER v2 router is used.
#   - No training happens in this module.
# ==========================================================

import os
import json
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

from datasets import load_from_disk
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling
)

from peft import PeftModel


# ==========================================================
# 1. Paths
# ==========================================================

MODEL_NAME = "distilgpt2"

import sys
try:
    MODULE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    MODULE_DIR = os.path.abspath(os.path.join(os.getcwd(), 'code', 'modules'))
if MODULE_DIR not in sys.path:
    sys.path.append(MODULE_DIR)
import local_config

CLIENT_PATH = local_config.CLIENT_PATH
ADAPTER_PATH = local_config.MODEL_SAVE_PATH
CAGER_PATH = os.path.join(local_config.WORKSPACE_DIR, "cager_v2_models")
RESULT_PATH = os.path.join(local_config.WORKSPACE_DIR, "cager_v2_evaluation")
BASELINE_PATH = local_config.EVALUATION_PATH

os.makedirs(
    RESULT_PATH,
    exist_ok=True
)


# ==========================================================
# 2. Configuration
# ==========================================================

DEVICE = local_config.DEVICE

HIDDEN_SIZE = 768

ROUTER_SIZE = 256

NUM_HEADS = 4

NUM_EXPERTS = 11

TOP_K = 3

BATCH_SIZE = 8

TEMPERATURE = 1.0


CLIENTS = sorted(

    [
        c
        for c in os.listdir(
            ADAPTER_PATH
        )
        if c.startswith("client_")
    ],

    key=lambda x: int(
        x.split("_")[1]
    )

)


if len(CLIENTS) != NUM_EXPERTS:

    raise ValueError(

        f"Expected {NUM_EXPERTS} clients "
        f"but found {len(CLIENTS)}"

    )


CLIENT_TO_ID = {

    client: i

    for i, client in enumerate(
        CLIENTS
    )

}


print("=" * 70)

print(
    "CAGER v2 EVALUATION"
)

print("=" * 70)

print(
    f"Experts     : {NUM_EXPERTS}"
)

print(
    f"Top-K       : {TOP_K}"
)

print(
    f"Batch Size  : {BATCH_SIZE}"
)

print(
    f"Device      : {DEVICE}"
)

print(
    "Clients     :",
    CLIENTS
)

print("=" * 70)


# ==========================================================
# 3. Adaptive Client Configuration
# ==========================================================

CONFIG_PATH = local_config.CONFIG_PATH


if not os.path.exists(
    CONFIG_PATH
):

    raise FileNotFoundError(
        CONFIG_PATH
    )


with open(
    CONFIG_PATH,
    "r"
) as f:

    CLIENT_CONFIG = json.load(f)


# ==========================================================
# 4. Tokenizer
# ==========================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

tokenizer.pad_token = tokenizer.eos_token


data_collator = (
    DataCollatorForLanguageModeling(

        tokenizer=tokenizer,

        mlm=False

    )
)


# ==========================================================
# 5. LoRA Rank Encoding
# ==========================================================

RANK_VALUES = {

    4: 0,

    8: 1,

    12: 2,

    16: 3

}


def get_client_features(
    client_name,
    batch_size
):

    config = CLIENT_CONFIG[
        client_name
    ]


    rank = int(
        config[
            "active_rank"
        ]
    )


    score = float(
        config[
            "personalization_score"
        ]
    )


    if rank not in RANK_VALUES:

        raise ValueError(

            f"Unsupported LoRA rank: "
            f"{rank}"

        )


    client_ids = torch.full(

        (batch_size,),

        CLIENT_TO_ID[
            client_name
        ],

        dtype=torch.long,

        device=DEVICE

    )


    rank_ids = torch.full(

        (batch_size,),

        RANK_VALUES[
            rank
        ],

        dtype=torch.long,

        device=DEVICE

    )


    score_tensor = torch.full(

        (batch_size,),

        score,

        dtype=torch.float32,

        device=DEVICE

    )


    return (

        client_ids,

        rank_ids,

        score_tensor

    )


# ==========================================================
# 6. CAGER v2 Architecture
# ==========================================================

class CAGERv2(
    nn.Module
):

    def __init__(self):

        super().__init__()


        # --------------------------------------------------
        # Client identity
        # --------------------------------------------------

        self.client_embedding = nn.Embedding(

            NUM_EXPERTS,

            ROUTER_SIZE

        )


        # --------------------------------------------------
        # Adaptive LoRA rank
        # --------------------------------------------------

        self.rank_embedding = nn.Embedding(

            4,

            ROUTER_SIZE

        )


        # --------------------------------------------------
        # Input projection
        # --------------------------------------------------

        self.query_projection = nn.Linear(

            HIDDEN_SIZE,

            ROUTER_SIZE

        )


        # --------------------------------------------------
        # Personalization score
        # --------------------------------------------------

        self.score_projection = nn.Sequential(

            nn.Linear(
                1,
                ROUTER_SIZE
            ),

            nn.ReLU()

        )


        # --------------------------------------------------
        # Cross Attention
        # --------------------------------------------------

        self.cross_attention = nn.MultiheadAttention(

            embed_dim=ROUTER_SIZE,

            num_heads=NUM_HEADS,

            batch_first=True

        )


        # --------------------------------------------------
        # Normalization
        # --------------------------------------------------

        self.norm = nn.LayerNorm(
            ROUTER_SIZE
        )


        # --------------------------------------------------
        # Gated Router
        # --------------------------------------------------

        self.router = nn.Sequential(

            nn.Linear(

                ROUTER_SIZE * 2,

                ROUTER_SIZE

            ),

            nn.GELU(),

            nn.Dropout(0.1),

            nn.Linear(

                ROUTER_SIZE,

                NUM_EXPERTS

            )

        )


    def forward(

        self,

        hidden_states,

        client_ids,

        rank_ids,

        personalization_scores

    ):

        # --------------------------------------------------
        # Input query
        # --------------------------------------------------

        query = self.query_projection(

            hidden_states

        )


        # --------------------------------------------------
        # Client identity token
        # --------------------------------------------------

        client_token = (

            self.client_embedding(
                client_ids
            )

        )


        # --------------------------------------------------
        # LoRA rank token
        # --------------------------------------------------

        rank_token = (

            self.rank_embedding(
                rank_ids
            )

        )


        # --------------------------------------------------
        # Personalization token
        # --------------------------------------------------

        score_token = (

            self.score_projection(

                personalization_scores
                .unsqueeze(-1)

            )

        )


        # --------------------------------------------------
        # Multi-token client context
        # --------------------------------------------------

        context = torch.stack(

            [

                client_token,

                rank_token,

                score_token

            ],

            dim=1

        )


        # --------------------------------------------------
        # Cross Attention
        # --------------------------------------------------

        attended, attention_weights = (

            self.cross_attention(

                query=query,

                key=context,

                value=context

            )

        )


        attended = self.norm(
            attended
        )


        # --------------------------------------------------
        # Pool representations
        # --------------------------------------------------

        pooled_query = query.mean(
            dim=1
        )


        pooled_attended = attended.mean(
            dim=1
        )


        # --------------------------------------------------
        # Router input
        # --------------------------------------------------

        router_input = torch.cat(

            [

                pooled_query,

                pooled_attended

            ],

            dim=-1

        )


        # --------------------------------------------------
        # Expert scores
        # --------------------------------------------------

        expert_logits = self.router(

            router_input

        )


        # --------------------------------------------------
        # Expert probabilities
        # --------------------------------------------------

        gates = F.softmax(

            expert_logits,

            dim=-1

        )


        return (

            gates,

            attention_weights

        )


# ==========================================================
# 7. Load Shared CAGER Router
# ==========================================================

ROUTER_PATH = os.path.join(

    CAGER_PATH,

    "shared_cager_router.pt"

)


if not os.path.exists(
    ROUTER_PATH
):

    raise FileNotFoundError(

        "Shared CAGER router not found:\n"
        f"{ROUTER_PATH}"

    )


router = CAGERv2().to(
    DEVICE
)


router.load_state_dict(

    torch.load(

        ROUTER_PATH,

        map_location=DEVICE

    )

)


router.eval()


print(
    "\nShared CAGER v2 router loaded."
)


# ==========================================================
# 8. Load Frozen Base Model
# ==========================================================

print("\nLoading base model...")


base_model = (
    AutoModelForCausalLM
    .from_pretrained(
        MODEL_NAME
    )
)


base_model.to(
    DEVICE
)

base_model.eval()


for parameter in base_model.parameters():

    parameter.requires_grad = False


print(
    "Base model loaded and frozen."
)


# ==========================================================
# 9. Load Frozen Expert Pool
# ==========================================================

print("\n")

print("=" * 70)

print(
    "Loading Frozen Expert Pool"
)

print("=" * 70)


first_client = CLIENTS[0]

first_adapter_path = os.path.join(

    ADAPTER_PATH,

    first_client

)


expert_model = (
    PeftModel.from_pretrained(

        base_model,

        first_adapter_path,

        is_trainable=False

    )
)


for client in CLIENTS[1:]:

    adapter_path = os.path.join(

        ADAPTER_PATH,

        client

    )


    print(
        f"Loading expert: "
        f"{client}"
    )


    expert_model.load_adapter(

        adapter_path,

        adapter_name=client

    )


for parameter in expert_model.parameters():

    parameter.requires_grad = False


expert_model.to(
    DEVICE
)

expert_model.eval()


print(
    "\nAll 11 experts loaded and frozen."
)


# ==========================================================
# 10. Top-K Routing
# ==========================================================

def apply_top_k(
    gates
):

    top_values, top_indices = torch.topk(

        gates,

        k=TOP_K,

        dim=-1

    )


    normalized_values = (

        top_values

        /

        (
            top_values.sum(
                dim=-1,
                keepdim=True
            )

            + 1e-8

        )

    )


    return (

        top_indices,

        normalized_values

    )


# ==========================================================
# 11. Evaluate Selected Experts
# ==========================================================

def evaluate_selected_experts(

    input_ids,

    attention_mask,

    selected_indices,

    selected_weights

):

    batch_size = input_ids.size(0)

    seq_length = input_ids.size(1)

    vocab_size = (
        expert_model.config.vocab_size
    )


    combined_logits = torch.zeros(

        batch_size,

        seq_length,

        vocab_size,

        device=DEVICE,

        dtype=torch.float32

    )


    # ------------------------------------------------------
    # Find unique experts selected in this batch
    # ------------------------------------------------------

    unique_experts = torch.unique(
        selected_indices
    )


    for expert_index in unique_experts:

        expert_id = (
            expert_index.item()
        )


        # --------------------------------------------------
        # Adapter name
        # --------------------------------------------------

        if expert_id == 0:

            adapter_name = "default"

        else:

            adapter_name = CLIENTS[
                expert_id
            ]


        # --------------------------------------------------
        # Select samples using this expert
        # --------------------------------------------------

        sample_mask = (

            selected_indices
            == expert_id

        )


        sample_mask = sample_mask.any(
            dim=1
        )


        sample_positions = (
            torch.where(
                sample_mask
            )[0]
        )


        if sample_positions.numel() == 0:

            continue


        expert_model.set_adapter(
            adapter_name
        )


        # --------------------------------------------------
        # Run expert only on selected samples
        # --------------------------------------------------

        with torch.no_grad():

            outputs = expert_model(

                input_ids=input_ids[
                    sample_positions
                ],

                attention_mask=attention_mask[
                    sample_positions
                ]

            )


        expert_logits = (
            outputs.logits
        )


        # --------------------------------------------------
        # Add weighted prediction
        # --------------------------------------------------

        for local_index, global_index in enumerate(
            sample_positions
        ):

            for k in range(
                TOP_K
            ):

                if (

                    selected_indices[
                        global_index,
                        k
                    ].item()

                    == expert_id

                ):

                    weight = (

                        selected_weights[
                            global_index,
                            k
                        ]

                    )

                    combined_logits[
                        global_index
                    ] += (

                        weight
                        *
                        expert_logits[
                            local_index
                        ]

                    )


    return combined_logits


# ==========================================================
# 12. Load Adaptive PEFT Baseline
# ==========================================================

BASELINE_CSV = os.path.join(

    BASELINE_PATH,

    "personalized_evaluation.csv"

)


baseline_df = None


if os.path.exists(
    BASELINE_CSV
):

    baseline_df = pd.read_csv(
        BASELINE_CSV
    )

    print(
        "\nAdaptive PEFT baseline loaded."
    )

else:

    print(
        "\nWARNING: Adaptive PEFT baseline "
        "CSV not found."
    )


# ==========================================================
# 13. Evaluate One Client
# ==========================================================

def evaluate_client(
    client_name
):

    print("\n")

    print("=" * 70)

    print(
        f"Evaluating : "
        f"{client_name}"
    )

    print("=" * 70)


    # ------------------------------------------------------
    # Test dataset
    # ------------------------------------------------------

    test_path = os.path.join(

        CLIENT_PATH,

        client_name,

        "test"

    )


    test_dataset = load_from_disk(
        test_path
    )

    if "text" in test_dataset.column_names:
        test_dataset = test_dataset.remove_columns(
            ["text"]
        )

    if local_config.QUICK_TEST:
        test_dataset = test_dataset.select(range(min(10, len(test_dataset))))
        print(f"  [Quick Test] Subsampled test dataset to {len(test_dataset)} samples.")


    loader = DataLoader(

        test_dataset,

        batch_size=BATCH_SIZE,

        shuffle=False,

        collate_fn=data_collator

    )


    # ------------------------------------------------------
    # Metrics
    # ------------------------------------------------------

    total_loss = 0.0

    total_tokens = 0

    top1_correct = 0

    top5_correct = 0


    # ------------------------------------------------------
    # Expert statistics
    # ------------------------------------------------------

    gate_sum = torch.zeros(

        NUM_EXPERTS,

        device=DEVICE

    )


    selection_count = {

        client: 0

        for client in CLIENTS

    }


    # ------------------------------------------------------
    # Client features
    # ------------------------------------------------------

    client_id = CLIENT_TO_ID[
        client_name
    ]


    print(
        f"Test Samples : "
        f"{len(test_dataset)}"
    )


    # ------------------------------------------------------
    # Evaluation
    # ------------------------------------------------------

    with torch.no_grad():

        for batch_index, batch in enumerate(
            loader
        ):

            input_ids = (

                batch[
                    "input_ids"
                ].to(DEVICE)

            )


            attention_mask = (

                batch[
                    "attention_mask"
                ].to(DEVICE)

            )


            labels = (

                batch[
                    "labels"
                ].to(DEVICE)

            )


            batch_size = (
                input_ids.size(0)
            )


            # ==================================================
            # Base hidden representation
            # ==================================================

            base_outputs = base_model(

                input_ids=input_ids,

                attention_mask=attention_mask,

                output_hidden_states=True

            )


            hidden_states = (

                base_outputs
                .hidden_states[-1]
            )


            # ==================================================
            # Client context
            # ==================================================

            (

                client_ids,

                rank_ids,

                score_tensor

            ) = get_client_features(

                client_name,

                batch_size

            )


            # ==================================================
            # CAGER routing
            # ==================================================

            gates, attention_weights = router(

                hidden_states,

                client_ids,

                rank_ids,

                score_tensor

            )


            # --------------------------------------------------
            # Gate statistics
            # --------------------------------------------------

            gate_sum += gates.sum(
                dim=0
            )


            # ==================================================
            # Top-K experts
            # ==================================================

            selected_indices, selected_weights = (

                apply_top_k(
                    gates
                )

            )


            # --------------------------------------------------
            # Selection statistics
            # --------------------------------------------------

            for row in selected_indices:

                for expert_index in row:

                    expert_name = CLIENTS[
                        expert_index.item()
                    ]

                    selection_count[
                        expert_name
                    ] += 1


            # ==================================================
            # Expert prediction
            # ==================================================

            combined_logits = (

                evaluate_selected_experts(

                    input_ids,

                    attention_mask,

                    selected_indices,

                    selected_weights

                )

            )


            # ==================================================
            # Causal LM shift
            # ==================================================

            shift_logits = (

                combined_logits[
                    :, :-1, :
                ]

            )


            shift_labels = (

                labels[
                    :, 1:
                ]

            )


            valid_mask = (

                shift_labels
#                 != -100

            )


            # ==================================================
            # Loss
            # ==================================================

            token_loss = F.cross_entropy(

                shift_logits.reshape(

                    -1,

                    shift_logits.size(-1)

                ),

                shift_labels.reshape(
                    -1
                ),

                ignore_index=-100,

                reduction="sum"

            )


            total_loss += (
                token_loss.item()
            )


            valid_tokens = (
                valid_mask.sum().item()
            )


            total_tokens += (
                valid_tokens
            )


            # ==================================================
            # Top-1
            # ==================================================

            top1_predictions = torch.argmax(

                shift_logits,

                dim=-1

            )


            top1_correct += (

                (

                    top1_predictions
                    == shift_labels

                )

                &

                valid_mask

            ).sum().item()


            # ==================================================
            # Top-5
            # ==================================================

            top5_predictions = torch.topk(

                shift_logits,

                k=5,

                dim=-1

            ).indices


            expanded_labels = (

                shift_labels
                .unsqueeze(-1)
                .expand_as(
                    top5_predictions
                )

            )


            top5_correct += (

                (

                    top5_predictions
                    == expanded_labels

                ).any(
                    dim=-1
                )

                &

                valid_mask

            ).sum().item()


            # ==================================================
            # Progress
            # ==================================================

            if (
                (batch_index + 1)
                % 100
                == 0
            ):

                print(

                    f"Processed batches : "
                    f"{batch_index + 1}"

                )


    # ======================================================
    # Final Metrics
    # ======================================================

    average_loss = (

        total_loss
        /
        max(
            total_tokens,
            1
        )

    )


    perplexity = math.exp(
        min(
            average_loss,
            20
        )
    )


    top1_accuracy = (

        top1_correct
        /
        max(
            total_tokens,
            1
        )

    ) * 100


    top5_accuracy = (

        top5_correct
        /
        max(
            total_tokens,
            1
        )

    ) * 100


    # ======================================================
    # Gate Distribution
    # ======================================================

    total_gate_mass = gate_sum.sum()


    gate_distribution = {

        client:

        float(

            gate_sum[index]
            /
            total_gate_mass

        )

        for index, client
        in enumerate(CLIENTS)

    }


    # ======================================================
    # Baseline
    # ======================================================

    baseline_top1 = None

    baseline_top5 = None

    baseline_ppl = None


    if baseline_df is not None:

        row = baseline_df[

            baseline_df[
                "client"
            ]
            == client_name

        ]


        if len(row) > 0:

            baseline_top1 = float(

                row[
                    "personalized_top1_accuracy"
                ].iloc[0]

            )


            baseline_top5 = float(

                row[
                    "personalized_top5_accuracy"
                ].iloc[0]

            )


            baseline_ppl = float(

                row[
                    "personalized_perplexity"
                ].iloc[0]

            )


    # ======================================================
    # Improvements
    # ======================================================

    top1_gain = None

    top5_gain = None

    ppl_change = None


    if baseline_top1 is not None:

        top1_gain = (

            top1_accuracy
            -
            baseline_top1

        )


    if baseline_top5 is not None:

        top5_gain = (

            top5_accuracy
            -
            baseline_top5

        )


    if baseline_ppl is not None:

        ppl_change = (

            baseline_ppl
            -
            perplexity

        )


    # ======================================================
    # Display
    # ======================================================

    print("\n")

    print("=" * 70)

    print(
        f"CAGER v2 RESULTS : "
        f"{client_name}"
    )

    print("=" * 70)

    print(
        f"Loss              : "
        f"{average_loss:.4f}"
    )

    print(
        f"Perplexity        : "
        f"{perplexity:.4f}"
    )

    print(
        f"Top-1 Accuracy    : "
        f"{top1_accuracy:.2f}%"
    )

    print(
        f"Top-5 Accuracy    : "
        f"{top5_accuracy:.2f}%"
    )

    print(
        f"Tokens Evaluated  : "
        f"{total_tokens}"
    )


    if baseline_top1 is not None:

        print("\n")

        print(
            f"Adaptive PEFT Top-1 : "
            f"{baseline_top1:.2f}%"
        )

        print(
            f"CAGER v2 Top-1      : "
            f"{top1_accuracy:.2f}%"
        )

        print(
            f"Top-1 Gain          : "
            f"{top1_gain:+.2f} percentage points"
        )


        print()

        print(
            f"Adaptive PEFT PPL   : "
            f"{baseline_ppl:.4f}"
        )

        print(
            f"CAGER v2 PPL        : "
            f"{perplexity:.4f}"
        )

        print(
            f"PPL Change          : "
            f"{ppl_change:+.4f}"
        )


    # ======================================================
    # Expert Distribution
    # ======================================================

    print("\n")

    print(
        "Expert Gate Distribution"
    )

    print("-" * 70)


    for index, client in enumerate(
        CLIENTS
    ):

        print(

            f"{client:<12}"
            f"{gate_distribution[client]:.4f}"
            f"    "
            f"Top-K Count = "
            f"{selection_count[client]}"

        )


    print("=" * 70)


    # ======================================================
    # Save Client Result
    # ======================================================

    result = {

        "client":
            client_name,

        "tokens":
            int(total_tokens),

        "loss":
            float(average_loss),

        "perplexity":
            float(perplexity),

        "top1_accuracy":
            float(top1_accuracy),

        "top5_accuracy":
            float(top5_accuracy),

        "adaptive_peft_top1":
            baseline_top1,

        "adaptive_peft_top5":
            baseline_top5,

        "adaptive_peft_perplexity":
            baseline_ppl,

        "top1_gain":
            top1_gain,

        "top5_gain":
            top5_gain,

        "perplexity_change":
            ppl_change,

        "expert_gate_distribution":
            gate_distribution,

        "expert_selection_count":
            selection_count

    }


    result_path = os.path.join(

        RESULT_PATH,

        f"{client_name}_cager_v2.json"

    )


    with open(
        result_path,
        "w"
    ) as f:

        json.dump(

            result,

            f,

            indent=4

        )


    print(
        f"\nSaved : "
        f"{result_path}"
    )


    return result


# ==========================================================
# 14. Evaluate All Clients
# ==========================================================

all_results = []


print("\n")

print("=" * 70)

print(
    "STARTING CAGER v2 EVALUATION"
)

print("=" * 70)


clients_to_eval = CLIENTS

for client in clients_to_eval:

    result = evaluate_client(
        client
    )

    all_results.append(
        result
    )


# ==========================================================
# 15. Create Summary DataFrame
# ==========================================================

summary_rows = []


for result in all_results:

    summary_rows.append({

        "client":
            result["client"],

        "cager_top1":
            result["top1_accuracy"],

        "adaptive_peft_top1":
            result["adaptive_peft_top1"],

        "top1_gain":
            result["top1_gain"],

        "cager_top5":
            result["top5_accuracy"],

        "adaptive_peft_top5":
            result["adaptive_peft_top5"],

        "top5_gain":
            result["top5_gain"],

        "cager_perplexity":
            result["perplexity"],

        "adaptive_peft_perplexity":
            result["adaptive_peft_perplexity"],

        "perplexity_change":
            result["perplexity_change"]

    })


summary_df = pd.DataFrame(
    summary_rows
)


# ==========================================================
# 16. Save Summary
# ==========================================================

summary_csv = os.path.join(

    RESULT_PATH,

    "cager_v2_summary.csv"

)


summary_df.to_csv(

    summary_csv,

    index=False

)


# ==========================================================
# 17. Overall Results
# ==========================================================

valid_top1 = (

    summary_df[
        "cager_top1"
    ]
    .dropna()

)


valid_baseline_top1 = (

    summary_df[
        "adaptive_peft_top1"
    ]
    .dropna()

)


valid_top5 = (

    summary_df[
        "cager_top5"
    ]
    .dropna()

)


valid_baseline_top5 = (

    summary_df[
        "adaptive_peft_top5"
    ]
    .dropna()

)


valid_ppl = (

    summary_df[
        "cager_perplexity"
    ]
    .dropna()

)


valid_baseline_ppl = (

    summary_df[
        "adaptive_peft_perplexity"
    ]
    .dropna()

)


overall_cager_top1 = (
    valid_top1.mean()
)

overall_baseline_top1 = (
    valid_baseline_top1.mean()
)

overall_cager_top5 = (
    valid_top5.mean()
)

overall_baseline_top5 = (
    valid_baseline_top5.mean()
)

overall_cager_ppl = (
    valid_ppl.mean()
)

overall_baseline_ppl = (
    valid_baseline_ppl.mean()
)


overall_top1_gain = (

    overall_cager_top1
    -
    overall_baseline_top1

)


overall_top5_gain = (

    overall_cager_top5
    -
    overall_baseline_top5

)


overall_ppl_change = (

    overall_baseline_ppl
    -
    overall_cager_ppl

)

# Consolidated (token-weighted) overall metrics
total_tokens = sum(r.get("tokens", 0) for r in all_results)

overall_cager_correct_top1 = sum(round((r.get("top1_accuracy", 0.0) / 100.0) * r.get("tokens", 0)) for r in all_results)
overall_baseline_correct_top1 = sum(round((r.get("adaptive_peft_top1", 0.0) / 100.0) * r.get("tokens", 0)) for r in all_results if r.get("adaptive_peft_top1") is not None)

overall_cager_correct_top5 = sum(round((r.get("top5_accuracy", 0.0) / 100.0) * r.get("tokens", 0)) for r in all_results)
overall_baseline_correct_top5 = sum(round((r.get("adaptive_peft_top5", 0.0) / 100.0) * r.get("tokens", 0)) for r in all_results if r.get("adaptive_peft_top5") is not None)

overall_cager_total_loss = sum(r.get("loss", 0.0) * r.get("tokens", 0) for r in all_results)
overall_baseline_total_loss = sum(math.log(r.get("adaptive_peft_perplexity", 1.0)) * r.get("tokens", 0) for r in all_results if r.get("adaptive_peft_perplexity") is not None)

overall_cager_top1_weighted = (overall_cager_correct_top1 / max(total_tokens, 1)) * 100.0
overall_baseline_top1_weighted = (overall_baseline_correct_top1 / max(total_tokens, 1)) * 100.0
overall_top1_gain_weighted = overall_cager_top1_weighted - overall_baseline_top1_weighted

overall_cager_top5_weighted = (overall_cager_correct_top5 / max(total_tokens, 1)) * 100.0
overall_baseline_top5_weighted = (overall_baseline_correct_top5 / max(total_tokens, 1)) * 100.0
overall_top5_gain_weighted = overall_cager_top5_weighted - overall_baseline_top5_weighted

overall_cager_ppl_weighted = math.exp(min(overall_cager_total_loss / max(total_tokens, 1), 20))
overall_baseline_ppl_weighted = math.exp(min(overall_baseline_total_loss / max(total_tokens, 1), 20))
overall_ppl_change_weighted = overall_baseline_ppl_weighted - overall_cager_ppl_weighted

# ==========================================================
# 18. Final Report
# ==========================================================

print("\n")

print("=" * 80)

print(
    "CAGER v2 FINAL RESULTS"
)

print("=" * 80)

print("\n--- CLIENT-WISE AVERAGE METRICS ---")

print(
    f"Adaptive PEFT Avg Top-1 : "
    f"{overall_baseline_top1:.2f}%"
)

print(
    f"CAGER v2 Avg Top-1      : "
    f"{overall_cager_top1:.2f}%"
)

print(
    f"Average Top-1 Gain      : "
    f"{overall_top1_gain:+.2f} percentage points"
)

print()

print(
    f"Adaptive PEFT Avg Top-5 : "
    f"{overall_baseline_top5:.2f}%"
)

print(
    f"CAGER v2 Avg Top-5      : "
    f"{overall_cager_top5:.2f}%"
)

print(
    f"Average Top-5 Gain      : "
    f"{overall_top5_gain:+.2f} percentage points"
)

print()

print(
    f"Adaptive PEFT Avg PPL   : "
    f"{overall_baseline_ppl:.4f}"
)

print(
    f"CAGER v2 Avg PPL        : "
    f"{overall_cager_ppl:.4f}"
)

print(
    f"Average PPL Change      : "
    f"{overall_ppl_change:+.4f}"
)

print("\n--- OVERALL CONSOLIDATED METRICS (TOKEN-WEIGHTED) ---")

print(
    f"Total Evaluated Tokens  : {total_tokens}"
)

print(
    f"Overall Baseline Top-1  : {overall_baseline_top1_weighted:.2f}%"
)

print(
    f"Overall CAGER v2 Top-1  : {overall_cager_top1_weighted:.2f}%"
)

print(
    f"Overall Top-1 Gain      : {overall_top1_gain_weighted:+.2f}%"
)

print()

print(
    f"Overall Baseline Top-5  : {overall_baseline_top5_weighted:.2f}%"
)

print(
    f"Overall CAGER v2 Top-5  : {overall_cager_top5_weighted:.2f}%"
)

print(
    f"Overall Top-5 Gain      : {overall_top5_gain_weighted:+.2f}%"
)

print()

print(
    f"Overall Baseline PPL    : {overall_baseline_ppl_weighted:.4f}"
)

print(
    f"Overall CAGER v2 PPL    : {overall_cager_ppl_weighted:.4f}"
)

print(
    f"Overall PPL Change      : {overall_ppl_change_weighted:+.4f}"
)

print()

print("=" * 80)


# ==========================================================
# 19. Save Overall Metrics
# ==========================================================

overall_result = {

    "adaptive_peft_average_top1":
        float(overall_baseline_top1),

    "cager_v2_average_top1":
        float(overall_cager_top1),

    "top1_gain":
        float(overall_top1_gain),

    "adaptive_peft_average_top5":
        float(overall_baseline_top5),

    "cager_v2_average_top5":
        float(overall_cager_top5),

    "top5_gain":
        float(overall_top5_gain),

    "adaptive_peft_average_perplexity":
        float(overall_baseline_ppl),

    "cager_v2_average_perplexity":
        float(overall_cager_ppl),

    "perplexity_change":
        float(overall_ppl_change),

    "total_tokens_evaluated":
        int(total_tokens),

    "overall_baseline_top1_accuracy":
        float(overall_baseline_top1_weighted),

    "overall_cager_v2_top1_accuracy":
        float(overall_cager_top1_weighted),

    "overall_top1_accuracy_gain":
        float(overall_top1_gain_weighted),

    "overall_baseline_top5_accuracy":
        float(overall_baseline_top5_weighted),

    "overall_cager_v2_top5_accuracy":
        float(overall_cager_top5_weighted),

    "overall_top5_accuracy_gain":
        float(overall_top5_gain_weighted),

    "overall_baseline_perplexity":
        float(overall_baseline_ppl_weighted),

    "overall_cager_v2_perplexity":
        float(overall_cager_ppl_weighted),

    "overall_perplexity_change":
        float(overall_ppl_change_weighted),

    "clients_evaluated":
        len(all_results)

}


overall_path = os.path.join(

    RESULT_PATH,

    "cager_v2_overall_results.json"

)


with open(
    overall_path,
    "w"
) as f:

    json.dump(

        overall_result,

        f,

        indent=4

    )


print(
    "\nSummary CSV :",
    summary_csv
)

print(
    "Overall JSON:",
    overall_path
)


print("\n")

print("=" * 80)

print(
    "CAGER v2 EVALUATION COMPLETED"
)

print("=" * 80)

CAGER v2 EVALUATION
Experts     : 11
Top-K       : 3
Batch Size  : 8
Device      : cpu
Clients     : ['client_1', 'client_2', 'client_3', 'client_4', 'client_5', 'client_6', 'client_7', 'client_8', 'client_9', 'client_10', 'client_11']

Shared CAGER v2 router loaded.

Loading base model...


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 6439.22it/s]


Base model loaded and frozen.


Loading Frozen Expert Pool
Loading expert: client_2
Loading expert: client_3
Loading expert: client_4
Loading expert: client_5
Loading expert: client_6
Loading expert: client_7
Loading expert: client_8
Loading expert: client_9
Loading expert: client_10
Loading expert: client_11

All 11 experts loaded and frozen.

Adaptive PEFT baseline loaded.


STARTING CAGER v2 EVALUATION


Evaluating : client_1
  [Quick Test] Subsampled test dataset to 7 samples.
Test Samples : 7


CAGER v2 RESULTS : client_1
Loss              : 0.0012
Perplexity        : 1.0012
Top-1 Accuracy    : 0.00%
Top-5 Accuracy    : 0.00%
Tokens Evaluated  : 371218


Adaptive PEFT Top-1 : 9.68%
CAGER v2 Top-1      : 0.00%
Top-1 Gain          : -9.68 percentage points

Adaptive PEFT PPL   : 1224.4828
CAGER v2 PPL        : 1.0012
PPL Change          : +1223.4816


Expert Gate Distribution
----------------------------------------------------------------------
client_1    0.0694    Top-K Count = 0

## Module 8D: Personalized CAGER v3 Router
Trains CAGER v3 with expert compatibility scores and self-expert prior.

In [16]:
# ==========================================================
# Module 8D
# Personalized CAGER v3
#
# Main improvements over CAGER v2:
# 1. Client-aware expert compatibility
# 2. Own-expert personalization prior
# 3. Cross-attention retained
# 4. Adaptive LoRA rank retained
# 5. Personalization score retained
# 6. Top-K sparse routing
#
# IMPORTANT:
# Adaptive PEFT experts remain completely frozen.
# Only the CAGER router is trained.
# ==========================================================

import os
import json
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_from_disk
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling
)
from peft import PeftModel

# ==========================================================
# 1. Paths & Local Config
# ==========================================================

MODEL_NAME = "distilgpt2"
try:
    MODULE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    MODULE_DIR = os.path.abspath(os.path.join(os.getcwd(), 'code', 'modules'))
if MODULE_DIR not in sys.path:
    sys.path.append(MODULE_DIR)
import local_config

CLIENT_PATH = local_config.CLIENT_PATH
ADAPTER_PATH = local_config.MODEL_SAVE_PATH
CONFIG_PATH = local_config.CONFIG_PATH
SAVE_PATH = os.path.join(local_config.WORKSPACE_DIR, "cager_v3_models")
os.makedirs(SAVE_PATH, exist_ok=True)

# ==========================================================
# 2. Configuration & Hyperparameters
# ==========================================================

DEVICE = local_config.DEVICE
HIDDEN_SIZE = 768
ROUTER_SIZE = 256
NUM_HEADS = 4
TOP_K = 3
NUM_EXPERTS = 11

BATCH_SIZE = 2 if local_config.QUICK_TEST else 8
CALIBRATION_SAMPLES = 10 if local_config.QUICK_TEST else 200
LEARNING_RATE = 1e-4
NUM_EPOCHS = 1

SELF_EXPERT_BIAS = 1.5
COMPATIBILITY_WEIGHT = 1.0
PERSONALIZATION_WEIGHT = 0.5

# ==========================================================
# 3. Client List & Identity Mapping
# ==========================================================

CLIENTS = sorted(
    [c for c in os.listdir(ADAPTER_PATH) if c.startswith("client_")],
    key=lambda x: int(x.split("_")[1])
) if os.path.exists(ADAPTER_PATH) else [f"client_{i}" for i in range(1, 12)]

CLIENT_TO_ID = {client: index for index, client in enumerate(CLIENTS)}

# Lazy load configuration
CLIENT_CONFIG = None

def get_client_config():
    global CLIENT_CONFIG
    if CLIENT_CONFIG is None:
        if os.path.exists(CONFIG_PATH):
            with open(CONFIG_PATH, "r") as f:
                CLIENT_CONFIG = json.load(f)
        else:
            CLIENT_CONFIG = {}
    return CLIENT_CONFIG

RANK_VALUES = {4: 0, 8: 1, 12: 2, 16: 3}

def get_client_features(client_name, batch_size=1):
    cfg = get_client_config().get(client_name, {"active_rank": 8, "personalization_score": 0.35})
    rank = int(cfg.get("active_rank", cfg.get("rank", 8)))
    score = float(cfg.get("personalization_score", cfg.get("score", 0.35)))
    
    rank_idx = RANK_VALUES.get(rank, 1)
    client_idx = CLIENT_TO_ID.get(client_name, 0)
    
    client_ids = torch.full((batch_size,), client_idx, dtype=torch.long, device=DEVICE)
    rank_ids = torch.full((batch_size,), rank_idx, dtype=torch.long, device=DEVICE)
    scores = torch.full((batch_size,), score, dtype=torch.float32, device=DEVICE)
    return client_ids, rank_ids, scores

# ==========================================================
# 4. Personalized CAGER v3 Router Model Definition
# ==========================================================

class PersonalizedCAGERv3(nn.Module):
    def __init__(self):
        super().__init__()
        self.client_embedding = nn.Embedding(NUM_EXPERTS, ROUTER_SIZE)
        self.rank_embedding = nn.Embedding(4, ROUTER_SIZE)
        self.score_projection = nn.Sequential(
            nn.Linear(1, ROUTER_SIZE),
            nn.GELU(),
            nn.Linear(ROUTER_SIZE, ROUTER_SIZE)
        )
        self.query_projection = nn.Linear(HIDDEN_SIZE, ROUTER_SIZE)
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=ROUTER_SIZE,
            num_heads=NUM_HEADS,
            batch_first=True
        )
        self.norm = nn.LayerNorm(ROUTER_SIZE)
        self.client_projection = nn.Sequential(
            nn.Linear(ROUTER_SIZE, ROUTER_SIZE),
            nn.GELU(),
            nn.Linear(ROUTER_SIZE, ROUTER_SIZE)
        )
        self.expert_embedding = nn.Parameter(
            torch.randn(NUM_EXPERTS, ROUTER_SIZE) * 0.02
        )
        self.compatibility = nn.Sequential(
            nn.Linear(ROUTER_SIZE * 2, ROUTER_SIZE),
            nn.GELU(),
            nn.Linear(ROUTER_SIZE, 1)
        )
        self.global_router = nn.Sequential(
            nn.Linear(ROUTER_SIZE, ROUTER_SIZE),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(ROUTER_SIZE, NUM_EXPERTS)
        )

    def forward(self, hidden_states, client_ids, rank_ids, personalization_scores):
        query = self.query_projection(hidden_states)
        client_token = self.client_embedding(client_ids)
        rank_token = self.rank_embedding(rank_ids)
        score_token = self.score_projection(personalization_scores.unsqueeze(-1))

        context = torch.stack([client_token, rank_token, score_token], dim=1)
        attended, attention_weights = self.cross_attention(
            query=query, key=context, value=context
        )
        attended = self.norm(attended)

        pooled_input = query.mean(dim=1)
        pooled_context = attended.mean(dim=1)

        client_representation = pooled_input + pooled_context
        client_representation = self.client_projection(client_representation)

        global_logits = self.global_router(client_representation)
        batch_size = client_representation.size(0)

        client_expanded = client_representation.unsqueeze(1).expand(batch_size, NUM_EXPERTS, ROUTER_SIZE)
        expert_expanded = self.expert_embedding.unsqueeze(0).expand(batch_size, NUM_EXPERTS, ROUTER_SIZE)

        compatibility_input = torch.cat([client_expanded, expert_expanded], dim=-1)
        compatibility_scores = self.compatibility(compatibility_input).squeeze(-1)

        self_expert_mask = torch.zeros_like(global_logits)
        self_expert_mask.scatter_(1, client_ids.unsqueeze(1), SELF_EXPERT_BIAS)

        final_logits = (
            global_logits +
            COMPATIBILITY_WEIGHT * compatibility_scores +
            PERSONALIZATION_WEIGHT * self_expert_mask
        )
        gates = F.softmax(final_logits, dim=-1)

        return gates, final_logits, attention_weights, compatibility_scores

# ==========================================================
# 5. Training Helpers & Execution
# ==========================================================

def load_calibration_dataset(client_name):
    path = os.path.join(CLIENT_PATH, client_name, "train")
    dataset = load_from_disk(path)
    number = min(CALIBRATION_SAMPLES, len(dataset))
    dataset = dataset.shuffle(seed=42).select(range(number))
    if "text" in dataset.column_names:
        dataset = dataset.remove_columns(["text"])
    return dataset

def calculate_expert_losses(expert_model, input_ids, attention_mask, labels):
    losses = []
    for expert_index, client in enumerate(CLIENTS):
        adapter_name = "default" if expert_index == 0 else client
        expert_model.set_adapter(adapter_name)
        with torch.no_grad():
            outputs = expert_model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        shift_logits = logits[:, :-1, :]
        shift_labels = labels[:, 1:]

        token_loss = F.cross_entropy(
            shift_logits.reshape(-1, shift_logits.size(-1)),
            shift_labels.reshape(-1),
            ignore_index=-100,
            reduction="none"
        ).reshape(shift_labels.shape)

        valid_mask = (shift_labels != -100)
        sample_loss = (token_loss * valid_mask).sum(dim=1) / valid_mask.sum(dim=1).clamp(min=1)
        losses.append(sample_loss)

    return torch.stack(losses, dim=1)

def create_target(expert_losses):
    return F.softmax(-expert_losses / 0.5, dim=-1)

def main():
    print("=" * 70)
    print("PERSONALIZED CAGER v3 TRAINING")
    print("=" * 70)
    print(f"Experts              : {NUM_EXPERTS}")
    print(f"Top-K                : {TOP_K}")
    print(f"Calibration Samples  : {CALIBRATION_SAMPLES}")
    print(f"Self Expert Bias     : {SELF_EXPERT_BIAS}")
    print(f"Compatibility Weight : {COMPATIBILITY_WEIGHT}")
    print(f"Personalization Wt.  : {PERSONALIZATION_WEIGHT}")
    print(f"Device               : {DEVICE}")
    print("Clients              :", CLIENTS)
    print("=" * 70)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    print("\nLoading frozen base model...")
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
    base_model.eval()
    for param in base_model.parameters():
        param.requires_grad = False

    print("\nLoading Adaptive PEFT Experts...")
    first_client = CLIENTS[0]
    first_adapter = os.path.join(ADAPTER_PATH, first_client)
    expert_model = PeftModel.from_pretrained(base_model, first_adapter, is_trainable=False)

    for client in CLIENTS[1:]:
        adapter_path = os.path.join(ADAPTER_PATH, client)
        print(f"Loading expert: {client}")
        expert_model.load_adapter(adapter_path, adapter_name=client)

    for param in expert_model.parameters():
        param.requires_grad = False
    expert_model.to(DEVICE)
    expert_model.eval()

    ROUTER_FILE = os.path.join(SAVE_PATH, "personalized_cager_v3_router.pt")
    STATE_FILE = os.path.join(SAVE_PATH, "training_state.json")

    router = PersonalizedCAGERv3().to(DEVICE)
    optimizer = torch.optim.AdamW(router.parameters(), lr=LEARNING_RATE)
    completed_clients = []

    if os.path.exists(ROUTER_FILE):
        print("\nExisting CAGER v3 checkpoint found.")
        checkpoint = torch.load(ROUTER_FILE, map_location=DEVICE)
        if isinstance(checkpoint, dict) and "router" in checkpoint:
            router.load_state_dict(checkpoint["router"])
            if "optimizer" in checkpoint:
                try:
                    optimizer.load_state_dict(checkpoint["optimizer"])
                except Exception:
                    pass
        else:
            router.load_state_dict(checkpoint)
        print("Router checkpoint loaded.")

    if os.path.exists(STATE_FILE):
        with open(STATE_FILE, "r") as f:
            state = json.load(f)
        completed_clients = state.get("completed_clients", [])
        print("Completed clients:", completed_clients)

    router.train()

    for client_name in CLIENTS:
        if client_name in completed_clients:
            print(f"\n{client_name} already completed. Skipping...")
            continue

        print("\n" + "=" * 70)
        print(f"Personalized CAGER v3 Training : {client_name}")
        print("=" * 70)

        dataset = load_calibration_dataset(client_name)
        print(f"Calibration Samples : {len(dataset)}")
        loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=data_collator)

        total_loss = 0.0
        batch_count = 0

        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            with torch.no_grad():
                outputs = base_model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
                hidden_states = outputs.hidden_states[-1].detach()
                expert_losses = calculate_expert_losses(expert_model, input_ids, attention_mask, labels)
                routing_target = create_target(expert_losses)

            client_ids, rank_ids, scores = get_client_features(client_name, input_ids.size(0))
            gates, final_logits, attention_weights, compatibility_scores = router(
                hidden_states, client_ids, rank_ids, scores
            )

            routing_loss = F.kl_div(
                torch.log(gates.clamp(min=1e-8)),
                routing_target,
                reduction="batchmean"
            )

            own_expert_probability = gates.gather(1, client_ids.unsqueeze(1)).squeeze(1)
            self_expert_loss = -torch.log(own_expert_probability + 1e-8).mean()
            compatibility_regularization = compatibility_scores.pow(2).mean()

            loss = routing_loss + 0.15 * self_expert_loss + 0.001 * compatibility_regularization

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(router.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()
            batch_count += 1

            if batch_count % 10 == 0:
                print(f"Batch {batch_count} | Loss {loss.item():.4f} | Routing {routing_loss.item():.4f} | Self {self_expert_loss.item():.4f}")

        average_loss = total_loss / max(batch_count, 1)
        print(f"\n{client_name} Average Loss : {average_loss:.4f}")
        completed_clients.append(client_name)

        torch.save({"router": router.state_dict(), "optimizer": optimizer.state_dict()}, ROUTER_FILE)
        with open(STATE_FILE, "w") as f:
            json.dump({
                "completed_clients": completed_clients,
                "calibration_samples": CALIBRATION_SAMPLES,
                "top_k": TOP_K,
                "self_expert_bias": SELF_EXPERT_BIAS,
                "personalization_weight": PERSONALIZATION_WEIGHT,
                "compatibility_weight": COMPATIBILITY_WEIGHT
            }, f, indent=4)
        print("CAGER v3 checkpoint saved.")

    router.eval()
    torch.save({
        "router": router.state_dict(),
        "config": {
            "num_experts": NUM_EXPERTS,
            "top_k": TOP_K,
            "self_expert_bias": SELF_EXPERT_BIAS,
            "personalization_weight": PERSONALIZATION_WEIGHT,
            "compatibility_weight": COMPATIBILITY_WEIGHT
        }
    }, ROUTER_FILE)

    print("\n" + "=" * 70)
    print("PERSONALIZED CAGER v3 TRAINING COMPLETED")
    print("=" * 70)
    print(f"Clients Completed : {len(completed_clients)} / {NUM_EXPERTS}")
    print(f"Router Saved      : {ROUTER_FILE}")
    print("=" * 70)

if __name__ == "__main__":
    main()

PERSONALIZED CAGER v3 TRAINING
Experts              : 11
Top-K                : 3
Calibration Samples  : 10
Self Expert Bias     : 1.5
Compatibility Weight : 1.0
Personalization Wt.  : 0.5
Device               : cpu
Clients              : ['client_1', 'client_2', 'client_3', 'client_4', 'client_5', 'client_6', 'client_7', 'client_8', 'client_9', 'client_10', 'client_11']

Loading frozen base model...


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 7771.21it/s]



Loading Adaptive PEFT Experts...
Loading expert: client_2
Loading expert: client_3
Loading expert: client_4
Loading expert: client_5
Loading expert: client_6
Loading expert: client_7
Loading expert: client_8
Loading expert: client_9
Loading expert: client_10
Loading expert: client_11

Existing CAGER v3 checkpoint found.
Router checkpoint loaded.
Completed clients: ['client_1', 'client_2', 'client_3', 'client_4', 'client_5', 'client_6', 'client_7', 'client_8', 'client_9', 'client_10', 'client_11']

client_1 already completed. Skipping...

client_2 already completed. Skipping...

client_3 already completed. Skipping...

client_4 already completed. Skipping...

client_5 already completed. Skipping...

client_6 already completed. Skipping...

client_7 already completed. Skipping...

client_8 already completed. Skipping...

client_9 already completed. Skipping...

client_10 already completed. Skipping...

client_11 already completed. Skipping...

PERSONALIZED CAGER v3 TRAINING COMPLETED
Cl

## Module 8E: Personalized CAGER v3 Evaluation
Evaluates the CAGER v3 router output.

In [17]:
# ==========================================================
# Module 8E
# Personalized CAGER v3 Evaluation
#
# Evaluates:
#
#   Adaptive PEFT
#          +
#   Personalized CAGER v3
#
# Frozen:
#   - DistilGPT2 base model
#   - All 11 LoRA experts
#   - CAGER v3 router
#
# No training is performed.
#
# Metrics:
#   1. Loss
#   2. Perplexity
#   3. Top-1 Accuracy
#   4. Top-5 Accuracy
#   5. Accuracy gain over Adaptive PEFT
#   6. Perplexity change
#   7. Expert gate distribution
#   8. Top-K expert selection
#   9. Own-expert selection rate
# ==========================================================


import os
import json
import math
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

from datasets import load_from_disk
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling
)

from peft import PeftModel

# Import local configuration
try:
    MODULE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    MODULE_DIR = os.path.abspath(os.path.join(os.getcwd(), 'code', 'modules'))
if MODULE_DIR not in sys.path:
    sys.path.append(MODULE_DIR)
import local_config

# ==========================================================
# 1. PATHS
# ==========================================================

PROJECT_PATH = local_config.WORKSPACE_DIR
CLIENT_PATH = local_config.CLIENT_PATH
ADAPTER_PATH = local_config.MODEL_SAVE_PATH
CONFIG_PATH = local_config.CONFIG_PATH
CAGER_V3_PATH = os.path.join(PROJECT_PATH, "cager_v3_models")
BASELINE_PATH = local_config.EVALUATION_PATH
RESULT_PATH = os.path.join(PROJECT_PATH, "cager_v3_evaluation")

os.makedirs(
    RESULT_PATH,
    exist_ok=True
)


# ==========================================================
# 2. CONFIGURATION
# ==========================================================

MODEL_NAME = "distilgpt2"
DEVICE = local_config.DEVICE

HIDDEN_SIZE = 768
ROUTER_SIZE = 256
NUM_HEADS = 4
NUM_EXPERTS = 11
TOP_K = 3
BATCH_SIZE = 8


CLIENTS = sorted(
    [
        c
        for c in os.listdir(
            ADAPTER_PATH
        )
        if c.startswith("client_")
    ],
    key=lambda x: int(
        x.split("_")[1]
    )
)


if len(CLIENTS) != NUM_EXPERTS:
    raise ValueError(
        f"Expected {NUM_EXPERTS} experts but found {len(CLIENTS)}"
    )


CLIENT_TO_ID = {
    client: index
    for index, client in enumerate(
        CLIENTS
    )
}


print("=" * 75)
print("PERSONALIZED CAGER v3 EVALUATION")
print("=" * 75)
print(f"Number of Experts : {NUM_EXPERTS}")
print(f"Top-K             : {TOP_K}")
print(f"Batch Size        : {BATCH_SIZE}")
print(f"Device            : {DEVICE}")
print("Clients           :", CLIENTS)
print("=" * 75)


# ==========================================================
# 3. VERIFY REQUIRED FILES
# ==========================================================

ROUTER_PATH = os.path.join(
    CAGER_V3_PATH,
    "personalized_cager_v3_router.pt"
)

BASELINE_CSV = os.path.join(
    BASELINE_PATH,
    "personalized_evaluation.csv"
)


if not os.path.exists(ROUTER_PATH):
    raise FileNotFoundError(
        f"\nCAGER v3 router not found:\n{ROUTER_PATH}\n\nMake sure Module 8D completed."
    )


if not os.path.exists(BASELINE_CSV):
    raise FileNotFoundError(
        f"\nAdaptive PEFT baseline not found:\n{BASELINE_CSV}"
    )


print("\nRequired files verified.")


# ==========================================================
# 4. LOAD CLIENT CONFIGURATION
# ==========================================================

if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(CONFIG_PATH)


with open(CONFIG_PATH, "r") as f:
    CLIENT_CONFIG = json.load(f)


# ==========================================================
# 5. TOKENIZER
# ==========================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token


data_collator = (
    DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )
)


# ==========================================================
# 6. LORA RANK ENCODING
# ==========================================================

RANK_VALUES = {
    4: 0,
    8: 1,
    12: 2,
    16: 3
}


def get_client_features(
    client_name,
    batch_size
):
    config = CLIENT_CONFIG[client_name]
    rank = int(config["active_rank"])
    score = float(config["personalization_score"])

    if rank not in RANK_VALUES:
        raise ValueError(
            f"Unsupported LoRA rank {rank}"
        )

    client_ids = torch.full(
        (batch_size,),
        CLIENT_TO_ID[client_name],
        dtype=torch.long,
        device=DEVICE
    )

    rank_ids = torch.full(
        (batch_size,),
        RANK_VALUES[rank],
        dtype=torch.long,
        device=DEVICE
    )

    scores = torch.full(
        (batch_size,),
        score,
        dtype=torch.float32,
        device=DEVICE
    )

    return (
        client_ids,
        rank_ids,
        scores
    )


# ==========================================================
# 7. PERSONALIZED CAGER v3 ARCHITECTURE
# ==========================================================

class PersonalizedCAGERv3(
    nn.Module
):

    def __init__(self):
        super().__init__()

        # --------------------------------------------------
        # Client identity
        # --------------------------------------------------
        self.client_embedding = nn.Embedding(
            NUM_EXPERTS,
            ROUTER_SIZE
        )

        # --------------------------------------------------
        # LoRA rank
        # --------------------------------------------------
        self.rank_embedding = nn.Embedding(
            4,
            ROUTER_SIZE
        )

        # --------------------------------------------------
        # Personalization score
        # --------------------------------------------------
        self.score_projection = nn.Sequential(
            nn.Linear(1, ROUTER_SIZE),
            nn.GELU(),
            nn.Linear(ROUTER_SIZE, ROUTER_SIZE)
        )

        # --------------------------------------------------
        # Input representation
        # --------------------------------------------------
        self.query_projection = nn.Linear(
            HIDDEN_SIZE,
            ROUTER_SIZE
        )

        # --------------------------------------------------
        # Cross attention
        # --------------------------------------------------
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=ROUTER_SIZE,
            num_heads=NUM_HEADS,
            batch_first=True
        )

        # --------------------------------------------------
        # Normalization
        # --------------------------------------------------
        self.norm = nn.LayerNorm(
            ROUTER_SIZE
        )

        # --------------------------------------------------
        # Personalized representation
        # --------------------------------------------------
        self.client_projection = nn.Sequential(
            nn.Linear(
                ROUTER_SIZE,
                ROUTER_SIZE
            ),
            nn.GELU(),
            nn.Linear(
                ROUTER_SIZE,
                ROUTER_SIZE
            )
        )

        # --------------------------------------------------
        # Expert representations
        # --------------------------------------------------
        self.expert_embedding = nn.Parameter(
            torch.randn(
                NUM_EXPERTS,
                ROUTER_SIZE
            )
            * 0.02
        )

        # --------------------------------------------------
        # Client-expert compatibility
        # --------------------------------------------------
        self.compatibility = nn.Sequential(
            nn.Linear(
                ROUTER_SIZE * 2,
                ROUTER_SIZE
            ),
            nn.GELU(),
            nn.Linear(
                ROUTER_SIZE,
                1
            )
        )

        # --------------------------------------------------
        # Global router
        # --------------------------------------------------
        self.global_router = nn.Sequential(
            nn.Linear(
                ROUTER_SIZE,
                ROUTER_SIZE
            ),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(
                ROUTER_SIZE,
                NUM_EXPERTS
            )
        )

    def forward(
        self,
        hidden_states,
        client_ids,
        rank_ids,
        personalization_scores
    ):
        # ==================================================
        # Query
        # ==================================================
        query = self.query_projection(
            hidden_states
        )

        # ==================================================
        # Client context
        # ==================================================
        client_token = (
            self.client_embedding(client_ids)
        )

        rank_token = (
            self.rank_embedding(rank_ids)
        )

        score_token = (
            self.score_projection(
                personalization_scores
                .unsqueeze(-1)
            )
        )

        context = torch.stack(
            [
                client_token,
                rank_token,
                score_token
            ],
            dim=1
        )

        # ==================================================
        # Cross Attention
        # ==================================================
        attended, attention_weights = (
            self.cross_attention(
                query=query,
                key=context,
                value=context
            )
        )

        attended = self.norm(
            attended
        )

        # ==================================================
        # Pool
        # ==================================================
        pooled_input = query.mean(dim=1)
        pooled_context = attended.mean(dim=1)

        # ==================================================
        # Personalized representation
        # ==================================================
        client_representation = (
            pooled_input
            +
            pooled_context
        )

        client_representation = (
            self.client_projection(
                client_representation
            )
        )

        # ==================================================
        # Global routing
        # ==================================================
        global_logits = (
            self.global_router(
                client_representation
            )
        )

        # ==================================================
        # Expert-client compatibility
        # ==================================================
        batch_size = (
            client_representation
            .size(0)
        )

        client_expanded = (
            client_representation
            .unsqueeze(1)
            .expand(
                batch_size,
                NUM_EXPERTS,
                ROUTER_SIZE
            )
        )

        expert_expanded = (
            self.expert_embedding
            .unsqueeze(0)
            .expand(
                batch_size,
                NUM_EXPERTS,
                ROUTER_SIZE
            )
        )

        compatibility_input = torch.cat(
            [
                client_expanded,
                expert_expanded
            ],
            dim=-1
        )

        compatibility_scores = (
            self.compatibility(
                compatibility_input
            )
            .squeeze(-1)
        )

        # ==================================================
        # IMPORTANT:
        #
        # The self-expert bias is part of the saved v3
        # architecture.
        # ==================================================
        self_expert_mask = torch.zeros_like(
            global_logits
        )

        self_expert_mask.scatter_(
            1,
            client_ids.unsqueeze(1),
            1.5
        )

        # ==================================================
        # Final routing logits
        # ==================================================
        final_logits = (
            global_logits
            +
            compatibility_scores
            +
            0.5
            *
            self_expert_mask
        )

        gates = F.softmax(
            final_logits,
            dim=-1
        )

        return (
            gates,
            final_logits,
            attention_weights,
            compatibility_scores
        )


# ==========================================================
# 8. LOAD CAGER v3 ROUTER
# ==========================================================

print("\nLoading CAGER v3 router...")

router = PersonalizedCAGERv3().to(
    DEVICE
)

checkpoint = torch.load(
    ROUTER_PATH,
    map_location=DEVICE
)

# ----------------------------------------------------------
# Support both checkpoint formats
# ----------------------------------------------------------
if isinstance(checkpoint, dict) and "router" in checkpoint:
    router.load_state_dict(
        checkpoint["router"]
    )
else:
    router.load_state_dict(
        checkpoint
    )

router.eval()

for parameter in router.parameters():
    parameter.requires_grad = False

print("CAGER v3 router loaded successfully.")


# ==========================================================
# 9. LOAD BASE MODEL
# ==========================================================

print("\nLoading frozen DistilGPT2...")

base_model = (
    AutoModelForCausalLM
    .from_pretrained(
        MODEL_NAME
    )
)

base_model.to(
    DEVICE
)
base_model.eval()

for parameter in base_model.parameters():
    parameter.requires_grad = False

print("Base model loaded and frozen.")


# ==========================================================
# 10. LOAD ALL FROZEN EXPERTS
# ==========================================================

print("\n")
print("=" * 75)
print("Loading Frozen Adaptive PEFT Experts")
print("=" * 75)

first_client = CLIENTS[0]

first_adapter_path = os.path.join(
    ADAPTER_PATH,
    first_client
)

expert_model = (
    PeftModel.from_pretrained(
        base_model,
        first_adapter_path,
        is_trainable=False
    )
)

for client in CLIENTS[1:]:
    adapter_path = os.path.join(
        ADAPTER_PATH,
        client
    )
    print(f"Loading expert: {client}")
    expert_model.load_adapter(
        adapter_path,
        adapter_name=client
    )

for parameter in expert_model.parameters():
    parameter.requires_grad = False

expert_model.to(
    DEVICE
)
expert_model.eval()

print("\nAll 11 Adaptive PEFT experts loaded.")
print("Expert parameters are frozen.")


# ==========================================================
# 11. TOP-K ROUTING
# ==========================================================

def apply_top_k(
    gates
):
    values, indices = torch.topk(
        gates,
        k=TOP_K,
        dim=-1
    )

    values = (
        values
        /
        (
            values.sum(
                dim=-1,
                keepdim=True
            )
            + 1e-8
        )
    )

    return (
        indices,
        values
    )


# ==========================================================
# 12. EXPERT PREDICTION
# ==========================================================

def get_selected_expert_logits(
    input_ids,
    attention_mask,
    selected_indices,
    selected_weights
):
    batch_size = input_ids.size(0)
    sequence_length = input_ids.size(1)
    vocab_size = expert_model.config.vocab_size

    combined_logits = torch.zeros(
        batch_size,
        sequence_length,
        vocab_size,
        device=DEVICE,
        dtype=torch.float32
    )

    # ------------------------------------------------------
    # Unique experts selected in this batch
    # ------------------------------------------------------
    unique_experts = torch.unique(
        selected_indices
    )

    for expert_tensor in unique_experts:
        expert_index = (
            expert_tensor.item()
        )

        # --------------------------------------------------
        # Adapter name
        # --------------------------------------------------
        if expert_index == 0:
            adapter_name = "default"
        else:
            adapter_name = CLIENTS[expert_index]

        # --------------------------------------------------
        # Samples that selected this expert
        # --------------------------------------------------
        sample_mask = (
            selected_indices
            == expert_index
        ).any(dim=1)

        sample_positions = torch.where(
            sample_mask
        )[0]

        if sample_positions.numel() == 0:
            continue

        expert_model.set_adapter(
            adapter_name
        )

        # --------------------------------------------------
        # Expert forward pass
        # --------------------------------------------------
        with torch.no_grad():
            outputs = expert_model(
                input_ids=input_ids[sample_positions],
                attention_mask=attention_mask[sample_positions]
            )

        expert_logits = outputs.logits

        # --------------------------------------------------
        # Add expert contribution
        # --------------------------------------------------
        for local_position, global_position in enumerate(
            sample_positions
        ):
            for k in range(
                TOP_K
            ):
                if (
                    selected_indices[global_position, k].item()
                    == expert_index
                ):
                    weight = (
                        selected_weights[global_position, k]
                    )

                    combined_logits[global_position] += (
                        weight
                        *
                        expert_logits[local_position]
                    )

    return combined_logits


# ==========================================================
# 13. LOAD BASELINE
# ==========================================================

baseline_df = pd.read_csv(
    BASELINE_CSV
)

print("\nAdaptive PEFT baseline loaded.")


# ==========================================================
# 14. EVALUATE ONE CLIENT
# ==========================================================

def evaluate_client(
    client_name
):
    print("\n")
    print("=" * 75)
    print(f"CAGER v3 Evaluation : {client_name}")
    print("=" * 75)

    # ------------------------------------------------------
    # Test data
    # ------------------------------------------------------
    test_path = os.path.join(
        CLIENT_PATH,
        client_name,
        "test"
    )

    test_dataset = load_from_disk(
        test_path
    )

    if "text" in test_dataset.column_names:
        test_dataset = test_dataset.remove_columns(
            ["text"]
        )

    # Apply Quick Test sub-sampling to keep verification fast
    if local_config.QUICK_TEST:
        test_dataset = test_dataset.select(range(min(10, len(test_dataset))))
        print(f"  [Quick Test] Subsampled test dataset to {len(test_dataset)} samples.")

    loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=data_collator
    )

    print(
        f"Test Samples : {len(test_dataset)}"
    )

    # ------------------------------------------------------
    # Metrics
    # ------------------------------------------------------
    total_loss = 0.0
    total_tokens = 0
    top1_correct = 0
    top5_correct = 0

    # ------------------------------------------------------
    # Routing statistics
    # ------------------------------------------------------
    gate_sum = torch.zeros(
        NUM_EXPERTS,
        device=DEVICE
    )

    selection_count = {
        client: 0
        for client in CLIENTS
    }

    own_expert_selected = 0
    total_topk_slots = 0
    total_samples = 0

    # ------------------------------------------------------
    # Evaluation
    # ------------------------------------------------------
    with torch.no_grad():
        for batch_index, batch in enumerate(
            loader
        ):
            input_ids = (
                batch["input_ids"].to(DEVICE)
            )

            attention_mask = (
                batch["attention_mask"].to(DEVICE)
            )

            labels = (
                batch["labels"].to(DEVICE)
            )

            batch_size = input_ids.size(0)
            total_samples += batch_size

            # ==================================================
            # Base hidden representation
            # ==================================================
            base_outputs = base_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )

            hidden_states = (
                base_outputs
                .hidden_states[-1]
            )

            # ==================================================
            # Client features
            # ==================================================
            (
                client_ids,
                rank_ids,
                score_tensor
            ) = get_client_features(
                client_name,
                batch_size
            )

            # ==================================================
            # CAGER v3
            # ==================================================
            (
                gates,
                final_logits,
                attention_weights,
                compatibility_scores
            ) = router(
                hidden_states,
                client_ids,
                rank_ids,
                score_tensor
            )

            # ==================================================
            # Gate statistics
            # ==================================================
            gate_sum += gates.sum(
                dim=0
            )

            # ==================================================
            # Top-K
            # ==================================================
            (
                selected_indices,
                selected_weights
            ) = apply_top_k(
                gates
            )

            # ==================================================
            # Selection statistics
            # ==================================================
            for row in selected_indices:
                row_list = [
                    x.item()
                    for x in row
                ]

                for expert_index in row_list:
                    expert_name = CLIENTS[expert_index]
                    selection_count[expert_name] += 1

                own_id = CLIENT_TO_ID[client_name]

                if own_id in row_list:
                    own_expert_selected += 1

                total_topk_slots += TOP_K

            # ==================================================
            # Expert prediction
            # ==================================================
            combined_logits = (
                get_selected_expert_logits(
                    input_ids,
                    attention_mask,
                    selected_indices,
                    selected_weights
                )
            )

            # ==================================================
            # Causal shift
            # ==================================================
            shift_logits = (
                combined_logits[:, :-1, :]
            )

            shift_labels = (
                labels[:, 1:]
            )

            valid_mask = (
                shift_labels != -100
            )

            # ==================================================
            # Loss
            # ==================================================
            token_loss = F.cross_entropy(
                shift_logits.reshape(
                    -1,
                    shift_logits.size(-1)
                ),
                shift_labels.reshape(-1),
                ignore_index=-100,
                reduction="sum"
            )

            total_loss += token_loss.item()
            valid_tokens = (
                valid_mask.sum().item()
            )
            total_tokens += valid_tokens

            # ==================================================
            # Top-1
            # ==================================================
            predictions = torch.argmax(
                shift_logits,
                dim=-1
            )

            top1_correct += (
                (
                    predictions
                    == shift_labels
                )
                &
                valid_mask
            ).sum().item()

            # ==================================================
            # Top-5
            # ==================================================
            top5_predictions = torch.topk(
                shift_logits,
                k=5,
                dim=-1
            ).indices

            expanded_labels = (
                shift_labels
                .unsqueeze(-1)
                .expand_as(
                    top5_predictions
                )
            )

            top5_correct += (
                (
                    top5_predictions
                    == expanded_labels
                ).any(dim=-1)
                &
                valid_mask
            ).sum().item()

            # ==================================================
            # Progress
            # ==================================================
            if (
                (batch_index + 1)
                % 100
                == 0
            ):
                print(
                    f"Processed batches : {batch_index + 1}"
                )

    # ======================================================
    # Final Metrics
    # ======================================================
    average_loss = (
        total_loss
        /
        max(
            total_tokens,
            1
        )
    )

    perplexity = math.exp(
        min(
            average_loss,
            20
        )
    )

    top1_accuracy = (
        top1_correct
        /
        max(
            total_tokens,
            1
        )
    ) * 100

    top5_accuracy = (
        top5_correct
        /
        max(
            total_tokens,
            1
        )
    ) * 100

    # ======================================================
    # Gate distribution
    # ======================================================
    gate_total = gate_sum.sum()

    gate_distribution = {
        client:
        float(
            gate_sum[index]
            /
            gate_total
        )
        for index, client
        in enumerate(CLIENTS)
    }

    # ======================================================
    # Own expert selection
    # ======================================================
    own_expert_selection_rate = (
        own_expert_selected
        /
        max(
            total_samples,
            1
        )
    ) * 100

    # ======================================================
    # Adaptive PEFT baseline
    # ======================================================
    baseline_row = baseline_df[
        baseline_df["client"] == client_name
    ]

    baseline_top1 = None
    baseline_top5 = None
    baseline_ppl = None

    if len(baseline_row) > 0:
        baseline_top1 = float(
            baseline_row["personalized_top1_accuracy"].iloc[0]
        )
        baseline_top5 = float(
            baseline_row["personalized_top5_accuracy"].iloc[0]
        )
        baseline_ppl = float(
            baseline_row["personalized_perplexity"].iloc[0]
        )

    # ======================================================
    # Gains
    # ======================================================
    top1_gain = None
    top5_gain = None
    perplexity_change = None

    if baseline_top1 is not None:
        top1_gain = (
            top1_accuracy
            -
            baseline_top1
        )

    if baseline_top5 is not None:
        top5_gain = (
            top5_accuracy
            -
            baseline_top5
        )

    if baseline_ppl is not None:
        perplexity_change = (
            baseline_ppl
            -
            perplexity
        )

    # ======================================================
    # DISPLAY
    # ======================================================
    print("\n")
    print("=" * 75)
    print(f"CAGER v3 RESULTS : {client_name}")
    print("=" * 75)
    print(f"Loss              : {average_loss:.4f}")
    print(f"Perplexity        : {perplexity:.4f}")
    print(f"Top-1 Accuracy    : {top1_accuracy:.2f}%")
    print(f"Top-5 Accuracy    : {top5_accuracy:.2f}%")
    print(f"Tokens Evaluated  : {total_tokens}")
    print()
    print(
        f"Adaptive PEFT Top-1 : {baseline_top1:.2f}%"
        if baseline_top1 is not None
        else "Adaptive PEFT Top-1 : N/A"
    )
    print(f"CAGER v3 Top-1      : {top1_accuracy:.2f}%")
    if top1_gain is not None:
        print(f"Top-1 Gain          : {top1_gain:+.2f} pp")

    print()
    print(
        f"Adaptive PEFT Top-5 : {baseline_top5:.2f}%"
        if baseline_top5 is not None
        else "Adaptive PEFT Top-5 : N/A"
    )
    print(f"CAGER v3 Top-5      : {top5_accuracy:.2f}%")
    if top5_gain is not None:
        print(f"Top-5 Gain          : {top5_gain:+.2f} pp")

    print()
    print(
        f"Adaptive PEFT PPL   : {baseline_ppl:.4f}"
        if baseline_ppl is not None
        else "Adaptive PEFT PPL   : N/A"
    )
    print(f"CAGER v3 PPL        : {perplexity:.4f}")
    if perplexity_change is not None:
        print(f"PPL Change          : {perplexity_change:+.4f}")

    print()
    print(f"Own Expert Selection: {own_expert_selection_rate:.2f}%")
    print("=" * 75)

    # ======================================================
    # Expert Distribution
    # ======================================================
    print("\n")
    print("EXPERT GATE DISTRIBUTION")
    print("-" * 75)

    for index, client in enumerate(
        CLIENTS
    ):
        print(
            f"{client:<12}Gate = {gate_distribution[client]:.4f}    Top-K = {selection_count[client]}"
        )

    print("=" * 75)

    # ======================================================
    # Save result
    # ======================================================
    result = {
        "client":
            client_name,
        "tokens":
            int(total_tokens),
        "loss":
            float(average_loss),
        "perplexity":
            float(perplexity),
        "top1_accuracy":
            float(top1_accuracy),
        "top5_accuracy":
            float(top5_accuracy),
        "adaptive_peft_top1":
            baseline_top1,
        "adaptive_peft_top5":
            baseline_top5,
        "adaptive_peft_perplexity":
            baseline_ppl,
        "top1_gain":
            top1_gain,
        "top5_gain":
            top5_gain,
        "perplexity_change":
            perplexity_change,
        "own_expert_selection_rate":
            float(own_expert_selection_rate),
        "expert_gate_distribution":
            gate_distribution,
        "expert_selection_count":
            selection_count
    }

    output_path = os.path.join(
        RESULT_PATH,
        f"{client_name}_cager_v3.json"
    )

    with open(output_path, "w") as f:
        json.dump(
            result,
            f,
            indent=4
        )

    print(f"\nSaved : {output_path}")

    return result


# ==========================================================
# 15. EVALUATE ALL CLIENTS
# ==========================================================

all_results = []

print("\n")
print("=" * 75)
print("STARTING CAGER v3 EVALUATION")
print("=" * 75)

# Limit evaluated clients during quick test
clients_to_eval = CLIENTS

for client in clients_to_eval:
    result = evaluate_client(
        client
    )
    all_results.append(
        result
    )


# ==========================================================
# 16. SUMMARY TABLE
# ==========================================================

summary_rows = []

for result in all_results:
    summary_rows.append({
        "client":
            result["client"],
        "adaptive_peft_top1":
            result["adaptive_peft_top1"],
        "cager_v3_top1":
            result["top1_accuracy"],
        "top1_gain_pp":
            result["top1_gain"],
        "adaptive_peft_top5":
            result["adaptive_peft_top5"],
        "cager_v3_top5":
            result["top5_accuracy"],
        "top5_gain_pp":
            result["top5_gain"],
        "adaptive_peft_ppl":
            result["adaptive_peft_perplexity"],
        "cager_v3_ppl":
            result["perplexity"],
        "ppl_change":
            result["perplexity_change"],
        "own_expert_selection":
            result["own_expert_selection_rate"]
    })

summary_df = pd.DataFrame(
    summary_rows
)


# ==========================================================
# 17. SAVE SUMMARY CSV
# ==========================================================

summary_csv = os.path.join(
    RESULT_PATH,
    "cager_v3_summary.csv"
)

summary_df.to_csv(
    summary_csv,
    index=False
)


# ==========================================================
# 18. OVERALL RESULTS
# ==========================================================

overall_adaptive_top1 = (
    summary_df["adaptive_peft_top1"]
    .dropna()
    .mean()
)

overall_cager_top1 = (
    summary_df["cager_v3_top1"]
    .dropna()
    .mean()
)

overall_adaptive_top5 = (
    summary_df["adaptive_peft_top5"]
    .dropna()
    .mean()
)

overall_cager_top5 = (
    summary_df["cager_v3_top5"]
    .dropna()
    .mean()
)

overall_adaptive_ppl = (
    summary_df["adaptive_peft_ppl"]
    .dropna()
    .mean()
)

overall_cager_ppl = (
    summary_df["cager_v3_ppl"]
    .dropna()
    .mean()
)

overall_top1_gain = (
    overall_cager_top1
    -
    overall_adaptive_top1
)

overall_top5_gain = (
    overall_cager_top5
    -
    overall_adaptive_top5
)

overall_ppl_change = (
    overall_adaptive_ppl
    -
    overall_cager_ppl
)

overall_own_expert = (
    summary_df["own_expert_selection"]
    .mean()
)

# Consolidated (token-weighted) overall metrics
total_tokens = sum(r.get("tokens", 0) for r in all_results)

overall_cager_correct_top1 = sum(round((r["top1_accuracy"] / 100.0) * r["tokens"]) for r in all_results)
overall_baseline_correct_top1 = sum(round((r["adaptive_peft_top1"] / 100.0) * r["tokens"]) for r in all_results if r.get("adaptive_peft_top1") is not None)

overall_cager_correct_top5 = sum(round((r["top5_accuracy"] / 100.0) * r["tokens"]) for r in all_results)
overall_baseline_correct_top5 = sum(round((r["adaptive_peft_top5"] / 100.0) * r["tokens"]) for r in all_results if r.get("adaptive_peft_top5") is not None)

overall_cager_total_loss = sum(r["loss"] * r["tokens"] for r in all_results)
overall_baseline_total_loss = sum(math.log(r["adaptive_peft_perplexity"]) * r["tokens"] for r in all_results if r.get("adaptive_peft_perplexity") is not None)

overall_cager_top1_weighted = (overall_cager_correct_top1 / max(total_tokens, 1)) * 100.0
overall_baseline_top1_weighted = (overall_baseline_correct_top1 / max(total_tokens, 1)) * 100.0
overall_top1_gain_weighted = overall_cager_top1_weighted - overall_baseline_top1_weighted

overall_cager_top5_weighted = (overall_cager_correct_top5 / max(total_tokens, 1)) * 100.0
overall_baseline_top5_weighted = (overall_baseline_correct_top5 / max(total_tokens, 1)) * 100.0
overall_top5_gain_weighted = overall_cager_top5_weighted - overall_baseline_top5_weighted

overall_cager_ppl_weighted = math.exp(min(overall_cager_total_loss / max(total_tokens, 1), 20))
overall_baseline_ppl_weighted = math.exp(min(overall_baseline_total_loss / max(total_tokens, 1), 20))
overall_ppl_change_weighted = overall_baseline_ppl_weighted - overall_cager_ppl_weighted

# ==========================================================
# 19. FINAL REPORT
# ==========================================================

print("\n")
print("=" * 85)
print("PERSONALIZED CAGER v3 — FINAL RESULTS")
print("=" * 85)

print("\n--- CLIENT-WISE AVERAGE METRICS ---")
print(f"Adaptive PEFT Avg Top-1 : {overall_adaptive_top1:.2f}%")
print(f"CAGER v3 Avg Top-1      : {overall_cager_top1:.2f}%")
print(f"Average Top-1 Gain      : {overall_top1_gain:+.2f} pp")
print()
print(f"Adaptive PEFT Avg Top-5 : {overall_adaptive_top5:.2f}%")
print(f"CAGER v3 Avg Top-5      : {overall_cager_top5:.2f}%")
print(f"Average Top-5 Gain      : {overall_top5_gain:+.2f} pp")
print()
print(f"Adaptive PEFT Avg PPL   : {overall_adaptive_ppl:.4f}")
print(f"CAGER v3 Avg PPL        : {overall_cager_ppl:.4f}")
print(f"Average PPL Change      : {overall_ppl_change:+.4f}")
print()
print(f"Average Own-Expert Selection Rate         : {overall_own_expert:.2f}%")

print("\n--- OVERALL CONSOLIDATED METRICS (TOKEN-WEIGHTED) ---")
print(f"Total Evaluated Tokens  : {total_tokens}")
print(f"Overall Baseline Top-1  : {overall_baseline_top1_weighted:.2f}%")
print(f"Overall CAGER v3 Top-1  : {overall_cager_top1_weighted:.2f}%")
print(f"Overall Top-1 Gain      : {overall_top1_gain_weighted:+.2f}%")
print()
print(f"Overall Baseline Top-5  : {overall_baseline_top5_weighted:.2f}%")
print(f"Overall CAGER v3 Top-5  : {overall_cager_top5_weighted:.2f}%")
print(f"Overall Top-5 Gain      : {overall_top5_gain_weighted:+.2f}%")
print()
print(f"Overall Baseline PPL    : {overall_baseline_ppl_weighted:.4f}")
print(f"Overall CAGER v3 PPL    : {overall_cager_ppl_weighted:.4f}")
print(f"Overall PPL Change      : {overall_ppl_change_weighted:+.4f}")
print()
print("=" * 85)


# ==========================================================
# 20. SAVE OVERALL JSON
# ==========================================================

overall_result = {
    "adaptive_peft_average_top1":
        float(overall_adaptive_top1),
    "cager_v3_average_top1":
        float(overall_cager_top1),
    "top1_gain_pp":
        float(overall_top1_gain),
    "adaptive_peft_average_top5":
        float(overall_adaptive_top5),
    "cager_v3_average_top5":
        float(overall_cager_top5),
    "top5_gain_pp":
        float(overall_top5_gain),
    "adaptive_peft_average_perplexity":
        float(overall_adaptive_ppl),
    "cager_v3_average_perplexity":
        float(overall_cager_ppl),
    "perplexity_change":
        float(overall_ppl_change),
    "average_own_expert_selection_rate":
        float(overall_own_expert),
    "total_tokens_evaluated":
        int(total_tokens),
    "overall_baseline_top1_accuracy":
        float(overall_baseline_top1_weighted),
    "overall_cager_v3_top1_accuracy":
        float(overall_cager_top1_weighted),
    "overall_top1_accuracy_gain":
        float(overall_top1_gain_weighted),
    "overall_baseline_top5_accuracy":
        float(overall_baseline_top5_weighted),
    "overall_cager_v3_top5_accuracy":
        float(overall_cager_top5_weighted),
    "overall_top5_accuracy_gain":
        float(overall_top5_gain_weighted),
    "overall_baseline_perplexity":
        float(overall_baseline_ppl_weighted),
    "overall_cager_v3_perplexity":
        float(overall_cager_ppl_weighted),
    "overall_perplexity_change":
        float(overall_ppl_change_weighted),
    "clients_evaluated":
        len(all_results)
}

overall_json = os.path.join(
    RESULT_PATH,
    "cager_v3_overall_results.json"
)

with open(overall_json, "w") as f:
    json.dump(
        overall_result,
        f,
        indent=4
    )

print(f"\nSummary CSV : {summary_csv}")
print(f"Overall JSON: {overall_json}")

print("\n")
print("=" * 85)
print("CAGER v3 EVALUATION COMPLETED")
print("=" * 85)


PERSONALIZED CAGER v3 EVALUATION
Number of Experts : 11
Top-K             : 3
Batch Size        : 8
Device            : cpu
Clients           : ['client_1', 'client_2', 'client_3', 'client_4', 'client_5', 'client_6', 'client_7', 'client_8', 'client_9', 'client_10', 'client_11']

Required files verified.

Loading CAGER v3 router...
CAGER v3 router loaded successfully.

Loading frozen DistilGPT2...


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 6339.46it/s]


Base model loaded and frozen.


Loading Frozen Adaptive PEFT Experts
Loading expert: client_2
Loading expert: client_3
Loading expert: client_4
Loading expert: client_5
Loading expert: client_6
Loading expert: client_7
Loading expert: client_8
Loading expert: client_9
Loading expert: client_10
Loading expert: client_11

All 11 Adaptive PEFT experts loaded.
Expert parameters are frozen.

Adaptive PEFT baseline loaded.


STARTING CAGER v3 EVALUATION


CAGER v3 Evaluation : client_1
  [Quick Test] Subsampled test dataset to 7 samples.
Test Samples : 7


CAGER v3 RESULTS : client_1
Loss              : 7.1108
Perplexity        : 1225.1146
Top-1 Accuracy    : 9.68%
Top-5 Accuracy    : 29.03%
Tokens Evaluated  : 62

Adaptive PEFT Top-1 : 9.68%
CAGER v3 Top-1      : 9.68%
Top-1 Gain          : +0.00 pp

Adaptive PEFT Top-5 : 29.03%
CAGER v3 Top-5      : 29.03%
Top-5 Gain          : +0.00 pp

Adaptive PEFT PPL   : 1224.4828
CAGER v3 PPL        : 1225.1146
PPL Change          : -0.6318

Own Expe

## Module 9: Comparative Analysis & Ablation Study
Compiles overall model performance, ablation tables, and comparisons.

In [18]:
# ==========================================================
# Module 9
# Comparative Analysis + Ablation Study
#
# Purpose:
#   Compare all major stages of the proposed system.
#
# Models:
#   1. Base DistilGPT2
#   2. Adaptive PEFT
#   3. CAGER v1
#   4. CAGER v2
#   5. Personalized CAGER v3
#
# Metrics:
#   - Top-1 Accuracy
#   - Top-5 Accuracy
#   - Perplexity
#   - Gain over Adaptive PEFT
#   - Per-client improvement
#   - Own-expert selection
#
# IMPORTANT:
#   This module performs analysis only.
#   No model is retrained.
# ==========================================================


import os
import json
import glob
import math
import sys

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

# Import local configuration
try:
    MODULE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    MODULE_DIR = os.path.abspath(os.path.join(os.getcwd(), 'code', 'modules'))
if MODULE_DIR not in sys.path:
    sys.path.append(MODULE_DIR)
import local_config

# ==========================================================
# 1. PROJECT PATH
# ==========================================================

PROJECT_PATH = local_config.WORKSPACE_DIR


# ==========================================================
# 2. RESULT DIRECTORIES
# ==========================================================

BASELINE_PATH = local_config.EVALUATION_PATH
CAGER_V1_PATH = os.path.join(PROJECT_PATH, "cager_evaluation")
CAGER_V2_PATH = os.path.join(PROJECT_PATH, "cager_v2_evaluation")
CAGER_V3_PATH = os.path.join(PROJECT_PATH, "cager_v3_evaluation")
OUTPUT_PATH = os.path.join(PROJECT_PATH, "module_9_analysis")

os.makedirs(
    OUTPUT_PATH,
    exist_ok=True
)


# ==========================================================
# 3. CLIENT LIST
# ==========================================================

CLIENTS = [
    f"client_{i}"
    for i in range(1, 12)
]


print("=" * 75)
print("MODULE 9")
print("COMPARATIVE ANALYSIS + ABLATION STUDY")
print("=" * 75)
print("Clients:", CLIENTS)
print("Output:", OUTPUT_PATH)
print("=" * 75)


# ==========================================================
# 4. UTILITY FUNCTIONS
# ==========================================================

def load_json_files(folder):
    """
    Loads every JSON file in a directory.
    """
    results = {}
    if not os.path.exists(folder):
        print(
            f"\nWarning: folder not found:\n{folder}"
        )
        return results

    files = glob.glob(
        os.path.join(
            folder,
            "*.json"
        )
    )

    for file_path in files:
        try:
            with open(
                file_path,
                "r"
            ) as f:
                data = json.load(f)

            filename = os.path.basename(
                file_path
            )

            results[filename] = data

        except Exception as e:
            print(
                f"Could not read {file_path}: {e}"
            )

    return results


# ==========================================================
# 5. FIND VALUE FROM MULTIPLE POSSIBLE KEYS
# ==========================================================

def get_value(
    data,
    possible_keys,
    default=np.nan
):
    """
    Looks for the first available key.

    This makes the analysis compatible with
    slightly different result JSON formats.
    """
    if not isinstance(
        data,
        dict
    ):
        return default

    for key in possible_keys:
        if key in data:
            value = data[key]

            if value is None:
                continue

            try:
                return float(value)
            except:
                return value

    return default


# ==========================================================
# 6. FIND CLIENT FROM RESULT
# ==========================================================

def find_client(
    data,
    filename
):
    if isinstance(
        data,
        dict
    ):
        if "client" in data:
            return str(
                data["client"]
            )

    for client in CLIENTS:
        if client in filename:
            return client

    return None


# ==========================================================
# 7. LOAD RESULTS
# ==========================================================

print("\nLoading saved results...")

baseline_files = load_json_files(
    BASELINE_PATH
)

v1_files = load_json_files(
    CAGER_V1_PATH
)

v2_files = load_json_files(
    CAGER_V2_PATH
)

v3_files = load_json_files(
    CAGER_V3_PATH
)


print(f"Adaptive PEFT files : {len(baseline_files)}")
print(f"CAGER v1 files      : {len(v1_files)}")
print(f"CAGER v2 files      : {len(v2_files)}")
print(f"CAGER v3 files      : {len(v3_files)}")


# ==========================================================
# 8. CONVERT FILE DICTIONARY TO CLIENT DICTIONARY
# ==========================================================

def organize_by_client(
    results
):
    organized = {}

    for filename, data in results.items():
        if isinstance(data, list):
            for item in data:
                if isinstance(item, dict) and "client" in item:
                    client = item["client"]
                    organized[client] = item
        else:
            client = find_client(
                data,
                filename
            )

            if client is not None:
                organized[client] = data

    return organized


baseline = organize_by_client(
    baseline_files
)

v1 = organize_by_client(
    v1_files
)

v2 = organize_by_client(
    v2_files
)

v3 = organize_by_client(
    v3_files
)


print("\nClients found:")
print("Adaptive PEFT:", sorted(baseline.keys()))
print("CAGER v1:", sorted(v1.keys()))
print("CAGER v2:", sorted(v2.keys()))
print("CAGER v3:", sorted(v3.keys()))


# ==========================================================
# 9. METRIC KEY DEFINITIONS
# ==========================================================

TOP1_KEYS = [
    "top1_accuracy",
    "personalized_top1_accuracy",
    "cager_top1",
    "cager_v2_top1",
    "cager_v3_top1",
    "top1"
]

TOP5_KEYS = [
    "top5_accuracy",
    "personalized_top5_accuracy",
    "cager_top5",
    "cager_v2_top5",
    "cager_v3_top5",
    "top5"
]

PPL_KEYS = [
    "perplexity",
    "personalized_perplexity",
    "cager_perplexity",
    "cager_v2_perplexity",
    "cager_v3_perplexity",
    "ppl"
]


# ==========================================================
# 10. EXTRACT METRICS
# ==========================================================

def extract_metrics(
    data
):
    if data is None:
        return {
            "top1": np.nan,
            "top5": np.nan,
            "ppl": np.nan
        }

    return {
        "top1":
            get_value(
                data,
                TOP1_KEYS
            ),
        "top5":
            get_value(
                data,
                TOP5_KEYS
            ),
        "ppl":
            get_value(
                data,
                PPL_KEYS
            )
    }


# ==========================================================
# 11. BUILD COMPARISON TABLE
# ==========================================================

rows = []

for client in CLIENTS:
    baseline_metrics = extract_metrics(
        baseline.get(client)
    )

    v1_metrics = extract_metrics(
        v1.get(client)
    )

    v2_metrics = extract_metrics(
        v2.get(client)
    )

    v3_metrics = extract_metrics(
        v3.get(client)
    )

    rows.append({
        "client":
            client,
        "adaptive_peft_top1":
            baseline_metrics["top1"],
        "cager_v1_top1":
            v1_metrics["top1"],
        "cager_v2_top1":
            v2_metrics["top1"],
        "cager_v3_top1":
            v3_metrics["top1"],
        "adaptive_peft_top5":
            baseline_metrics["top5"],
        "cager_v1_top5":
            v1_metrics["top5"],
        "cager_v2_top5":
            v2_metrics["top5"],
        "cager_v3_top5":
            v3_metrics["top5"],
        "adaptive_peft_ppl":
            baseline_metrics["ppl"],
        "cager_v1_ppl":
            v1_metrics["ppl"],
        "cager_v2_ppl":
            v2_metrics["ppl"],
        "cager_v3_ppl":
            v3_metrics["ppl"]
    })


comparison_df = pd.DataFrame(
    rows
)


# ==========================================================
# 12. PRINT COMPARISON
# ==========================================================

print("\n")
print("=" * 100)
print("PER-CLIENT TOP-1 COMPARISON")
print("=" * 100)
print(
    comparison_df[
        [
            "client",
            "adaptive_peft_top1",
            "cager_v1_top1",
            "cager_v2_top1",
            "cager_v3_top1"
        ]
    ].to_string(
        index=False
    )
)


# ==========================================================
# 13. AVERAGE RESULTS
# ==========================================================

average_results = {
    "Adaptive PEFT": {
        "Top-1":
            comparison_df[
                "adaptive_peft_top1"
            ].mean(),
        "Top-5":
            comparison_df[
                "adaptive_peft_top5"
            ].mean(),
        "Perplexity":
            comparison_df[
                "adaptive_peft_ppl"
            ].mean()
    },

    "CAGER v1": {
        "Top-1":
            comparison_df[
                "cager_v1_top1"
            ].mean(),
        "Top-5":
            comparison_df[
                "cager_v1_top5"
            ].mean(),
        "Perplexity":
            comparison_df[
                "cager_v1_ppl"
            ].mean()
    },

    "CAGER v2": {
        "Top-1":
            comparison_df[
                "cager_v2_top1"
            ].mean(),
        "Top-5":
            comparison_df[
                "cager_v2_top5"
            ].mean(),
        "Perplexity":
            comparison_df[
                "cager_v2_ppl"
            ].mean()
    },

    "Personalized CAGER v3": {
        "Top-1":
            comparison_df[
                "cager_v3_top1"
            ].mean(),
        "Top-5":
            comparison_df[
                "cager_v3_top5"
            ].mean(),
        "Perplexity":
            comparison_df[
                "cager_v3_ppl"
            ].mean()
    }
}


# ==========================================================
# 14. PRINT AVERAGES
# ==========================================================

print("\n")
print("=" * 90)
print("OVERALL COMPARISON")
print("=" * 90)
print(
    f"{'Model':<30}"
    f"{'Top-1':>15}"
    f"{'Top-5':>15}"
    f"{'PPL':>15}"
)
print("-" * 90)

for model, metrics in average_results.items():
    print(
        f"{model:<30}"
        f"{metrics['Top-1']:>14.2f}%"
        f"{metrics['Top-5']:>14.2f}%"
        f"{metrics['Perplexity']:>15.4f}"
    )

print("=" * 90)


# ==========================================================
# 15. GAIN OVER ADAPTIVE PEFT
# ==========================================================

adaptive_top1 = average_results["Adaptive PEFT"]["Top-1"]
adaptive_top5 = average_results["Adaptive PEFT"]["Top-5"]
adaptive_ppl = average_results["Adaptive PEFT"]["Perplexity"]


for model in [
    "CAGER v1",
    "CAGER v2",
    "Personalized CAGER v3"
]:
    average_results[model]["Top-1 Gain"] = (
        average_results[model]["Top-1"]
        -
        adaptive_top1
    )

    average_results[model]["Top-5 Gain"] = (
        average_results[model]["Top-5"]
        -
        adaptive_top5
    )

    # Positive means PPL improved
    average_results[model]["PPL Improvement"] = (
        adaptive_ppl
        -
        average_results[model]["Perplexity"]
    )


# ==========================================================
# 16. PRINT GAINS
# ==========================================================

print("\n")
print("=" * 90)
print("IMPROVEMENT OVER ADAPTIVE PEFT")
print("=" * 90)
print(
    f"{'Model':<30}"
    f"{'Top-1 Gain':>18}"
    f"{'Top-5 Gain':>18}"
    f"{'PPL Improvement':>20}"
)
print("-" * 90)

for model in [
    "CAGER v1",
    "CAGER v2",
    "Personalized CAGER v3"
]:
    print(
        f"{model:<30}"
        f"{average_results[model]['Top-1 Gain']:>17.2f} pp"
        f"{average_results[model]['Top-5 Gain']:>17.2f} pp"
        f"{average_results[model]['PPL Improvement']:>19.4f}"
    )

print("=" * 90)


# ==========================================================
# 17. PER-CLIENT CAGER v3 IMPROVEMENT
# ==========================================================

comparison_df["v3_top1_gain"] = (
    comparison_df["cager_v3_top1"]
    -
    comparison_df["adaptive_peft_top1"]
)

comparison_df["v3_top5_gain"] = (
    comparison_df["cager_v3_top5"]
    -
    comparison_df["adaptive_peft_top5"]
)

comparison_df["v3_ppl_improvement"] = (
    comparison_df["adaptive_peft_ppl"]
    -
    comparison_df["cager_v3_ppl"]
)


# ==========================================================
# 18. CLIENT-WISE IMPROVEMENT
# ==========================================================

print("\n")
print("=" * 90)
print("PERSONALIZED CAGER v3 — CLIENT-WISE IMPROVEMENT")
print("=" * 90)
print(
    comparison_df[
        [
            "client",
            "adaptive_peft_top1",
            "cager_v3_top1",
            "v3_top1_gain",
            "v3_top5_gain",
            "v3_ppl_improvement"
        ]
    ].to_string(
        index=False
    )
)
print("=" * 90)


# ==========================================================
# 19. COUNT IMPROVED CLIENTS
# ==========================================================

valid_gain = comparison_df["v3_top1_gain"].dropna()

improved_clients = (valid_gain > 0).sum()
decreased_clients = (valid_gain < 0).sum()
unchanged_clients = (valid_gain == 0).sum()


print("\n")
print("CAGER v3 Client Improvement Statistics")
print("-" * 60)
print("Improved Clients :", improved_clients)
print("Decreased Clients:", decreased_clients)
print("Unchanged Clients:", unchanged_clients)
print("-" * 60)


# ==========================================================
# 20. ABLATION TABLE
# ==========================================================

ablation_rows = []


# ----------------------------------------------------------
# Adaptive PEFT
# ----------------------------------------------------------

ablation_rows.append({
    "Experiment":
        "Adaptive PEFT",
    "Client Awareness":
        "Yes",
    "Cross Attention":
        "No",
    "Expert Compatibility":
        "No",
    "Own Expert Prior":
        "No",
    "Top-1":
        adaptive_top1,
    "Top-5":
        adaptive_top5,
    "Perplexity":
        adaptive_ppl
})


# ----------------------------------------------------------
# CAGER v1
# ----------------------------------------------------------

ablation_rows.append({
    "Experiment":
        "CAGER v1",
    "Client Awareness":
        "Partial",
    "Cross Attention":
        "Yes",
    "Expert Compatibility":
        "No",
    "Own Expert Prior":
        "No",
    "Top-1":
        average_results["CAGER v1"]["Top-1"],
    "Top-5":
        average_results["CAGER v1"]["Top-5"],
    "Perplexity":
        average_results["CAGER v1"]["Perplexity"]
})


# ----------------------------------------------------------
# CAGER v2
# ----------------------------------------------------------

ablation_rows.append({
    "Experiment":
        "CAGER v2",
    "Client Awareness":
        "Yes",
    "Cross Attention":
        "Yes",
    "Expert Compatibility":
        "No",
    "Own Expert Prior":
        "No",
    "Top-1":
        average_results["CAGER v2"]["Top-1"],
    "Top-5":
        average_results["CAGER v2"]["Top-5"],
    "Perplexity":
        average_results["CAGER v2"]["Perplexity"]
})


# ----------------------------------------------------------
# CAGER v3
# ----------------------------------------------------------

ablation_rows.append({
    "Experiment":
        "Personalized CAGER v3",
    "Client Awareness":
        "Yes",
    "Cross Attention":
        "Yes",
    "Expert Compatibility":
        "Yes",
    "Own Expert Prior":
        "Yes",
    "Top-1":
        average_results["Personalized CAGER v3"]["Top-1"],
    "Top-5":
        average_results["Personalized CAGER v3"]["Top-5"],
    "Perplexity":
        average_results["Personalized CAGER v3"]["Perplexity"]
})


ablation_df = pd.DataFrame(
    ablation_rows
)


# ==========================================================
# 21. PRINT ABLATION TABLE
# ==========================================================

print("\n")
print("=" * 110)
print("ABLATION STUDY")
print("=" * 110)
print(
    ablation_df.to_string(
        index=False
    )
)
print("=" * 110)


# ==========================================================
# 22. SAVE COMPARISON CSV
# ==========================================================

comparison_csv = os.path.join(
    OUTPUT_PATH,
    "model_comparison.csv"
)

comparison_df.to_csv(
    comparison_csv,
    index=False
)


# ==========================================================
# 23. SAVE ABLATION CSV
# ==========================================================

ablation_csv = os.path.join(
    OUTPUT_PATH,
    "ablation_study.csv"
)

ablation_df.to_csv(
    ablation_csv,
    index=False
)


# ==========================================================
# 24. SAVE OVERALL JSON
# ==========================================================

overall_json = os.path.join(
    OUTPUT_PATH,
    "overall_comparison.json"
)

clean_average_results = {}

for model, metrics in average_results.items():
    clean_average_results[model] = {}
    for key, value in metrics.items():
        if isinstance(
            value,
            (np.float32, np.float64)
        ):
            value = float(value)
        clean_average_results[model][key] = value


with open(
    overall_json,
    "w"
) as f:
    json.dump(
        clean_average_results,
        f,
        indent=4
    )


# ==========================================================
# 25. GRAPH 1
# TOP-1 ACCURACY
# ==========================================================

models = [
    "Adaptive PEFT",
    "CAGER v1",
    "CAGER v2",
    "Personalized CAGER v3"
]

top1_values = [
    average_results[model]["Top-1"]
    for model in models
]

plt.figure(
    figsize=(10, 6)
)
plt.bar(
    models,
    top1_values
)
plt.ylabel(
    "Top-1 Accuracy (%)"
)
plt.xlabel(
    "Model"
)
plt.title(
    "Top-1 Accuracy Comparison"
)
plt.xticks(
    rotation=20
)
plt.tight_layout()

top1_graph = os.path.join(
    OUTPUT_PATH,
    "top1_accuracy_comparison.png"
)

plt.savefig(
    top1_graph,
    dpi=300
)
plt.close()


# ==========================================================
# 26. GRAPH 2
# TOP-5 ACCURACY
# ==========================================================

top5_values = [
    average_results[model]["Top-5"]
    for model in models
]

plt.figure(
    figsize=(10, 6)
)
plt.bar(
    models,
    top5_values
)
plt.ylabel(
    "Top-5 Accuracy (%)"
)
plt.xlabel(
    "Model"
)
plt.title(
    "Top-5 Accuracy Comparison"
)
plt.xticks(
    rotation=20
)
plt.tight_layout()

top5_graph = os.path.join(
    OUTPUT_PATH,
    "top5_accuracy_comparison.png"
)

plt.savefig(
    top5_graph,
    dpi=300
)
plt.close()


# ==========================================================
# 27. GRAPH 3
# PERPLEXITY
# ==========================================================

ppl_values = [
    average_results[model]["Perplexity"]
    for model in models
]

plt.figure(
    figsize=(10, 6)
)
plt.bar(
    models,
    ppl_values
)
plt.ylabel(
    "Perplexity"
)
plt.xlabel(
    "Model"
)
plt.title(
    "Perplexity Comparison"
)
plt.xticks(
    rotation=20
)
plt.tight_layout()

ppl_graph = os.path.join(
    OUTPUT_PATH,
    "perplexity_comparison.png"
)

plt.savefig(
    ppl_graph,
    dpi=300
)
plt.close()


# ==========================================================
# 28. GRAPH 4
# PER-CLIENT ADAPTIVE PEFT VS CAGER v3
# ==========================================================

valid_df = comparison_df.dropna(
    subset=[
        "adaptive_peft_top1",
        "cager_v3_top1"
    ]
)

x = np.arange(
    len(valid_df)
)

width = 0.35

plt.figure(
    figsize=(12, 6)
)

plt.bar(
    x - width / 2,
    valid_df["adaptive_peft_top1"],
    width,
    label="Adaptive PEFT"
)

plt.bar(
    x + width / 2,
    valid_df["cager_v3_top1"],
    width,
    label="Personalized CAGER v3"
)

plt.xlabel(
    "Client"
)
plt.ylabel(
    "Top-1 Accuracy (%)"
)
plt.title(
    "Per-Client Adaptive PEFT vs Personalized CAGER v3"
)
plt.xticks(
    x,
    valid_df["client"],
    rotation=45
)
plt.legend()
plt.tight_layout()

client_graph = os.path.join(
    OUTPUT_PATH,
    "per_client_adaptive_vs_cager_v3.png"
)

plt.savefig(
    client_graph,
    dpi=300
)
plt.close()


# ==========================================================
# 29. GRAPH 5
# CAGER v3 IMPROVEMENT
# ==========================================================

valid_gain_df = comparison_df.dropna(
    subset=[
        "v3_top1_gain"
    ]
)

plt.figure(
    figsize=(12, 6)
)
plt.bar(
    valid_gain_df["client"],
    valid_gain_df["v3_top1_gain"]
)
plt.axhline(
    y=0,
    linestyle="--"
)
plt.xlabel(
    "Client"
)
plt.ylabel(
    "Top-1 Gain (percentage points)"
)
plt.title(
    "Personalized CAGER v3 Gain over Adaptive PEFT"
)
plt.xticks(
    rotation=45
)
plt.tight_layout()

gain_graph = os.path.join(
    OUTPUT_PATH,
    "cager_v3_top1_gain.png"
)

plt.savefig(
    gain_graph,
    dpi=300
)
plt.close()


# ==========================================================
# 30. FINAL REPORT
# ==========================================================

print("\n")
print("=" * 90)
print("MODULE 9 COMPLETED")
print("=" * 90)
print()
print("Comparison CSV :", comparison_csv)
print("Ablation CSV   :", ablation_csv)
print("Overall JSON   :", overall_json)
print()
print("Graphs saved in:")
print(OUTPUT_PATH)
print()
print("Improved clients :", improved_clients)
print("Decreased clients:", decreased_clients)
print()
print("=" * 90)


MODULE 9
COMPARATIVE ANALYSIS + ABLATION STUDY
Clients: ['client_1', 'client_2', 'client_3', 'client_4', 'client_5', 'client_6', 'client_7', 'client_8', 'client_9', 'client_10', 'client_11']
Output: d:/project\module_9_analysis

Loading saved results...
Adaptive PEFT files : 2
CAGER v1 files      : 11
CAGER v2 files      : 12
CAGER v3 files      : 12

Clients found:
Adaptive PEFT: ['client_1', 'client_10', 'client_11', 'client_2', 'client_3', 'client_4', 'client_5', 'client_6', 'client_7', 'client_8', 'client_9']
CAGER v1: ['client_1', 'client_10', 'client_11', 'client_2', 'client_3', 'client_4', 'client_5', 'client_6', 'client_7', 'client_8', 'client_9']
CAGER v2: ['client_1', 'client_10', 'client_11', 'client_2', 'client_3', 'client_4', 'client_5', 'client_6', 'client_7', 'client_8', 'client_9']
CAGER v3: ['client_1', 'client_10', 'client_11', 'client_2', 'client_3', 'client_4', 'client_5', 'client_6', 'client_7', 'client_8', 'client_9']


PER-CLIENT TOP-1 COMPARISON
   client  adapt

## Module 10: Final Results, Visualization & Review-2 Report
Generates final research conclusions, JSON summaries, and DPI-300 publication graphs.

In [19]:
# ==========================================================
# MODULE 10
# FINAL RESULTS, VISUALIZATION & REVIEW-2 REPORT
#
# Uses:
#   Module 9 outputs
#
# Generates:
#   1. Final model comparison
#   2. Ablation analysis
#   3. Client-wise analysis
#   4. Best / worst clients
#   5. Improvement statistics
#   6. Review-2 tables
#   7. Research conclusion
#   8. Final graphs
#
# NO MODEL TRAINING
# ==========================================================

import os
import json
import math
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import local configuration
try:
    MODULE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    MODULE_DIR = os.path.abspath(os.path.join(os.getcwd(), 'code', 'modules'))
if MODULE_DIR not in sys.path:
    sys.path.append(MODULE_DIR)
import local_config

# ==========================================================
# 1. PATH CONFIGURATION
# ==========================================================

PROJECT_PATH = local_config.WORKSPACE_DIR
MODULE9_PATH = os.path.join(PROJECT_PATH, "module_9_analysis")
MODULE10_PATH = os.path.join(PROJECT_PATH, "module_10_final_results")

os.makedirs(
    MODULE10_PATH,
    exist_ok=True
)


# ==========================================================
# 2. INPUT FILES
# ==========================================================

COMPARISON_FILE = os.path.join(
    MODULE9_PATH,
    "model_comparison.csv"
)

ABLATION_FILE = os.path.join(
    MODULE9_PATH,
    "ablation_study.csv"
)

OVERALL_FILE = os.path.join(
    MODULE9_PATH,
    "overall_comparison.json"
)


# ==========================================================
# 3. VERIFY FILES
# ==========================================================

print("=" * 80)
print("MODULE 10")
print("FINAL RESULTS & REVIEW 2 ANALYSIS")
print("=" * 80)

required_files = [
    COMPARISON_FILE,
    ABLATION_FILE,
    OVERALL_FILE
]

for file_path in required_files:
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"\nRequired file not found:\n{file_path}\n\nRun Module 9 first."
        )
    print("Found:", file_path)

print("=" * 80)


# ==========================================================
# 4. LOAD DATA
# ==========================================================

comparison_df = pd.read_csv(
    COMPARISON_FILE
)

ablation_df = pd.read_csv(
    ABLATION_FILE
)

with open(
    OVERALL_FILE,
    "r"
) as f:
    overall_results = json.load(f)

print("\nData loaded successfully.")


# ==========================================================
# 5. DISPLAY RAW COMPARISON
# ==========================================================

print("\n")
print("=" * 100)
print("FINAL MODEL COMPARISON")
print("=" * 100)
print(
    comparison_df.to_string(
        index=False
    )
)


# ==========================================================
# 6. OVERALL MODEL SUMMARY
# ==========================================================

models = [
    "Adaptive PEFT",
    "CAGER v1",
    "CAGER v2",
    "Personalized CAGER v3"
]

summary_rows = []

for model in models:
    model_result = overall_results.get(
        model,
        {}
    )
    summary_rows.append({
        "Model":
            model,
        "Top-1 Accuracy (%)":
            model_result.get(
                "Top-1",
                np.nan
            ),
        "Top-5 Accuracy (%)":
            model_result.get(
                "Top-5",
                np.nan
            ),
        "Perplexity":
            model_result.get(
                "Perplexity",
                np.nan
            ),
        "Top-1 Gain (pp)":
            model_result.get(
                "Top-1 Gain",
                0
            ),
        "Top-5 Gain (pp)":
            model_result.get(
                "Top-5 Gain",
                0
            ),
        "PPL Improvement":
            model_result.get(
                "PPL Improvement",
                0
            )
    })

final_summary_df = pd.DataFrame(
    summary_rows
)


print("\n")
print("=" * 100)
print("OVERALL MODEL PERFORMANCE")
print("=" * 100)
print(
    final_summary_df.to_string(
        index=False
    )
)


# ==========================================================
# 7. FIND BEST MODEL
# ==========================================================

valid_top1 = final_summary_df.dropna(
    subset=[
        "Top-1 Accuracy (%)"
    ]
)

if len(valid_top1) > 0:
    best_top1_row = valid_top1.loc[
        valid_top1["Top-1 Accuracy (%)"].idxmax()
    ]
    best_top1_model = (
        best_top1_row["Model"]
    )
    best_top1_value = (
        best_top1_row["Top-1 Accuracy (%)"]
    )
else:
    best_top1_model = "N/A"
    best_top1_value = np.nan


# ==========================================================
# 8. BEST TOP-5 MODEL
# ==========================================================

valid_top5 = final_summary_df.dropna(
    subset=[
        "Top-5 Accuracy (%)"
    ]
)

if len(valid_top5) > 0:
    best_top5_row = valid_top5.loc[
        valid_top5["Top-5 Accuracy (%)"].idxmax()
    ]
    best_top5_model = (
        best_top5_row["Model"]
    )
    best_top5_value = (
        best_top5_row["Top-5 Accuracy (%)"]
    )
else:
    best_top5_model = "N/A"
    best_top5_value = np.nan


# ==========================================================
# 9. BEST PERPLEXITY
# ==========================================================

valid_ppl = final_summary_df.dropna(
    subset=[
        "Perplexity"
    ]
)

if len(valid_ppl) > 0:
    best_ppl_row = valid_ppl.loc[
        valid_ppl["Perplexity"].idxmin()
    ]
    best_ppl_model = (
        best_ppl_row["Model"]
    )
    best_ppl_value = (
        best_ppl_row["Perplexity"]
    )
else:
    best_ppl_model = "N/A"
    best_ppl_value = np.nan


# ==========================================================
# 10. ADAPTIVE PEFT BASELINE
# ==========================================================

adaptive_row = final_summary_df[
    final_summary_df["Model"]
    == "Adaptive PEFT"
]

if len(adaptive_row) > 0:
    adaptive_top1 = float(
        adaptive_row["Top-1 Accuracy (%)"].iloc[0]
    )
    adaptive_top5 = float(
        adaptive_row["Top-5 Accuracy (%)"].iloc[0]
    )
    adaptive_ppl = float(
        adaptive_row["Perplexity"].iloc[0]
    )
else:
    adaptive_top1 = np.nan
    adaptive_top5 = np.nan
    adaptive_ppl = np.nan


# ==========================================================
# 11. CAGER v3 RESULT
# ==========================================================

v3_row = final_summary_df[
    final_summary_df["Model"]
    == "Personalized CAGER v3"
]

if len(v3_row) > 0:
    v3_top1 = float(
        v3_row["Top-1 Accuracy (%)"].iloc[0]
    )
    v3_top5 = float(
        v3_row["Top-5 Accuracy (%)"].iloc[0]
    )
    v3_ppl = float(
        v3_row["Perplexity"].iloc[0]
    )
else:
    v3_top1 = np.nan
    v3_top5 = np.nan
    v3_ppl = np.nan


# ==========================================================
# 12. FINAL CAGER v3 GAINS
# ==========================================================

if not np.isnan(adaptive_top1) and not np.isnan(v3_top1):
    final_top1_gain = (
        v3_top1
        -
        adaptive_top1
    )
else:
    final_top1_gain = np.nan


if not np.isnan(adaptive_top5) and not np.isnan(v3_top5):
    final_top5_gain = (
        v3_top5
        -
        adaptive_top5
    )
else:
    final_top5_gain = np.nan


if not np.isnan(adaptive_ppl) and not np.isnan(v3_ppl):
    final_ppl_improvement = (
        adaptive_ppl
        -
        v3_ppl
    )
else:
    final_ppl_improvement = np.nan


# ==========================================================
# 13. PERCENTAGE IMPROVEMENT
# ==========================================================

if (
    not np.isnan(adaptive_top1)
    and
    adaptive_top1 != 0
    and
    not np.isnan(v3_top1)
):
    top1_percentage_improvement = (
        (
            v3_top1
            -
            adaptive_top1
        )
        /
        adaptive_top1
    ) * 100
else:
    top1_percentage_improvement = np.nan


if (
    not np.isnan(adaptive_top5)
    and
    adaptive_top5 != 0
    and
    not np.isnan(v3_top5)
):
    top5_percentage_improvement = (
        (
            v3_top5
            -
            adaptive_top5
        )
        /
        adaptive_top5
    ) * 100
else:
    top5_percentage_improvement = np.nan


# ==========================================================
# 14. CLIENT-WISE CAGER v3 ANALYSIS
# ==========================================================

client_analysis = comparison_df.copy()

if (
    "cager_v3_top1" in client_analysis.columns
    and
    "adaptive_peft_top1" in client_analysis.columns
):
    client_analysis["Top1_Gain_pp"] = (
        client_analysis["cager_v3_top1"]
        -
        client_analysis["adaptive_peft_top1"]
    )


if (
    "cager_v3_top5" in client_analysis.columns
    and
    "adaptive_peft_top5" in client_analysis.columns
):
    client_analysis["Top5_Gain_pp"] = (
        client_analysis["cager_v3_top5"]
        -
        client_analysis["adaptive_peft_top5"]
    )


if (
    "cager_v3_ppl" in client_analysis.columns
    and
    "adaptive_peft_ppl" in client_analysis.columns
):
    client_analysis["PPL_Improvement"] = (
        client_analysis["adaptive_peft_ppl"]
        -
        client_analysis["cager_v3_ppl"]
    )


# ==========================================================
# 15. CLIENT IMPROVEMENT COUNTS
# ==========================================================

valid_clients = client_analysis.dropna(
    subset=[
        "Top1_Gain_pp"
    ]
)

improved_clients = valid_clients[
    valid_clients["Top1_Gain_pp"] > 0
]

decreased_clients = valid_clients[
    valid_clients["Top1_Gain_pp"] < 0
]

unchanged_clients = valid_clients[
    valid_clients["Top1_Gain_pp"] == 0
]

number_improved = len(improved_clients)
number_decreased = len(decreased_clients)
number_unchanged = len(unchanged_clients)


# ==========================================================
# 16. BEST CLIENT
# ==========================================================

if len(valid_clients) > 0:
    best_client_row = valid_clients.loc[
        valid_clients["Top1_Gain_pp"].idxmax()
    ]
    best_client = (
        best_client_row["client"]
    )
    best_client_gain = (
        best_client_row["Top1_Gain_pp"]
    )
else:
    best_client = "N/A"
    best_client_gain = np.nan


# ==========================================================
# 17. WORST CLIENT
# ==========================================================

if len(valid_clients) > 0:
    worst_client_row = valid_clients.loc[
        valid_clients["Top1_Gain_pp"].idxmin()
    ]
    worst_client = (
        worst_client_row["client"]
    )
    worst_client_gain = (
        worst_client_row["Top1_Gain_pp"]
    )
else:
    worst_client = "N/A"
    worst_client_gain = np.nan


# ==========================================================
# 18. PRINT CLIENT ANALYSIS
# ==========================================================

print("\n")
print("=" * 100)
print("PERSONALIZED CAGER v3 — CLIENT-WISE ANALYSIS")
print("=" * 100)

display_columns = [
    "client",
    "adaptive_peft_top1",
    "cager_v3_top1",
    "Top1_Gain_pp",
    "adaptive_peft_top5",
    "cager_v3_top5",
    "Top5_Gain_pp",
    "adaptive_peft_ppl",
    "cager_v3_ppl",
    "PPL_Improvement"
]

display_columns = [
    col
    for col in display_columns
    if col in client_analysis.columns
]

print(
    client_analysis[
        display_columns
    ].to_string(
        index=False
    )
)


# ==========================================================
# 19. ABLATION STUDY
# ==========================================================

print("\n")
print("=" * 110)
print("ABLATION STUDY")
print("=" * 110)
print(
    ablation_df.to_string(
        index=False
    )
)


# ==========================================================
# 20. ABLATION INTERPRETATION
# ==========================================================

ablation_text = []
ablation_text.append(
    "The ablation study evaluates the progressive development of the personalized federated next-word prediction framework."
)
ablation_text.append(
    "Adaptive PEFT establishes the personalized parameter-efficient baseline."
)
ablation_text.append(
    "CAGER v1 introduces cross-attention-based expert routing."
)
ablation_text.append(
    "CAGER v2 introduces client-aware routing using calibration-based expert performance."
)
ablation_text.append(
    "Personalized CAGER v3 further introduces expert-client compatibility and an own-expert personalization prior."
)


# ==========================================================
# 21. RESEARCH CONCLUSION
# ==========================================================

if np.isnan(final_top1_gain):
    accuracy_conclusion = (
        "CAGER v3 accuracy could not be compared because the required results were unavailable."
    )
elif final_top1_gain > 0:
    accuracy_conclusion = (
        "Personalized CAGER v3 improves average Top-1 next-word prediction accuracy over the Adaptive PEFT baseline."
    )
elif final_top1_gain < 0:
    accuracy_conclusion = (
        "Personalized CAGER v3 does not improve average Top-1 accuracy over the Adaptive PEFT baseline in the current experiment. The result indicates that the expert routing strategy requires further optimization."
    )
else:
    accuracy_conclusion = (
        "Personalized CAGER v3 produces the same average Top-1 accuracy as Adaptive PEFT."
    )


if np.isnan(final_ppl_improvement):
    ppl_conclusion = (
        "Perplexity comparison is unavailable."
    )
elif final_ppl_improvement > 0:
    ppl_conclusion = (
        "CAGER v3 reduces perplexity compared with Adaptive PEFT, indicating improved language model confidence."
    )
elif final_ppl_improvement < 0:
    ppl_conclusion = (
        "CAGER v3 increases perplexity compared with Adaptive PEFT, indicating that the current routing mechanism does not yet improve overall language-model confidence."
    )
else:
    ppl_conclusion = (
        "CAGER v3 produces the same perplexity as Adaptive PEFT."
    )


# ==========================================================
# 22. PERSONALIZATION CONCLUSION
# ==========================================================

if number_improved > number_decreased:
    personalization_conclusion = (
        f"CAGER v3 improves Top-1 accuracy for {number_improved} of {len(valid_clients)} evaluated clients, indicating that client-aware expert routing provides personalization benefits for a majority of evaluated clients."
    )
elif number_improved < number_decreased:
    personalization_conclusion = (
        f"CAGER v3 improves Top-1 accuracy for {number_improved} of {len(valid_clients)} clients while performance decreases for {number_decreased} clients. This indicates that personalization is not yet consistently effective across all client distributions."
    )
else:
    personalization_conclusion = (
        "CAGER v3 produces a mixed personalization effect across the evaluated clients."
    )


# ==========================================================
# 23. PRINT RESEARCH CONCLUSION
# ==========================================================

print("\n")
print("=" * 100)
print("RESEARCH CONCLUSION")
print("=" * 100)
print()
print("Accuracy:")
print(accuracy_conclusion)
print()
print("Perplexity:")
print(ppl_conclusion)
print()
print("Personalization:")
print(personalization_conclusion)
print()
if not np.isnan(best_client_gain):
    print(f"Best client : {best_client} ({best_client_gain:+.2f} pp)")
else:
    print(f"Best client : {best_client} (N/A)")
if not np.isnan(worst_client_gain):
    print(f"Worst client: {worst_client} ({worst_client_gain:+.2f} pp)")
else:
    print(f"Worst client: {worst_client} (N/A)")
print("=" * 100)


# ==========================================================
# 24. GRAPH 1 — TOP-1
# ==========================================================

plt.figure(
    figsize=(11, 6)
)
plt.bar(
    final_summary_df["Model"],
    final_summary_df["Top-1 Accuracy (%)"]
)
plt.ylabel(
    "Top-1 Accuracy (%)"
)
plt.xlabel(
    "Model"
)
plt.title(
    "Next-Word Prediction — Top-1 Accuracy"
)
plt.xticks(
    rotation=20
)
plt.tight_layout()

graph1 = os.path.join(
    MODULE10_PATH,
    "01_top1_accuracy.png"
)

plt.savefig(
    graph1,
    dpi=300
)
plt.close()


# ==========================================================
# 25. GRAPH 2 — TOP-5
# ==========================================================

plt.figure(
    figsize=(11, 6)
)
plt.bar(
    final_summary_df["Model"],
    final_summary_df["Top-5 Accuracy (%)"]
)
plt.ylabel(
    "Top-5 Accuracy (%)"
)
plt.xlabel(
    "Model"
)
plt.title(
    "Next-Word Prediction — Top-5 Accuracy"
)
plt.xticks(
    rotation=20
)
plt.tight_layout()

graph2 = os.path.join(
    MODULE10_PATH,
    "02_top5_accuracy.png"
)

plt.savefig(
    graph2,
    dpi=300
)
plt.close()


# ==========================================================
# 26. GRAPH 3 — PERPLEXITY
# ==========================================================

plt.figure(
    figsize=(11, 6)
)
plt.bar(
    final_summary_df["Model"],
    final_summary_df["Perplexity"]
)
plt.ylabel(
    "Perplexity"
)
plt.xlabel(
    "Model"
)
plt.title(
    "Language Modeling Perplexity Comparison"
)
plt.xticks(
    rotation=20
)
plt.tight_layout()

graph3 = os.path.join(
    MODULE10_PATH,
    "03_perplexity.png"
)

plt.savefig(
    graph3,
    dpi=300
)
plt.close()


# ==========================================================
# 27. GRAPH 4 — CLIENT-WISE TOP-1
# ==========================================================

if len(valid_clients) > 0 and "adaptive_peft_top1" in valid_clients.columns and "cager_v3_top1" in valid_clients.columns:
    x = np.arange(
        len(valid_clients)
    )
    width = 0.35

    plt.figure(
        figsize=(13, 7)
    )

    plt.bar(
        x - width / 2,
        valid_clients["adaptive_peft_top1"],
        width,
        label="Adaptive PEFT"
    )

    plt.bar(
        x + width / 2,
        valid_clients["cager_v3_top1"],
        width,
        label="Personalized CAGER v3"
    )

    plt.xlabel(
        "Client"
    )
    plt.ylabel(
        "Top-1 Accuracy (%)"
    )
    plt.title(
        "Per-Client Personalization Performance"
    )
    plt.xticks(
        x,
        valid_clients["client"],
        rotation=45
    )
    plt.legend()
    plt.tight_layout()

    graph4 = os.path.join(
        MODULE10_PATH,
        "04_clientwise_top1.png"
    )

    plt.savefig(
        graph4,
        dpi=300
    )
    plt.close()


# ==========================================================
# 28. GRAPH 5 — CAGER v3 GAIN
# ==========================================================

if len(valid_clients) > 0 and "Top1_Gain_pp" in valid_clients.columns:
    plt.figure(
        figsize=(13, 7)
    )

    plt.bar(
        valid_clients["client"],
        valid_clients["Top1_Gain_pp"]
    )

    plt.axhline(
        y=0,
        linestyle="--"
    )

    plt.xlabel(
        "Client"
    )
    plt.ylabel(
        "Top-1 Gain (percentage points)"
    )
    plt.title(
        "Personalized CAGER v3 Gain over Adaptive PEFT"
    )
    plt.xticks(
        rotation=45
    )
    plt.tight_layout()

    graph5 = os.path.join(
        MODULE10_PATH,
        "05_cager_v3_gain.png"
    )

    plt.savefig(
        graph5,
        dpi=300
    )
    plt.close()


# ==========================================================
# 29. GRAPH 6 — TOP-1 GAIN ACROSS ARCHITECTURES
# ==========================================================

gain_models = [
    "CAGER v1",
    "CAGER v2",
    "Personalized CAGER v3"
]

gain_values = [
    overall_results.get(
        model,
        {}
    ).get(
        "Top-1 Gain",
        np.nan
    )
    for model in gain_models
]

plt.figure(
    figsize=(10, 6)
)
plt.bar(
    gain_models,
    gain_values
)
plt.axhline(
    y=0,
    linestyle="--"
)
plt.ylabel(
    "Top-1 Gain over Adaptive PEFT (pp)"
)
plt.xlabel(
    "Architecture"
)
plt.title(
    "Evolution of Personalized Expert Routing"
)
plt.xticks(
    rotation=20
)
plt.tight_layout()

graph6 = os.path.join(
    MODULE10_PATH,
    "06_architecture_gain.png"
)

plt.savefig(
    graph6,
    dpi=300
)
plt.close()


# ==========================================================
# 30. SAVE FINAL CLIENT TABLE
# ==========================================================

client_csv = os.path.join(
    MODULE10_PATH,
    "final_client_analysis.csv"
)

client_analysis.to_csv(
    client_csv,
    index=False
)


# ==========================================================
# 31. SAVE FINAL MODEL TABLE
# ==========================================================

model_csv = os.path.join(
    MODULE10_PATH,
    "final_model_comparison.csv"
)

final_summary_df.to_csv(
    model_csv,
    index=False
)


# ==========================================================
# 32. SAVE ABLATION TABLE
# ==========================================================

ablation_output = os.path.join(
    MODULE10_PATH,
    "final_ablation_table.csv"
)

ablation_df.to_csv(
    ablation_output,
    index=False
)


# ==========================================================
# 33. REVIEW 2 SUMMARY
# ==========================================================

review2_summary = {
    "project_title":
        "Adaptive Personalized Federated Learning Framework for Privacy-Preserving Next-Word Prediction",
    "baseline":
        "Adaptive PEFT",
    "final_model":
        "Personalized CAGER v3",
    "adaptive_peft_top1":
        adaptive_top1,
    "cager_v3_top1":
        v3_top1,
    "top1_gain_pp":
        final_top1_gain,
    "top1_percentage_improvement":
        top1_percentage_improvement,
    "adaptive_peft_top5":
        adaptive_top5,
    "cager_v3_top5":
        v3_top5,
    "top5_gain_pp":
        final_top5_gain,
    "top5_percentage_improvement":
        top5_percentage_improvement,
    "adaptive_peft_perplexity":
        adaptive_ppl,
    "cager_v3_perplexity":
        v3_ppl,
    "perplexity_improvement":
        final_ppl_improvement,
    "improved_clients":
        int(number_improved),
    "decreased_clients":
        int(number_decreased),
    "unchanged_clients":
        int(number_unchanged),
    "best_client":
        best_client,
    "best_client_gain_pp":
        float(best_client_gain)
        if not np.isnan(best_client_gain)
        else None,
    "worst_client":
        worst_client,
    "worst_client_gain_pp":
        float(worst_client_gain)
        if not np.isnan(worst_client_gain)
        else None,
    "best_top1_model":
        best_top1_model,
    "best_top1_accuracy":
        float(best_top1_value)
        if not np.isnan(best_top1_value)
        else None,
    "best_top5_model":
        best_top5_model,
    "best_top5_accuracy":
        float(best_top5_value)
        if not np.isnan(best_top5_value)
        else None,
    "best_perplexity_model":
        best_ppl_model,
    "best_perplexity":
        float(best_ppl_value)
        if not np.isnan(best_ppl_value)
        else None
}

review2_json = os.path.join(
    MODULE10_PATH,
    "review2_summary.json"
)

with open(
    review2_json,
    "w"
) as f:
    json.dump(
        review2_summary,
        f,
        indent=4
    )


# ==========================================================
# 34. SAVE TEXT CONCLUSION
# ==========================================================

conclusion_file = os.path.join(
    MODULE10_PATH,
    "research_conclusion.txt"
)

with open(
    conclusion_file,
    "w"
) as f:
    f.write("RESEARCH CONCLUSION\n")
    f.write("===================\n\n")
    f.write("Project: Adaptive Personalized Federated Learning Framework for Privacy-Preserving Next-Word Prediction\n\n")
    f.write("Accuracy:\n")
    f.write(accuracy_conclusion + "\n\n")
    f.write("Perplexity:\n")
    f.write(ppl_conclusion + "\n\n")
    f.write("Personalization:\n")
    f.write(personalization_conclusion + "\n\n")
    if not np.isnan(best_client_gain):
        f.write(f"Best client: {best_client} ({best_client_gain:+.2f} pp)\n")
    else:
        f.write(f"Best client: {best_client} (N/A)\n")
    if not np.isnan(worst_client_gain):
        f.write(f"Worst client: {worst_client} ({worst_client_gain:+.2f} pp)\n")
    else:
        f.write(f"Worst client: {worst_client} (N/A)\n")


# ==========================================================
# 35. FINAL OUTPUT
# ==========================================================

print("\n")
print("=" * 100)
print("MODULE 10 COMPLETED")
print("=" * 100)
print()
print("Final Model Comparison:", model_csv)
print("Final Client Analysis :", client_csv)
print("Final Ablation Table  :", ablation_output)
print("Review 2 Summary      :", review2_json)
print("Research Conclusion   :", conclusion_file)
print()
print("Graphs saved in:")
print(MODULE10_PATH)
print()
print("=" * 100)
print("FINAL CAGER v3 RESULT")
print("=" * 100)
print(f"Adaptive PEFT Top-1 : {adaptive_top1:.2f}%" if not np.isnan(adaptive_top1) else "Adaptive PEFT Top-1 : N/A")
print(f"CAGER v3 Top-1      : {v3_top1:.2f}%" if not np.isnan(v3_top1) else "CAGER v3 Top-1      : N/A")
if not np.isnan(final_top1_gain):
    print(f"Top-1 Gain          : {final_top1_gain:+.2f} pp")
else:
    print("Top-1 Gain          : N/A")
print()
print(f"Adaptive PEFT Top-5 : {adaptive_top5:.2f}%" if not np.isnan(adaptive_top5) else "Adaptive PEFT Top-5 : N/A")
print(f"CAGER v3 Top-5      : {v3_top5:.2f}%" if not np.isnan(v3_top5) else "CAGER v3 Top-5      : N/A")
if not np.isnan(final_top5_gain):
    print(f"Top-5 Gain          : {final_top5_gain:+.2f} pp")
else:
    print("Top-5 Gain          : N/A")
print()
print(f"Adaptive PEFT PPL   : {adaptive_ppl:.4f}" if not np.isnan(adaptive_ppl) else "Adaptive PEFT PPL   : N/A")
print(f"CAGER v3 PPL        : {v3_ppl:.4f}" if not np.isnan(v3_ppl) else "CAGER v3 PPL        : N/A")
if not np.isnan(final_ppl_improvement):
    print(f"PPL Improvement     : {final_ppl_improvement:+.4f}")
else:
    print("PPL Improvement     : N/A")
print()
print(f"Improved Clients    : {number_improved}")
print(f"Decreased Clients   : {number_decreased}")
print(f"Unchanged Clients   : {number_unchanged}")
print()
print(f"Best Client         : {best_client}")
if not np.isnan(best_client_gain):
    print(f"Best Client Gain    : {best_client_gain:+.2f} pp")
else:
    print("Best Client Gain    : N/A")
print()
print("=" * 100)


MODULE 10
FINAL RESULTS & REVIEW 2 ANALYSIS
Found: d:/project\module_9_analysis\model_comparison.csv
Found: d:/project\module_9_analysis\ablation_study.csv
Found: d:/project\module_9_analysis\overall_comparison.json

Data loaded successfully.


FINAL MODEL COMPARISON
   client  adaptive_peft_top1  cager_v1_top1  cager_v2_top1  cager_v3_top1  adaptive_peft_top5  cager_v1_top5  cager_v2_top5  cager_v3_top5  adaptive_peft_ppl  cager_v1_ppl  cager_v2_ppl  cager_v3_ppl  v3_top1_gain  v3_top5_gain  v3_ppl_improvement
 client_1            9.677419       9.677419       0.000808       9.677419           29.032258      29.032258       0.002694      29.032258        1224.482810   1225.567846      1.001188   1225.114561      0.000000      0.000000           -0.631750
 client_2           10.769231      10.769231       0.001309      10.769231           30.769231      30.769231       0.004582      30.769231         662.760197    698.932290      1.001383    662.854341      0.000000      0.000000      

## Module 11: Interactive Next-Word Prediction Demo
Allows interactive typing and showcases Base Model, Single LoRA Expert, or CAGER v3 Router prediction.

In [20]:
# ==========================================================
# MODULE 11: MULTILINGUAL NEXT-WORD PREDICTION & MENU DEMO
#
# Supports:
#   - Multilingual & CodeMix datasets (English, Tamil, etc.)
#   - Interactive menu-driven console & notebook execution
#   - UTF-8 byte-boundary decoding for non-ASCII/Tamil tokens
#   - Personalized CAGER v3 Top-K Mixture of Experts routing
# ==========================================================

import os
import sys
import json
import argparse
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# Reconfigure stdout/stderr for UTF-8 output on Windows terminal
if sys.stdout.encoding != 'utf-8':
    try:
        sys.stdout.reconfigure(encoding='utf-8')
        sys.stderr.reconfigure(encoding='utf-8')
    except Exception:
        pass

# Import local configuration & CAGER v3 architecture
try:
    MODULE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    MODULE_DIR = os.path.abspath(os.path.join(os.getcwd(), 'code', 'modules'))
if MODULE_DIR not in sys.path:
    sys.path.append(MODULE_DIR)
import local_config
from module_8d_personalized_cager_v3 import PersonalizedCAGERv3, get_client_features

# Setup Paths & Device
PROJECT_PATH = local_config.WORKSPACE_DIR
CLIENT_MODEL_PATH = os.path.join(PROJECT_PATH, "client_models")
ROUTER_FILE = os.path.join(PROJECT_PATH, "cager_v3_models", "personalized_cager_v3_router.pt")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLIENTS = [f"client_{i}" for i in range(1, 12)]

# Load Tokenizer & Base Model
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("[+] Loading Base Model (DistilGPT2)...")
base_model = AutoModelForCausalLM.from_pretrained("distilgpt2").to(DEVICE)
base_model.eval()

# Load all client LoRA experts into PeftModel
print("[+] Loading Adaptive PEFT Expert Adapters...")
first_client_path = os.path.join(CLIENT_MODEL_PATH, "client_1")

if not os.path.exists(first_client_path):
    print("[-] Error: Client models not found in workspace. Please train Module 7 first.")
    sys.exit(1)

expert_model = PeftModel.from_pretrained(
    base_model,
    first_client_path,
    adapter_name="client_1"
).to(DEVICE)

for i in range(2, 12):
    client_name = f"client_{i}"
    client_path = os.path.join(CLIENT_MODEL_PATH, client_name)
    if os.path.exists(client_path):
        expert_model.load_adapter(client_path, adapter_name=client_name)

expert_model.eval()

# Load CAGER v3 Router
print("[+] Loading Personalized CAGER v3 Router Checkpoint...")
router = PersonalizedCAGERv3().to(DEVICE)
router_available = False

if os.path.exists(ROUTER_FILE):
    checkpoint = torch.load(ROUTER_FILE, map_location=DEVICE)
    if isinstance(checkpoint, dict) and "router" in checkpoint:
        router.load_state_dict(checkpoint["router"])
    else:
        router.load_state_dict(checkpoint)
    router_available = True
    print("[+] CAGER v3 Router loaded successfully.")
else:
    print("[!] Warning: Router file not found. CAGER v3 fallback enabled.")

router.eval()

def apply_top_k(gates, k=3):
    values, indices = torch.topk(gates, k=min(k, gates.size(-1)), dim=-1)
    values = values / (values.sum(dim=-1, keepdim=True) + 1e-8)
    return indices, values

def predict_logits(input_ids, attention_mask, mode="cager_v3", client_name="client_1"):
    """
    Computes next-token logits for the last token position using specified mode.
    """
    if mode == "base":
        with torch.no_grad():
            outputs = base_model(input_ids=input_ids, attention_mask=attention_mask)
            return outputs.logits[0, -1, :], None

    elif mode == "expert":
        expert_model.set_adapter(client_name)
        with torch.no_grad():
            outputs = expert_model(input_ids=input_ids, attention_mask=attention_mask)
            return outputs.logits[0, -1, :], None

    elif mode == "cager_v3":
        if not router_available:
            expert_model.set_adapter(client_name)
            with torch.no_grad():
                outputs = expert_model(input_ids=input_ids, attention_mask=attention_mask)
                return outputs.logits[0, -1, :], "Fallback (Single Expert)"

        with torch.no_grad():
            outputs = base_model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
            hidden_states = outputs.hidden_states[-1]
            
            client_tensor, rank_tensor, score_tensor = get_client_features(client_name, input_ids.size(0))
            gates, _, _, _ = router(hidden_states, client_tensor, rank_tensor, score_tensor)
            
            final_gates = gates[0]
            selected_indices, selected_weights = apply_top_k(final_gates.unsqueeze(0), k=3)
            selected_indices = selected_indices[0]
            selected_weights = selected_weights[0]
            
            vocab_size = expert_model.config.vocab_size
            combined_logits = torch.zeros(vocab_size, device=DEVICE)
            
            routing_desc = []
            for idx, weight in zip(selected_indices, selected_weights):
                expert_name = CLIENTS[idx.item()]
                weight_val = weight.item()
                routing_desc.append(f"{expert_name}: {weight_val*100:.1f}%")
                
                expert_model.set_adapter(expert_name)
                expert_outputs = expert_model(input_ids=input_ids, attention_mask=attention_mask)
                expert_logits = expert_outputs.logits[0, -1, :]
                combined_logits += weight_val * expert_logits
                
            routing_str = ", ".join(routing_desc)
            return combined_logits, routing_str

    raise ValueError(f"Unknown mode: {mode}")

def predict_one_word(prompt, mode="cager_v3", client_name="client_1"):
    """
    Predicts the single next word.
    Handles multi-byte UTF-8 token sequences for Tamil / CodeMix text without replacement chars (\ufffd).
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    curr_input_ids = inputs["input_ids"]
    curr_attention_mask = inputs["attention_mask"]
    
    initial_text = tokenizer.decode(curr_input_ids[0], skip_special_tokens=True)
    generated_ids = curr_input_ids[0].tolist()
    
    routing_str = ""
    max_subwords = 6
    
    with torch.no_grad():
        for i in range(max_subwords):
            logits, curr_routing = predict_logits(curr_input_ids, curr_attention_mask, mode=mode, client_name=client_name)
            if i == 0:
                routing_str = curr_routing
                
            logits[tokenizer.eos_token_id] = -float('inf')
            logits[tokenizer.pad_token_id] = -float('inf')
            next_token_id = torch.argmax(logits).item()
            
            generated_ids.append(next_token_id)
            curr_input_ids = torch.tensor([generated_ids], device=DEVICE)
            curr_attention_mask = torch.ones_like(curr_input_ids)
            
            full_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
            added_text = full_text[len(initial_text):]
            
            # UTF-8 completion check
            if "\ufffd" not in added_text and len(added_text.strip()) > 0:
                if added_text.endswith(" ") or added_text.endswith("\n") or any(c in added_text for c in " .,!?"):
                    break
                if i >= 1 and not added_text.endswith("\ufffd"):
                    break

    full_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    added_text = full_text[len(initial_text):].strip()
    return added_text, routing_str

def complete_sentence(prompt, mode="cager_v3", client_name="client_1", max_new_tokens=8):
    """
    Generates sentence completion up to max_new_tokens.
    """
    current_prompt = prompt
    for _ in range(max_new_tokens):
        next_word, _ = predict_one_word(current_prompt, mode=mode, client_name=client_name)
        if not next_word:
            break
        if next_word.startswith(" ") or current_prompt.endswith(" ") or next_word.startswith("'"):
            current_prompt += next_word
        else:
            current_prompt += " " + next_word
            
        if next_word.strip() in [".", "?", "!"]:
            break
    return current_prompt

def run_interactive_menu():
    active_client = "client_1"
    
    while True:
        print("\n")
        print("=" * 60)
        print("Adaptive Personalized Federated Learning")
        print("Next Word Prediction")
        print("=" * 60)
        print(f"Active Client Profile : {active_client}")
        print("1. Predict Next Word")
        print("2. Complete Sentence")
        print("3. Change Client Profile (client_1 to client_11)")
        print("4. Exit")
        
        try:
            choice = input("\nEnter Choice : ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\nProgram Finished.")
            break
            
        if choice == "1":
            try:
                text = input("\nEnter Sentence : ").strip()
            except (KeyboardInterrupt, EOFError):
                break
            if not text:
                continue
            word, routing = predict_one_word(text, mode="cager_v3", client_name=active_client)
            print("\nInput Sentence")
            print("-------------------------")
            print(text)
            print("\nPredicted Next Word")
            print("-------------------------")
            print(word)
            if routing:
                print("\nActive Routing (CAGER v3)")
                print("-------------------------")
                print(routing)

        elif choice == "2":
            try:
                text = input("\nEnter Sentence : ").strip()
            except (KeyboardInterrupt, EOFError):
                break
            if not text:
                continue
            result = complete_sentence(text, mode="cager_v3", client_name=active_client)
            print("\nCompleted Sentence")
            print("-------------------------")
            print(result)

        elif choice == "3":
            print("\nAvailable Clients: " + ", ".join(CLIENTS))
            try:
                client_choice = input("Enter Client ID (e.g. client_1, client_4, client_10) : ").strip()
            except (KeyboardInterrupt, EOFError):
                break
            if client_choice in CLIENTS:
                active_client = client_choice
                print(f"\n[+] Active Client set to {active_client}")
            else:
                print("\n[-] Invalid Client ID. Keeping current client.")

        elif choice == "4":
            print("\nProgram Finished.")
            break

        else:
            print("\nInvalid Choice")

def main():
    parser = argparse.ArgumentParser(description="Adaptive Personalized FL Next-Word Prediction Console Tool")
    parser.add_argument("--prompt", type=str, default=None, help="Input prompt text")
    parser.add_argument("--client", type=str, default="client_1", help="Target client profile (e.g. client_1, client_4, client_10)")
    parser.add_argument("--non-interactive", action="store_true", help="Run benchmark suite non-interactively")
    args, _ = parser.parse_known_args()

    if args.prompt:
        active_client = args.client if args.client in CLIENTS else "client_1"
        word, routing = predict_one_word(args.prompt, mode="cager_v3", client_name=active_client)
        completed = complete_sentence(args.prompt, mode="cager_v3", client_name=active_client)
        print("\nInput Sentence      :", args.prompt)
        print("Predicted Next Word :", word)
        print("Completed Sentence  :", completed)
        if routing:
            print("CAGER v3 Routing    :", routing)
    elif getattr(args, "non_interactive", False):
        print("\n[+] Running Benchmark Suite Non-Interactively...")
        sample_prompts = [
            "The artificial intelligence model is",
            "Federated learning allows private data to",
            "நான் இன்று"
        ]
        for prompt in sample_prompts:
            word, routing = predict_one_word(prompt, mode="cager_v3", client_name="client_1")
            completed = complete_sentence(prompt, mode="cager_v3", client_name="client_1")
            print(f"\nPrompt     : {prompt}")
            print(f"Next Word  : {word}")
            print(f"Completed  : {completed}")
            print(f"Routing    : {routing}")
    else:
        run_interactive_menu()

if __name__ == "__main__":
    main()


[+] Loading Base Model (DistilGPT2)...


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 7992.56it/s]


[+] Loading Adaptive PEFT Expert Adapters...
[+] Loading Personalized CAGER v3 Router Checkpoint...
[+] CAGER v3 Router loaded successfully.


Adaptive Personalized Federated Learning
Next Word Prediction
Active Client Profile : client_1
1. Predict Next Word
2. Complete Sentence
3. Change Client Profile (client_1 to client_11)
4. Exit

Invalid Choice


Adaptive Personalized Federated Learning
Next Word Prediction
Active Client Profile : client_1
1. Predict Next Word
2. Complete Sentence
3. Change Client Profile (client_1 to client_11)
4. Exit

Completed Sentence
-------------------------
Artificial intelligence ( AI) is a technology that is designed


Adaptive Personalized Federated Learning
Next Word Prediction
Active Client Profile : client_1
1. Predict Next Word
2. Complete Sentence
3. Change Client Profile (client_1 to client_11)
4. Exit

Input Sentence
-------------------------
The movie

Predicted Next Word
-------------------------
is

Active Routing (CAGER v3)
-----------------